# [1.5.3] OthelloGPT (연습 문제)

> **ARENA [Streamlit Page](https://arena-chapter1-transformer-interp.streamlit.app/33_[1.5.3]_OthelloGPT)**
>
> **Colab: [exercises](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part53_othellogpt/1.5.3_OthelloGPT_exercises.ipynb?t=20260329) | [solutions](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part53_othellogpt/1.5.3_OthelloGPT_solutions.ipynb?t=20260329)**

문제나 버그가 있다면 [Slack group](https://join.slack.com/t/arena-uk/shared_invite/zt-3afdmdhye-Mdb3Sv~ss_V_mEaXEbkABA)의 `#errata` 채널로 보내주시고, 본 장의 학습 자료에 관한 질문은 전용 채널에서 해주시기 바랍니다.

마크다운 헤더 셀 왼쪽에 있는 화살표 기호를 클릭하면 각 섹션을 접어서 헤더만 보이게 할 수 있습니다.

다른 모든 장으로 이동하는 링크: [(0) Fundamentals](https://arena-chapter0-fundamentals.streamlit.app/), [(1) Transformer Interpretability](https://arena-chapter1-transformer-interp.streamlit.app/), [(2) RL](https://arena-chapter2-rl.streamlit.app/).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/headers/header-15-3.png" width="350">

# 소개

*참고 - 별도로 명시되지 않는 한, 여기서 1인칭은 주 연구자인 Neel Nanda를 가리킵니다.*

[Emergent World Representations](https://arxiv.org/pdf/2210.13382)은 Kenneth Li 등이 작성한 매우 흥미로운 최신 ICLR Oral 논문이며, [Kenneth's excellent post on the Gradient](https://thegradient.pub/othello/)에 요약되어 있습니다. 연구진은 무작위 게임(합법적인 다음 수를 균등하게 무작위로 선택하여 생성)을 제공하고 다음 수를 예측하도록 학습시켜, 보드 게임인 Othello의 합법적인 수를 두는 모델(Othello-GPT)을 학습시켰습니다. 핵심 결과는 Othello-GPT가 창발적인 세계 표현(emergent world representation)을 학습한다는 점입니다. 보드 상태를 명시적으로 제공받지 않고 단지 다음 수를 예측하는 과제만 수행했음에도 불구하고, 모델은 매 수마다 보드 상태를 계산하는 법을 학습합니다. (Othello-GPT의 목적은 좋은 수가 아니라 합법적인 수를 두는 것이라는 점에 유의하십시오. 다만 연구진은 좋은 수를 두도록 학습된 모델도 함께 연구했습니다.)

연구진은 두 가지 주요 증거를 제시합니다. 첫째, 비선형 probe(2층 ReLU MLP)를 통해 모델의 residual stream에서 보드 상태를 추출할 수 있습니다. 둘째, 이 probe를 사용하여 모델의 보드 표현에 인과적으로 개입(causally intervene)하여 변경할 수 있습니다(gradient descent를 사용하여 probe가 새로운 보드 상태를 출력하게 함). 이렇게 하면 모델은 이전 보드에서는 합법적이지 않았던 수, 심지어 합법적인 플레이로는 도달할 수 없는 보드 상태에서도 새로운 보드 상태에 맞는 합법적인 수를 둡니다!

저는 연구진의 정교한(따라서 잠재적으로 오해의 소지가 있는) 기법 중 상당 부분이 크게 단순화될 수 있음을 발견하여 이 핵심 결과를 강화했습니다. 모델은 단순히 창발적인 세계 표현을 학습하는 것이 아니라, 선형적인 창발적 세계 표현을 학습하며, 이는 선형적인 방식으로 인과적 개입이 가능합니다! 또한 모델은 "이 칸에 검은색/흰색 말이 있다"라고 표현하는 대신 "이 칸에 내 말/상대방 말이 있다"라고 표현합니다. 모델은 흑과 백의 수를 모두 두기 때문에, 모델의 관점에서는 이것이 훨씬 더 자연스럽습니다. 이러한 통찰을 통해 전체적인 그림이 훨씬 명확해지며, 모델은 훨씬 더 해석 가능해집니다!

### 연구 과정

연구 과정에 대한 더 자세한 내용은 [this post here](https://www.lesswrong.com/s/nhGNHyJHbrofpPbRG/p/TAz44Lb9n9yf52pv8)에서 읽어보실 수 있으며, 이를 강력히 추천합니다. 연습 문제는 실제 시간 순서에 따른 연구 과정과는 다르게 구성되어 있습니다 (예를 들어, 실제로는 probe를 학습시키는 것이 많은 노력이 드는 작업이며, 이러한 문제에 접근할 때 처음에는 logit lens나 attention pattern 분석과 같은 더 기본적인 기법을 먼저 사용하려 하겠지만, 여기서는 초반에 probe를 살펴봅니다).

이 연습 문제들을 풀면서, 여러분이 이 연구 과정에 어떻게 접근했을지 계속 생각해보시기를 권장합니다. 결과나 접근 방식 중 완전히 뜬금없게 느껴지는 것이 있습니까? 만약 그렇다면, 그것을 시도한 근거가 무엇이었을지 생각해보실 수 있습니까? 여러분이라면 무엇을 먼저 시도했을 것이며, 그 이유는 무엇입니까?

### 연습 문제 접근 방법

OthelloGPT를 분석하기 위한 설정 코드가 매우 많습니다 (이는 어느 정도 불가피합니다). 또한 서로 다른 차원과 의미를 가진 tensor들의 다양한 plot이 많아 이를 계속 추적하기 어려울 수 있습니다. 이 모든 것을 머릿속에 다 담아둘 필요는 없지만, **다음 섹션으로 넘어가기 전에 각 결과의 의미가 무엇인지 스스로 또는 페어 프로그래밍 파트너에게 설명하는 것을 강력히 추천합니다.** 때로는 연습 문제나 질문이 이를 유도하겠지만, 이를 습관으로 만드셔야 합니다!

첫 번째 섹션(주로 설정으로 구성됨)의 각 하위 섹션 끝에는 우리가 정의한 모든 객체, 그것들이 왜 중요한지, 그리고 어떻게 사용되는지에 대한 요약이 있습니다. 이후의 각 섹션 끝에도 핵심 결과와 그 의미를 검토하는 "이 섹션의 요약"이 있을 것입니다.

## 이 실습들의 목적 / 구조

표면적으로 이 실습들은 여러분이 OthelloGPT 모델과 해당 모델에 적용된 probing 및 분석 형태를 이해하도록 돕기 위해 설계되었습니다. 하지만 동시에 여러분을 더 뛰어난 interpretability 연구자로 만들기 위해 설계되었습니다! 그 결과, 대부분의 실습은 다음 사항들의 조합으로 구성됩니다:

1. OthelloGPT의 새로운 feature/component를 보여주는 것, 그리고
2. 더 넓은 mech interp 맥락에서 도구를 사용하고 결과를 해석하는 방법을 가르치는 것입니다.

이 실습들을 진행하다 보면, 구현하고 있는 기술의 까다로운 세부 사항이나 계산하고 있는 내용에 매몰되기 쉽습니다. 현재 어떤 질문을 던지려 하는지, 얻은 출력을 어떻게 해석할 것인지, 그리고 현재 사용 중인 도구들이 모델에 대한 더 나은 이해로 어떻게 안내하고 있는지 스스로 질문하며 계속해서 높은 수준의 관점을 유지하시기 바랍니다.

## 내용 및 학습 목표

### 1️⃣ 모델 설정 및 Linear Probes

이 섹션에서는 나머지 실습에서 사용할 모델을 설정하고 가중치를 로드합니다. 또한 이번 분석에 사용할 데이터셋과 객체들에 익숙해지는 시간을 갖겠습니다. 마지막으로, **linear probes**가 무엇인지와 어떻게 사용되는지 학습합니다.

> ##### 학습 목표
>
> - Othello-GPT 모델의 기본 구조를 이해합니다.
> - 실습 동안 보드 시각화에 사용할 기본 도구들에 익숙해집니다.
> - linear probe가 어떻게 작동하는지, 그리고 "흑/백 돌"과 "내/상대 돌" 사이에서 어떻게 basis를 변경할 수 있는지 확인합니다.
> - linear probe를 통해 개입하여 모델의 예측에 영향을 줍니다.

### 2️⃣ 모듈형 circuit 찾기

여기서는 probe를 사용하여 모델 내의 circuit를 분석하기 시작합니다. 이를 neuron의 output weights에 적용하여 어떤 neuron이 어떤 방식으로 중요한지 식별할 수 있으며, 모델의 어디에서 언제 정보가 표현되는지 파악할 수 있습니다.

> ##### 학습 목표
>
> - 여러 layer에 걸쳐 linear probe를 사용하는 방법을 학습합니다.
> - 특정 sequence 위치에서 activation patching을 적용하여 모델에 대한 가설을 테스트합니다.
> - neuron이 input 및 output weights 관점에서 어떻게 특징지어질 수 있는지 이해합니다.

### 3️⃣ Neuron 해석 가능성: 심층 분석

neuron 해석 가능성을 연습하기 위해, 특정 neuron 하나를 심층적으로 분석해 보겠습니다. 여기서 배우는 기술과 코드는 다른 모든 neuron에도 매우 잘 적용될 것입니다!

이 섹션의 취지는 실제 상황에서 다른 neuron에 적용할 수 있는 다양한 표준 작업들을 연습하는 것입니다. 분석을 마친 후에도 여전히 혼란스러운 부분이나 해결되지 않은 의문점들이 남아있을 수 있습니다!

> ##### 학습 목표
>
> - **direct logit attribution**을 적용하여 neuron의 output weights가 예측에 어떻게 직접적으로 영향을 미치는지 이해합니다.
> - SVD 기반 기술을 사용하여 neuron의 input/output 동작 중 얼마만큼이 특정 subspace에 의해 포착되는지 평가합니다.
> - **max activating datasets** 및 **spectrum plots**와 같은 기술을 사용하고, 그 장점과 한계를 이해합니다.

### 4️⃣ Probe 학습시키기

이 섹션에서는 실제로 linear probe를 학습시키는 방법을 살펴봅니다. 이 부분은 mechanistic interpretability보다는 표준 ML 기술에 가깝지만, 직접 이러한 분석을 수행하고 싶다면 학습 방법을 이해하는 것이 여전히 중요합니다!

> ##### 학습 목표
>
> - linear probe를 설정하고 학습시키는 방법을 학습합니다.
> - 여러 개의 probe를 동시에 학습시키고, 그 성능을 Weights & Biases에 기록하는 방법을 확인합니다.

### ☆ 보너스

마지막으로, 이번 OthelloGPT 분석에서 파생될 수 있는 향후 연구 방향들을 살펴보겠습니다. 이러한 연구 흐름을 직접 따라가 보며 어디로 이어지는지 확인해 보시는 것을 강력히 추천합니다!

## 설정 코드

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

chapter = "chapter1_transformer_interp"
repo = "ARENA_3.0"
branch = "main"

# Install dependencies
try:
    import transformer_lens
except:
    %pip install einops eindex-callum jaxtyping wandb transformer_lens==2.17.0 "git+https://github.com/neelnanda-io/neel-plotly"

# Get root directory, handling 3 different cases: (1) Colab, (2) notebook not in ARENA repo, (3) notebook in ARENA repo
root = (
    "/content"
    if IN_COLAB
    else "/root"
    if repo not in os.getcwd()
    else str(next(p for p in Path.cwd().parents if p.name == repo))
)

if Path(root).exists() and not Path(f"{root}/{chapter}").exists():
    if not IN_COLAB:
        !sudo apt-get install unzip
        %pip install jupyter ipython --upgrade

    if not os.path.exists(f"{root}/{chapter}"):
        !wget -P {root} https://github.com/callummcdougall/ARENA_3.0/archive/refs/heads/{branch}.zip
        !unzip {root}/{branch}.zip '{repo}-{branch}/{chapter}/exercises/*' -d {root}
        !mv {root}/{repo}-{branch}/{chapter} {root}/{chapter}
        !rm {root}/{branch}.zip
        !rmdir {root}/{repo}-{branch}


if f"{root}/{chapter}/exercises" not in sys.path:
    sys.path.append(f"{root}/{chapter}/exercises")

os.chdir(f"{root}/{chapter}/exercises")

In [ ]:
import copy
import os
import sys
from dataclasses import dataclass
from functools import partial
from pathlib import Path
from typing import Callable

import einops
import numpy as np
import pandas as pd
import plotly.express as px
import torch as t
import wandb
from eindex import eindex
from jaxtyping import Bool, Float, Int
from torch import Tensor
from tqdm.notebook import tqdm
from transformer_lens import ActivationCache, HookedTransformer, HookedTransformerConfig
from transformer_lens.hook_points import HookPoint
from transformer_lens.utils import download_file_from_hf, get_act_name, to_numpy

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")

# Make sure exercises are in the path
chapter = "chapter1_transformer_interp"
section = "part53_othellogpt"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

import part53_othellogpt.tests as tests
import part53_othellogpt.utils as utils
from neel_plotly import scatter

t.set_grad_enabled(False)

MAIN = __name__ == "__main__"

# 1️⃣ 모델 설정 및 Linear Probes

> ##### 학습 목표
>
> - Othello-GPT 모델의 기본 구조를 이해합니다.
> - 이번 실습 동안 보드 시각화에 사용할 기본 도구들에 익숙해집니다.
> - linear probe가 어떻게 작동하는지, 그리고 "흑/백 돌"과 "내 돌/상대 돌" 사이에서 어떻게 basis를 변경할 수 있는지 확인합니다.
> - 모델 예측에 영향을 주기 위해 linear probe로 개입(intervene)해 봅니다.

익숙하지 않은 분들을 위해 설명하자면, Othello는 체스나 바둑과 유사한 보드게임으로, 흑과 백 두 명의 플레이어가 참여하며 규칙은 아래 그림에 설명되어 있습니다. 저는 직관을 형성하는 데 [playing the AI on eOthello](https://www.eothello.com/) 이 도움이 되었습니다. 단 한 번의 수가 (수직, 수평 또는 대각선으로 연속된 선만 있다면) 멀리 떨어진 말들의 색깔을 바꿀 수 있으며, 이는 보드 상태를 계산하는 것이 실제로 꽤 어렵다는 것을 의미합니다! (제가 보기에는 체스보다 훨씬 더 어렵습니다)

<img src="https://res.cloudinary.com/lesswrong-2-0/image/upload/f_auto,q_auto/v1/mirroredImages/nmxzr2zsjNtjaHh7x/eoi5u5fodorjswnmbqk7" width="700">

## 배경 - Othello 및 OthelloGPT

하지만 모델이 단지 다음 수를 예측하기만 하면 됨에도 불구하고, 각 수마다 전체 보드 상태를 계산하는 법을 자발적으로 학습했다는 점은 매우 흥미로운 결과입니다. 현재 매우 뜨거운 쟁점 중 하나는 LLM이 단순히 통계적 상관관계의 묶음인지, 아니면 실제적인 이해와 계산 능력을 갖추고 있는지 여부입니다! 이는 다음 token을 예측하는 단순한 목적 함수가 (적어도 Othello라는 toy setting에서는) 풍부한 emergent structure를 생성할 수 있다는 암시적인 증거를 제공합니다. 단순히 수의 분포에 대한 표면적인 통계만 학습한 것이 아니라, 그 데이터를 생성한 기저의 프로세스를 모델링하는 법을 학습한 것입니다. 제 생각에 transformer가 통계적 상관관계와 패턴 매칭 이상의 무언가를 할 수 있다는 점은 이미 꽤 분명하지만(예: [induction heads](https://transformer-circuits.pub/2022/in-context-learning-and-induction-heads/index.html) 참조), 완전한 형태의 world model에 대한 더 명확한 증거를 얻게 된 것은 매우 멋진 일입니다!

제 조사의 맥락을 위해, 그들이 emergent world representation의 증거로 제시한 두 가지 요소인 probe와 causal intervention, 그리고 각각의 강점과 약점을 정확히 분석해 볼 가치가 있습니다.

### Probes

probe는 암시적인 증거를 제공하지만, 결코 결정적인 증거는 아닙니다. 모델에서 특정 feature를 추출하기 위해 probe를 학습시킬 때, 스스로를 속이기 쉽습니다. probe가 단순히 feature를 읽어내고 있는 것인지, 아니면 실제로 feature 자체를 계산하면서 모델로부터 훨씬 더 단순한 feature들을 읽어내고 있는 것인지를 추적하는 것이 매우 중요합니다. 극단적인 경우, 훨씬 더 강력한 모델을 "probe"로 부착하여 입력된 수들만 추출하게 한 뒤, 처음부터 보드 상태를 계산하게 할 수도 있습니다! 그들은 linear probe로는 보드 상태를 복구하는 것이 작동하지 않았음을 발견했습니다(오류율 20.4%): (즉, residual stream을 각 칸에 대해 비어 있음, 검은색, 흰색 logit에 해당하는 3개의 학습된 방향으로 투영하는 방식). 반면 가장 단순한 non-linear probe(단일 hidden ReLU layer를 가진 2층 MLP)는 매우 잘 작동했습니다(오류율 1.7%). 더 나아가 (아래 스크린샷의 table 2에 설명된 것처럼), 이러한 non-linear probe는 무작위로 초기화된 네트워크에서는 작동하지 않았으며, 일부 layer에서 다른 layer보다 더 잘 작동했습니다. 이는 probe가 모델로부터 실제 무언가를 학습하고 있음을 시사합니다.

<img src="https://res.cloudinary.com/lesswrong-2-0/image/upload/f_auto,q_auto/v1/mirroredImages/nmxzr2zsjNtjaHh7x/b5rxfns1wxvsuomh1d79" width="550">

### Causal interventions

probe 단독으로는 오해를 불러일으킬 수 있으며, 모델이 반드시 이 representation을 사용한다는 것을 알려주지는 않습니다. probe가 어떤 흔적만 남은 feature나 더 유용한 계산의 부수적인 효과를 추출하여, 모델이 솔루션을 계산하는 방식에 대해 오해의 소지가 있는 그림을 제공할 수 있기 때문입니다. 하지만 그들의 causal intervention은 이를 훨씬 더 설득력 있는 증거로 만듭니다. 그들은 상당히 복잡한 과정(아래 그림에 상세히 나와 있지만, 세부 사항까지 이해하실 필요는 없습니다)을 통해 개입합니다. 요약하자면, 새로운 보드 상태를 선택하고 모델의 residual stream에 gradient descent를 적용하여, probe가 모델의 residual stream이 새로운 보드 상태를 나타내고 있다고 생각하게 만드는 방식입니다. 저는 이와 같은 복잡한 기법에 대해 즉각적인 회의감을 갖습니다. gradient descent와 같은 강력한 방법을 적용할 때, 모델의 원래 작동 방식에서 크게 벗어나기가 너무 쉽기 때문입니다! 하지만 편집된 보드 상태를 적법한 수로 변환하는 비자명한 계산을 모델이 편집 후에 수행할 수 있었다는 사실은 매우 인상적인 결과입니다! 저는 이것이 probe가 실제 무언가를 발견했다는 점과, probe에 의해 발견된 representation이 모델의 실제 계산과 인과적으로 연결되어 있다는 점을 보여주는 매우 강력한 증거라고 생각합니다!

## Mechanistic Interpretability에 대한 단순한 시사점

저는 이 논문에 매우 관심이 많았습니다. 왜냐하면 emergent world model이라는 매혹적인 발견이 있었고 (또한 저는 일반적으로 모든 좋은 interp 논문에 관심이 많습니다), 그럼에도 불구하고 무언가 어색한 느낌이 있었기 때문입니다. 여기서 사용된 기술들은 "너무" 강력해 보였습니다. 결과는 무언가 분명히 실재한다는 느낌을 줄 만큼 강력했지만, 제 직관으로는 모델의 내부 구조를 정말로 이해했다면 linear probe나 intervention과 같은 훨씬 더 단순한 기술만으로도 모델을 이해하고 조작할 수 있어야 하며, 더 강력한 기술들에 의해 오도되기 쉽다고 생각합니다.

특히, 모델 내부에 대한 저의 가장 유력한 추측은 네트워크가 [**decomposable, linear representations**](https://dynalist.io/d/n2ZWtnoYHrU1s4vnFSAQ519J#z=KiDMIteKCEXt_EkR2sZCOfbG)를 형성한다는 것입니다. 즉, 모델이 다수의 유용한 feature들을 계산하고, 이를 activation 공간의 방향(direction)으로 표현한다는 것입니다. 이에 대한 훌륭한 설명은 [Toy Models of Superposition](https://transformer-circuits.pub/2022/toy_model/index.html#motivation)를 참조하십시오. 이는 각 feature가 독립적으로 변할 수 있기 때문에(모델의 관점에서 그렇습니다. 데이터 분포상으로는 서로 의존적일 가능성이 높습니다) decomposable하며, 해당 feature의 방향으로 투영함으로써 feature를 추출할 수 있기 때문에 linear합니다 (feature들이 orthogonal할 경우에 해당하며, [superposition](https://dynalist.io/d/n2ZWtnoYHrU1s4vnFSAQ519J#z=3br1psLRIjQCOv2T4RN3V6F2)와 같은 상황이라면 더 복잡해집니다). 이는 모델이 작동하는 자연스러운 방식입니다. 모델은 기본적으로 적절한 위치에 non-linearity가 삽입된 일련의 matrix multiplication이며, decomposable하고 linear한 representation은 linear map을 통해 어떤 feature의 조합이라도 추출할 수 있게 해줍니다!

이 프레임워크 하에서, 만약 어떤 feature가 linear probe로 발견될 수 있다면 모델은 이미 그것을 계산한 것이며, 그 feature가 downstream의 circuit에서 사용된다면 우리는 해당 feature 방향의 좌표만 변경하는 linear intervention을 통해 인과적으로 개입할 수 있어야 합니다. 따라서 linear probe는 작동하지 않지만 non-linear probe는 작동한다는 매혹적인 발견은, 모델이 feature에 대해 근본적으로 non-linear한 representation을 가지고 있거나(그리고 이를 downstream 계산에 직접 사용할 수 있거나!), 혹은 probe가 board state를 계산해내는 더 단순하고 자연스러운 feature들의 linear representation이 존재함을 시사합니다. 저의 사전 믿음은 더 단순한 feature들의 linear representation 쪽이었지만, causal intervention 결과는 non-linear representation에 대한 어느 정도의 증거처럼 느껴졌습니다. 그리고 non-linear representation 가설이 사실이라면 이는 매우 중요한 일이 될 것입니다! 모델을 reverse-engineer 하려면 모델의 계산이 activation과 weight에 어떻게 매핑되는지에 대한 명확한 그림이 필요한데, 이는 이러한 대응 관계가 어떻게 작동하는지에 대한 저의 많은 믿음을 깨뜨릴 것이기 때문입니다! 더욱이, linear representation은 reverse-engineer 하기에 정말로 *편리*하며, 이것이 아니라면 저는 mechanistic interpretability가 성공할 가능성에 대해 현저히 더 비관적이 될 것입니다.

## 모델 설정 및 시각화

신비로운 모델 동작에 대해 덜 혼란스러워지는 가장 좋은 방법은 이를 기계론적으로 분석하는 것입니다. 우리가 찾을 수 있는 feature와 circuit를 확대해서 살펴보고, 바텀업 방식으로 이해도를 쌓아 올리며, 이를 통해 실제로 어떤 일이 일어나고 있는지에 대한 근거 있는 믿음을 형성하는 것입니다.

시작하기 위해, 이번 장에서 사용할 모델을 로드하겠습니다. 몇 가지 중요한 기술적 세부 사항은 다음과 같습니다:

- Othello 게임은 60번의 수로 이루어집니다 (보드 크기는 8x8이며, 중앙 4칸은 이미 채워져 있으므로 60수가 지나면 모든 칸이 특정 색으로 채워집니다). 모델의 context length는 59인데, 이는 `moves[0:59]`를 입력받아 `moves[1:60]`를 예측하도록 학습되었기 때문입니다.
- 모델이 학습한 데이터셋은 무작위로 샘플링된 legal move들로만 구성되어 있습니다. Othello에서 한 수가 legal 하려면 가로, 세로 또는 대각선 방향으로 상대방의 기물을 캡처해야 하며, 이 경우 해당 기물들을 모두 자신의 색으로 뒤집습니다. 이는 **다음 legal move를 예측하는 것이 간단하지 않은 작업임을 의미합니다 (모델이 시간에 따른 캡처 상황과 보드 상태를 추적해야 한다고 추측할 수 있습니다)**.
    - 모델이 실제 다음 수에 대해 cross entropy loss로 학습되었기 때문에, 모델의 분포는 모든 다음 legal move에 대해 uniform하게 수렴합니다. 그 이유를 알 수 있습니까?

<details>
<summary>증명 - 모델이 다음 legal move에 대해 uniform distribution을 학습하는 이유</summary>

cross entropy loss는 $H(p, q) = -\sum_x p(x) \log q(x)$로 정의되며, [Gibbs' inequality](https://en.wikipedia.org/wiki/Gibbs%27_inequality)에 의해 $p(x) = q(x)$일 때 최소화됩니다. 따라서 모델의 분포 $q$가 기본 데이터 분포 $p$(다음 legal move에 대해 uniform함)와 일치할 때 loss가 최소화됩니다.

$\log q(x)$이 엄격한 concave 함수이므로 Jensen's inequality를 사용하여 이를 증명할 수도 있습니다. 또는, cross entropy loss를 기본 데이터 분포의 entropy와 모델 분포 및 기본 분포 사이의 KL divergence의 합으로 표현할 수 있습니다:

$$
H(p, q) = -\sum_x p(x) \log q(x) = \sum_x p(x) \log \frac{p(x)}{q(x)} - \sum_x p(x) \log p(x) = D_{\mathrm{KL}}(p \| q) + H(p)
$$

따라서 이 문장은 $D_{\mathrm{KL}}(p \| q)$가 항상 non-negative하며 $p(x) = q(x)$일 때만 0이 된다는 문장과 동일해집니다 (이는 유사한 방식으로 증명될 수 있습니다).

</details>

- vocabulary size는 61입니다. 이는 8x8 - 4 = 60개의 비어 있는 칸 중 어느 곳이든 둘 수 있고, 여기에 pass move가 추가되기 때문입니다. vocab은 `pass, A0, A1, ..., H7` 순서로 정렬되어 있습니다. `pass` move가 플레이된 게임은 필터링할 예정이므로, 이에 대해 걱정할 필요는 없습니다.
- 우리는 square를 세 가지 다른 방식으로 지칭합니다:
    1. **label** - 문자열 표현입니다. 즉, `"pass"`, `"A0"`, `"A1"`, ..., `"H7"` 입니다.
    2. **token id**, 또는 **id** - 모델 vocab에서의 token ID입니다. 즉, A0의 경우 `1`, ..., H7의 경우 `60` 입니다. `pass`의 token id인 `0`은 건너뛰며, 중앙 4칸은 항상 채워져 있어 해당 칸으로의 move나 예측이 없으므로 건너뜁니다.
    3. **square index**, 또는 **square** - 크기 64 보드에서 square의 zero-indexed 값입니다. 즉, A0의 경우 `0`, A1의 경우 `1`, ..., H7의 경우 `63` 입니다.
- Othello에서는 흑이 먼저 시작하며, 따라서 (pass가 없는 게임의 경우) 백이 마지막으로 둡니다. 우리는 아주 첫 번째 수는 예측하지 않으므로, 모델의 예측은 (백 1, 흑 2, 백 2, ..., 백 29, 흑 30, 백 30)에 대한 것입니다.


<!-- Here are some more files, which we won't be using directly in these exercises, but you might still like to have a look at:

* `tl_probing.py` is my probe training file. But it was used to train a second probe, linear_probe_L4_blank_vs_color_v1.pth . This probe actually didn't work very well for analysing the model (despite getting great accuracy) and I don't know why - it was trained on layer 4, to do a binary classification on blank vs not blank, and on my color vs their color *conditional *on not being blank (ie not evaluated if blank). For some reason, the "this cell is my color" direction has a significant dot product with the "is blank" direction, and this makes it much worse for e.g. interpreting neurons. I don't know why!
* `tl_scratch.py` is where I did some initial exploration, including activation patching between different final moves.
* `tl_exploration.py` is where I did my most recent exploration, verifying that the probe works, doing probe interventions (CTRL F for `newly_legal`) and using the probe to interpret neurons. -->

모델을 로드하기 위해 아래 코드를 실행하십시오:

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=8,
    d_model=512,
    d_head=64,
    n_heads=8,
    d_mlp=2048,
    d_vocab=61,
    n_ctx=59,
    act_fn="gelu",
    normalization_type="LNPre",
    device=device,
)
model = HookedTransformer(cfg)

state_dict_synthetic = download_file_from_hf("NeelNanda/Othello-GPT-Transformer-Lens", "synthetic_model.pth")
# state_dict_championship = download_file_from_hf("NeelNanda/Othello-GPT-Transformer-Lens", "championship_model.pth")

model.load_state_dict(state_dict_synthetic)

다음으로, 아래 코드를 실행하여 모델이 올바르게 작동하는지 확인합니다. 우리의 `sample_input` 은 10개의 수로 이루어진 시퀀스입니다 (흑의 첫 번째 수로 시작하여 백의 다섯 번째 수로 끝나는 게임의 처음 10수입니다).

In [ ]:
# An example input: 10 moves in a game
sample_input = t.tensor([[20, 19, 18, 10, 2, 1, 27, 3, 41, 42]]).to(device)

logits = model(sample_input)
logprobs = logits.log_softmax(-1)

assert logprobs.shape == (1, 10, 61)  # shape is [batch, seq_len, d_vocab]
assert logprobs[0, 0].topk(3).indices.tolist() == [
    21,
    33,
    19,
]  # these are the 3 legal moves, as we'll soon show

우리는 영리한 인덱싱을 통해 각 logprob 벡터(shape `(61,)`)를 shape `(8, 8)`의 보드 상태 tensor로 변환할 수 있습니다 (가운데 4개 칸에는 매우 낮은 음수 값을 넣습니다). 그런 다음, 여러분을 위해 작성된 헬퍼 함수 `utils.plot_board_values`를 사용하여 결과를 플롯할 수 있습니다. 이후 연습 문제에서 이 함수를 많이 사용하게 될 것이므로, 여기서 중요한 인자들을 살펴보겠습니다:

- `state`: shape `(*N, 8, 8)`의 tensor이며, 여기서 `*N`는 임의의 batch 차원입니다. 각 `N` tensor에 대해 하나의 8x8 그리드를 시각화합니다. 만약 shape이 `(8, 8)`라면 하나의 보드만 플롯합니다.
- `board_titles`: 여러 보드를 플롯하는 경우(즉, `N > 1`), 이 인자를 사용하여 각 보드에 라벨을 붙일 수 있습니다.
- `boards_per_row`: 여러 보드를 플롯할 때, 한 행에 표시할 보드의 수를 설정할 수 있습니다.
- `text`: `state`와 동일한 shape을 가진 선택적 문자열 리스트이며, 보드에 주석을 다는 데 사용할 수 있습니다. `state`의 shape으로 브로드캐스팅 가능해도 무방합니다 (예를 들어, `state`가 3D이더라도 `(8, 8)`일 수 있습니다).
- `kwargs`: 다른 인자들은 `px.imshow`로 전달됩니다 (예: `title`, `width`, `height`).

사용법이 기억나지 않을 때는 언제든지 다음 예제들에서 코드를 복사하여 붙여넣을 수 있으므로, 이 함수가 정확히 어떻게 작동하는지 이해하는 것은 중요하지 않습니다!

In [ ]:
MIDDLE_SQUARES = [27, 28, 35, 36]
ALL_SQUARES = [i for i in range(64) if i not in MIDDLE_SQUARES]

logprobs_board = t.full(size=(8, 8), fill_value=-13.0, device=device)
logprobs_board.flatten()[ALL_SQUARES] = logprobs[0, 0, 1:]  # the [1:] is to filter out logits for the "pass" move

utils.plot_board_values(logprobs_board, title="Example Log Probs", width=500)

<details>
<summary>참고 - 이 tensor indexing 마법은 어떻게 작동하나요?</summary>

`temp_board_state`은 `(8, 8)` 모양의 array입니다. `.flatten()`를 사용하면, `(64,)` 모양을 가진 **view**(즉, 동일한 기본 데이터)가 반환됩니다. 이를 `ALL_SQUARES`("중앙 사각형"을 제외한 모든 인덱스인 60개 인덱스 리스트)로 인덱싱하면, 이 또한 view를 반환합니다(여전히 동일한 데이터입니다). 그런 다음 이 60개 요소들을 모델의 log probs로 설정할 수 있습니다. 이렇게 하면 tensor의 모양을 변경하지 않고 원래 tensor의 값들을 변경할 수 있습니다.
</details>

어떤 경우에는 `text` 인자를 사용하여 모든 유효한 사각형에 token ID를 주석으로 다는 것이 더 쉬울 수 있습니다. 다음은 그 예시입니다 (이 코드를 이후의 실습에서 재사용하고 싶으실 것입니다):

In [ ]:
TOKEN_IDS_2D = np.array([str(i) if i in ALL_SQUARES else "" for i in range(64)]).reshape(8, 8)
BOARD_LABELS_2D = np.array(["ABCDEFGH"[i // 8] + f"{i % 8}" for i in range(64)]).reshape(8, 8)

print(TOKEN_IDS_2D)
print(BOARD_LABELS_2D)

utils.plot_board_values(
    t.stack([logprobs_board, logprobs_board]),  # shape (2, 8, 8)
    title="Example Log Probs (with annotated token IDs)",
    width=800,
    text=np.stack([TOKEN_IDS_2D, BOARD_LABELS_2D]),  # shape (2, 8, 8)
    board_titles=["Labelled by token ID", "Labelled by board label"],
)

동일한 방식으로 이 함수를 사용하여 여러 board state를 한 번에 plot할 수도 있습니다. 앞으로의 학습에 도움이 될 수 있도록, 인덱싱이 어떻게 작동하는지 다시 한번 주의 깊게 살펴보시기 바랍니다!

In [ ]:
logprobs_multi_board = t.full(size=(10, 8, 8), fill_value=-13.0, device=device)
logprobs_multi_board.flatten(1, -1)[:, ALL_SQUARES] = logprobs[0, :, 1:]  # we now do all 10 moves at once

utils.plot_board_values(
    logprobs_multi_board,
    title="Example Log Probs",
    width=1000,
    boards_per_row=5,
    board_titles=[f"Logprobs after move {i}" for i in range(1, 11)],
)

동일한 함수를 사용하여 처음 10개의 board state를 시각화하고, 위에서 보여준 예측들이 당시의 board state를 고려했을 때 타당한지 확인해 보겠습니다. 각 move 이후의 board state를 추적하기 위해 helper method `OthelloBoardState`를 사용하겠습니다.

In [ ]:
board_states = t.zeros((10, 8, 8), dtype=t.int32)
legal_moves = t.zeros((10, 8, 8), dtype=t.int32)

board = utils.OthelloBoardState()
for i, token_id in enumerate(sample_input.squeeze()):
    # board.umpire takes a square index (i.e. from 0 to 63) and makes a move on the board
    board.umpire(utils.id_to_square(token_id))

    # board.state gives us the 8x8 numpy array of 0 (blank), -1 (black), 1 (white)
    board_states[i] = t.from_numpy(board.state)

    # board.get_valid_moves() gives us a list of the indices of squares that are legal to play next
    legal_moves[i].flatten()[board.get_valid_moves()] = 1

# Turn `legal_moves` into strings, with "o" where the move is legal and empty string where illegal
legal_moves_annotation = np.where(to_numpy(legal_moves), "o", "").tolist()

utils.plot_board_values(
    board_states,
    title="Board states",
    width=1000,
    boards_per_row=5,
    board_titles=[f"State after move {i}" for i in range(1, 11)],
    text=legal_moves_annotation,
)

이 그래프에서 각 보드 상태는 상대방 색상의 기물을 잡는 단일 수에 의해 이전 상태로부터 진화하는 것을 볼 수 있습니다 (예를 들어, 첫 번째 수에서 black은 `C3`를 두어 `D3`에 있는 white를 잡고, 두 번째 수에서 white는 `C2`를 두어 `D3`에 있는 black을 대각선으로 다시 잡습니다). 또한, 표시된 legal moves가 모델에 의해 예측된 수와 일치하는 것을 확인할 수 있습니다.

## 데이터

이제 OthelloGPT를 위한 데이터를 로드해 보겠습니다. `id` 형식(즉, vocab이 `range(0, 61)`이고 pass move가 있는 게임을 필터링하므로 1부터 60까지 포함)과 `int` 형식(즉, 게임에 `A0`부터 `H7`까지의 move가 포함되어 있으므로 0부터 63까지 포함)으로 데이터를 로드하겠습니다.

In [ ]:
board_seqs_id = t.from_numpy(np.load(section_dir / "board_seqs_id_small.npy")).long()
board_seqs_square = t.from_numpy(np.load(section_dir / "board_seqs_square_small.npy")).long()

print(f"board_seqs_id: shape {tuple(board_seqs_id.shape)}, range: {board_seqs_id.min()} to {board_seqs_id.max()}")
print(
    f"board_seqs_square: shape {tuple(board_seqs_square.shape)}, range: {board_seqs_square.min()} to {board_seqs_square.max()}"
)

참고 - [GitHub readme](https://github.com/likenneth/othello_world)의 "Training Othello-GPT" 섹션에서 더 큰 데이터셋에 접근할 수 있습니다. Google Drive에서 데이터셋(합성 데이터 및 챔피언십 경기 데이터 모두 포함)을 다운로드할 수 있는 링크가 제공됩니다. 위의 코드를 사용하여 repo를 clone한 후, `data` 폴더에 저장하시면 됩니다.

## 유틸리티 만들기

이 시점에서 잠시 멈추고, 나중에 유용하게 사용할 집계 데이터를 수집하겠습니다. 여기에는 유효한 수(valid moves)의 tensor, 보드 상태(board states), 그리고 50개 게임에 걸친 모든 model activation의 cache가 포함됩니다 (실제로는 GPU memory에 무리 없이 들어갈 수 있는 최대한의 양을 확보하는 것이 좋습니다). 여러 게임에 대해 실험을 빠르게 실행할 수 있는 능력은 정말 편리합니다! 알고리즘 태스크를 다루는 작은 model의 큰 장점 중 하나는 바로 이러한 작업이 가능하다는 점입니다.

이 게임들을 **focus games**라고 부르겠습니다.

In [ ]:
def get_board_states_and_legal_moves(
    games_square: Int[Tensor, "n_games n_moves"],
) -> tuple[
    Int[Tensor, "n_games n_moves rows cols"],
    Int[Tensor, "n_games n_moves rows cols"],
    list,
]:
    """
    Returns the following:
        states:                 (n_games, n_moves, 8, 8): tensor of board states after each move
        legal_moves:            (n_games, n_moves, 8, 8): tensor of 1s for legal moves, 0s for
                                    illegal moves
        legal_moves_annotation: (n_games, n_moves, 8, 8): list containing strings of "o" for legal
                                    moves (for plotting)
    """
    # Create tensors to store the board state & legal moves
    n_games, n_moves = games_square.shape
    states = t.zeros((n_games, 60, 8, 8), dtype=t.int32)
    legal_moves = t.zeros((n_games, 60, 8, 8), dtype=t.int32)

    # Loop over each game, populating state & legal moves tensors after each move
    for n in range(n_games):
        board = utils.OthelloBoardState()
        for i in range(n_moves):
            board.umpire(games_square[n, i].item())
            states[n, i] = t.from_numpy(board.state)
            legal_moves[n, i].flatten()[board.get_valid_moves()] = 1

    # Convert legal moves to annotation
    legal_moves_annotation = np.where(to_numpy(legal_moves), "o", "").tolist()

    return states, legal_moves, legal_moves_annotation


num_games = 50

focus_games_id = board_seqs_id[:num_games]  # shape [50, 60]
focus_games_square = board_seqs_square[:num_games]  # shape [50, 60]
focus_states, focus_legal_moves, focus_legal_moves_annotation = get_board_states_and_legal_moves(focus_games_square)

print("focus states:", focus_states.shape)
print("focus_legal_moves", tuple(focus_legal_moves.shape))

# Plot the first 10 moves of the first game
utils.plot_board_values(
    focus_states[0, :10],
    title="Board states",
    width=1000,
    boards_per_row=5,
    board_titles=[f"Move {i}, {'white' if i % 2 == 1 else 'black'} to play" for i in range(1, 11)],
    text=np.where(to_numpy(focus_legal_moves[0, :10]), "o", "").tolist(),
)

이 focus game들을 위해 모델의 모든 activation과 logit을 캐싱해 두겠습니다.

In [ ]:
focus_logits, focus_cache = model.run_with_cache(focus_games_id[:, :-1].to(device))

print(focus_logits.shape)  # shape [num_games=50, n_ctx=59, d_vocab=61]

> #### 정의한 유용한 객체들의 요약
>
> 다음과 같은 객체들이 있습니다:
> 
> Models
> - `model`은 합법적인 Othello 수를 예측하도록 훈련된 8-layer autoregressive transformer입니다. vocab은 `range(0, 61)`이며, 여기서 0은 "pass"를 의미하고 다른 숫자들은 중앙 4개 칸을 제외한 60가지 가능한 수를 나타냅니다.
>
> All data
> - `board_seqs_id` (shape `(100k, 60)`)는 모든 100k 게임의 수들을 포함하고 있습니다 (token id 형태).
> - `board_seqs_square` (shape `(100k, 60)`)는 모든 100k 게임의 수들을 포함하고 있습니다 (int 형태).
> 
> Focus games data
> - `focus_games_id` (shape `(50, 60)`)는 50개 게임의 수들을 포함하고 있습니다 (token id 형태).
> - `focus_games_square` (shape `(50, 60)`)는 50개 게임의 수들을 포함하고 있습니다 (int 형태).
> - `focus_states` (shape `(50, 60, 8, 8)`)는 각 수 이후의 보드 상태를 포함하고 있습니다 (0 = 빈 칸, 1 = 흑돌, -1 = 백돌).
> - `focus_legal_moves` (shape `(50, 60, 8, 8)`)는 각 합법적인 수에 대해 1을, 불법적인 수에 대해 0을 포함하고 있습니다.
> - `focus_logits` (shape `(50, 59, 61)`)는 focus games에 대한 모델의 output logits를 포함하고 있습니다 - forward pass에서 마지막 수를 포함하지 않기 때문에 `59=model.cfg.n_ctx`이며, `61=model.cfg.d_vocab`은 "pass" 수와 60개의 플레이 가능한 칸들을 포함합니다.

## probe란 무엇입니까?

[MI Dynalist notes](https://dynalist.io/d/n2ZWtnoYHrU1s4vnFSAQ519J#q=probe)에서 발췌한 내용입니다:

> **[Probing](https://arxiv.org/pdf/1610.01644.pdf)**은 network activation 공간에서 특정 개념/feature에 해당하는 방향을 식별하는 기술입니다.
>
> 기본적으로, 해당 feature가 포함된 입력값들과 포함되지 않은 입력값들을 network에 많이 제공합니다. 그런 다음 특정 activation(예: layer 5의 출력)에 대해 이 두 집합을 구분하는 linear map을 학습시키면, activation 공간의 한 방향에 해당하는 1D linear map(**probe**)이 생성되며, 이는 해당 feature에 대응할 가능성이 높습니다.

Probe는 모델에 표현된 개념을 더 잘 이해하는 데 매우 유용한 도구가 될 수 있습니다. 하지만 염두에 두어야 할 두 가지 큰 주의사항이 있습니다:

1. Probe는 우리에게 방향을 제시하지만, 그 방향이 처음에 어떻게 모델에 들어오게 되었는지, 또는 모델이 그 방향을 어떻게 사용하고 있는지에 대한 인과적인 이야기(causal story)를 제공하지는 않습니다.
2. Probe(특히 nonlinear probe)는 그 표면 아래에 많은 계산 과정을 숨기고 있을 수 있습니다.

Othello를 분석한 원본 논문에서 저자들은 중요한 방향을 찾기 위해 nonlinear probing을 사용했습니다. 이는 모델이 기본적으로 정보를 linear한 방식으로 저장하며, 따라서 linear probe로 이에 접근할 수 있어야 한다는 근본적인 직관과 상충되는 방식이었습니다. 이번 실습에서는 linear probe를 사용할 것입니다.

## probe 사용하기

이 probe의 학습 과정은 다소 혼란스러웠으며, 다시 한다면 많은 부분을 다르게 처리했을 것입니다.

총 3가지의 서로 다른 probe 모드가 있었습니다:

- `full_linear_probe[0].shape = (d_model, 8, 8, 3)`은 흑색 차례, 즉 홀수 번째 수에 대해 학습되었습니다. 클래스는 `[empty, white, black]`입니다.
- `full_linear_probe[1].shape = (d_model, 8, 8, 3)`는 백색 차례, 즉 짝수 번째 수에 대해 학습되었습니다. 클래스는 `[empty, white, black]`입니다.
- `full_linear_probe[2].shape = (d_model, 8, 8, 3)`는 모든 수에 대해 학습되었습니다.

예를 들어, `full_linear_probe[0]`를 가져와 residual stream과 내적을 수행하면 `(8, 8, 3)` 형태의 tensor를 얻을 수 있으며, 이는 64개의 보드 칸 각각에 무엇이 포함되어 있는지에 대한 `8x8=64`개의 개별 예측을 나타냅니다 (즉, 이 tensor의 마지막 차원에 softmax를 적용하여 확률을 얻을 수 있습니다).

In [ ]:
full_linear_probe = t.load(section_dir / "main_linear_probe.pth", map_location=str(device), weights_only=True)

print(full_linear_probe.shape)

# Define indices along `full_linear_probe.shape[0]`, i.e. the different probe modes
black_to_play, white_to_play, _ = (0, 1, 2)
# Define indices along `full_linear_probe.shape[-1]`, i.e. the different classifications for each mode
empty, white, black = (0, 1, 2)

### 연습 문제 - probe cosine similarity 계산하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-20 minutes on this exercise.
> ```

`full_linear_probe[2]`은 성능이 그리 좋지 않기 때문에 많이 사용하지 않을 것입니다. 우리는 처음 두 가지 mode에 집중할 것입니다.

이 조사에서 발견된 핵심 결과는 **probe가 "black vs white"보다는 "상대방 것 vs 내 것"의 관점에서 방향을 학습한다**는 점입니다. 이 경우, 홀수 번째 수와 짝수 번째 수에 대한 probe는 서로 거의 반대 방향일 것으로 예상됩니다 ("상대방 것 vs 내 것"의 의미가 홀수 수와 짝수 수에서 서로 반대이기 때문입니다). [Here's a plot](https://res.cloudinary.com/lesswrong-2-0/image/upload/f_auto,q_auto/v1/mirroredImages/nmxzr2zsjNtjaHh7x/cuxy4pf353wazoq5emmn)를 시각화하면 실제로 이러한 결과가 나타남을 알 수 있습니다. 이는 각 probe mode의 64개 square 각각에 대한 "black minus white" 방향의 cosine similarity를 보여줍니다. 값이 -1에 가까운 off-diagonal stripe는 특정 square에 대한 "black minus white" 방향이 홀수 mode와 짝수 mode에서 취해졌을 때 거의 antipodal(대척점) 관계임을 나타냅니다 (참조 플롯에서 `(O)` 및 `(E)`로 표시됨).

위에서 정의된 `full_linear_probe`만을 사용하여 이 플롯을 재현해 보십시오. 이 텐서의 shape가 `(modes=3, d_model=512, rows=8, cols=8, options=3)`임을 기억하십시오. 처음 2개 mode는 black 차례 / white 차례이며, 세 가지 옵션은 empty / white / black입니다. `cosine_similarities` 텐서(shape는 `(64*2, 64*2)`이어야 함)로부터 플롯을 생성하는 코드는 이미 제공되었으므로, 여러분이 해야 할 일은 해당 텐서를 생성하는 것입니다.

In [ ]:
# YOUR CODE HERE - define `cosine_similarities`, then run the cell to create the plot

fig = px.imshow(
    to_numpy(cosine_similarities),
    title="Cosine Sim of B-W Linear Probe Directions by Cell",
    x=[f"{label} (O)" for label in BOARD_LABELS_2D.flatten()] + [f"{label} (E)" for label in BOARD_LABELS_2D.flatten()],
    y=[f"{label} (O)" for label in BOARD_LABELS_2D.flatten()] + [f"{label} (E)" for label in BOARD_LABELS_2D.flatten()],
    width=900,
    height=800,
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0.0,
)
fig.show()

<details><summary>솔루션</summary>

```python
# Get the "black vs white" probe directions for odd & even moves respectively
black_vs_white_dir_odd_moves = (
    full_linear_probe[black_to_play, :, :, :, black] - full_linear_probe[black_to_play, :, :, :, white]
)
black_vs_white_dir_even_moves = (
    full_linear_probe[white_to_play, :, :, :, black] - full_linear_probe[white_to_play, :, :, :, white]
)

# Flatten over (rows, cols) then concatenate them over this dimension
all_dirs = t.stack([black_vs_white_dir_odd_moves, black_vs_white_dir_even_moves])
all_dirs = einops.rearrange(all_dirs, "parity d_model rows cols -> d_model (parity rows cols)")

# Compute cosine similarities
all_dirs_normed = all_dirs / all_dirs.norm(dim=0, keepdim=True)
cosine_similarities = einops.einsum(
    all_dirs_normed,
    all_dirs_normed,
    "d_model mode_row_col_1, d_model mode_row_col_2 -> mode_row_col_1 mode_row_col_2",
)
```
</details>

### probe basis 변경하기

이제 probe 방향들이 매우 유사하다는 것을 확인했으므로, 이를 평균 내어 새로운 basis인 "상대방 vs 나의" basis( "흑 vs 백" basis가 아닌)로 probe를 생성해 보겠습니다. 예를 들어, 새로운 probe의 "상대방" 방향은 흑이 둘 차례일 때의 백 방향과 백이 둘 차례일 때의 흑 방향의 평균이 됩니다.

In [ ]:
linear_probe = t.stack(
    [
        # "Empty" direction = average of empty direction across probe modes
        full_linear_probe[[black_to_play, white_to_play], ..., [empty, empty]].mean(0),
        # "Theirs" direction = average of {x to play, classification != x} across probe modes
        full_linear_probe[[black_to_play, white_to_play], ..., [white, black]].mean(0),
        # "Mine" direction = average of {x to play, classification == x} across probe modes
        full_linear_probe[[black_to_play, white_to_play], ..., [black, white]].mean(0),
    ],
    dim=-1,
)

우리의 새로운 probe를 테스트하기 위해, focus games 중 game 0의 29번째 수에 적용해 보겠습니다. 이는 특이한 수이므로, 흑이 둘 차례입니다.

In [ ]:
def plot_probe_outputs(
    cache: ActivationCache,
    linear_probe: Tensor,
    layer: int,
    game_index: int,
    move: int,
    title: str = "Probe outputs",
):
    residual_stream = cache["resid_post", layer][game_index, move]
    probe_out = einops.einsum(residual_stream, linear_probe, "d_model, d_model row col options -> options row col")

    utils.plot_board_values(
        probe_out.softmax(dim=0),
        title=title,
        width=900,
        height=400,
        board_titles=["P(Empty)", "P(Their's)", "P(Mine)"],
        # text=BOARD_LABELS_2D,
    )


layer = 6
game_index = 0
move = 29

utils.plot_board_values(
    focus_states[game_index, move],
    title="Focus game states",
    width=400,
    height=400,
    text=focus_legal_moves_annotation[game_index][move],
)

plot_probe_outputs(
    focus_cache,
    linear_probe,
    layer,
    game_index,
    move,
    title="Probe outputs after move 29 (black to play)",
)

다시 layer 3으로 돌아가 보면, 모델이 이 시점에서 이미 꽤 괜찮은 board state representation을 가지고 있는 것으로 보이지만, 몇 가지 부족한 점이 있습니다 (가장 눈에 띄는 점은 C5와 특히 C6가 실제로는 black임에도 white라고 생각한다는 것입니다). 제 추측으로는 board state 계산 circuit이 아직 완전히 끝나지 않았으며 일종의 반복적인 추론을 수행하고 있는 것 같습니다. 만약 해당 cell들이 여러 번 점유되었다면, 그것이 점유된 그다음으로 이른 시점을 추적하기 위한 layer가 필요한 것일까요? 정확히는 모르겠지만, 이를 밝혀내는 것은 더 깊이 탐구하고 싶은 분들에게 아주 좋은 시작 프로젝트가 될 것입니다!

In [ ]:
layer = 3
game_index = 0
move = 29

plot_probe_outputs(
    focus_cache,
    linear_probe,
    layer,
    game_index,
    move,
    title="Probe outputs (layer 4) after move 29 (black to play)",
)

이제 한 단계 더 나아가 보겠습니다. representation이 완전히 뒤바뀌는 것을 확인할 수 있을 것입니다. 실제로 결과는 그렇게 나타납니다.

In [ ]:
layer = 4
game_index = 0
move = 30

utils.plot_board_values(
    focus_states[game_index, move],
    text=focus_legal_moves_annotation[game_index][move],
    title="Focus game states",
    width=400,
    height=400,
)
plot_probe_outputs(
    focus_cache,
    linear_probe,
    layer,
    game_index,
    move,
    title="Probe outputs (layer 4) after move 30 (white to play)",
)

이 경우 모델이 모서리 부분을 틀리게 처리한다는 점에 주목하십시오 (모서리가 비어 있는 것이 아니라 흰색이라고 잘못 생각합니다). 큰 문제는 아니지만 흥미로운 점입니다!

이 모델에서 왜 모서리가 다르게 처리될 수 있는지 이유를 생각해보실 수 있습니까?

<details>
<summary>힌트</summary>

한 가지 가능한 이유는 Othello의 규칙과 모서리가 갖는 특별한 중요성과 관련이 있습니다. 돌이 모서리에 놓이면 어떻게 됩니까?

</details>

<details>
<summary>한 가지 가능한 이유</summary>

Othello의 특징 중 하나는 모서리에 있는 돌은 절대 포위될 수 없으며, 따라서 한 번 놓이면 절대 색이 변하지 않는다는 것입니다. 아마도 모델이 효율성을 위해 모서리 부분에 대해 서로 다르고 덜 대칭적인 circuit를 갖도록 결정했을 수도 있습니다.

이 circuit를 찾아내는 것은 재미있는 보너스 연습이 될 것입니다!
</details>

### 정확도 계산하기

일화적인 예시를 통해 linear probe가 작동한다는 점을 충분히 납득하셨기를 바랍니다. 하지만 더 엄격하게 검증하기 위해, 50개의 게임 전체에 대해 정확도를 확인해 보겠습니다.

In [ ]:
# Create a tensor of "their vs mine" board states (by flipping even parities of the "focus_states" tensor)
focus_states_theirs_vs_mine = focus_states * (-1 + 2 * (t.arange(focus_states.shape[1]) % 2))[None, :, None, None]

# Convert values (0: empty, 1: theirs, -1: mine) to (0: empty, 1: theirs, 2: mine)
focus_states_theirs_vs_mine[focus_states_theirs_vs_mine == 1] = 2
focus_states_theirs_vs_mine[focus_states_theirs_vs_mine == -1] = 1

# Get probe values at layer 6, and compute the probe predictions
probe_out = einops.einsum(
    focus_cache["resid_post", 6],
    linear_probe,
    "game move d_model, d_model row col options -> game move row col options",
)
probe_predictions = probe_out.argmax(dim=-1)

# Get accuracy at odd, even & all moves (average over games & moves)
correct_middle_odd_answers = (probe_predictions.cpu() == focus_states_theirs_vs_mine[:, :-1])[:, 5:-5:2]
accuracies_odd = einops.reduce(correct_middle_odd_answers.float(), "game move row col -> row col", "mean")

correct_middle_even_answers = (probe_predictions.cpu() == focus_states_theirs_vs_mine[:, :-1])[:, 6:-5:2]
accuracies_even = einops.reduce(correct_middle_even_answers.float(), "game move row col -> row col", "mean")

correct_middle_answers = (probe_predictions.cpu() == focus_states_theirs_vs_mine[:, :-1])[:, 5:-5]
accuracies = einops.reduce(correct_middle_answers.float(), "game move row col -> row col", "mean")

# Plot accuracies
utils.plot_board_values(
    1 - t.stack([accuracies_odd, accuracies_even, accuracies], dim=0),
    title="Average Error Rate of Linear Probe",
    width=1000,
    height=400,
    board_titles=["Black to play", "White to play", "All moves"],
    zmax=0.25,
    zmin=-0.25,
)

일화적으로 관찰했던 것처럼, 모서리 근처에서 probe의 성능이 더 떨어진다는 것을 확인할 수 있습니다.

## probe를 이용한 개입(Intervening)

linear probe의 정말 흥미로운 결과 중 하나는 residual stream에서 해석 가능한 방향(interpretable directions)들의 집합을 얻을 수 있다는 점입니다! 이를 통해 모델의 representation을 해석할 수 있을 뿐만 아니라, 모델의 추론 과정에 개입(intervene)할 수도 있습니다. 이는 모델을 *정말로* 이해할 수 있다면, 모델의 동작을 정밀하고 세부적으로 제어할 수 있다는 좋은 개념 증명(proof of concept)이 됩니다.

첫 번째 단계는 probe를 의미 있는 방향들로 변환하는 것입니다. 각 square의 probe는 3개의 벡터를 가지지만, logit은 translation invariant한 softmax로 들어가므로 실제로는 두 개의 자유도(degrees of freedom)만 가집니다. 이를 두 개의 벡터로 변환하는 자연스러운 방법은, "이 cell이 비어 있는가"에 대한 방향을 주는 `blank - (mine + theirs)/2`와 "비어 있다는 조건 하에, 이것이 내 색상인가 아니면 상대의 색상인가"에 대한 방향을 주는 `mine - theirs`를 취하는 것입니다.

<details>
<summary>도움말 - 이 부분이 혼란스럽습니다.</summary>

간접 목적어 식별(indirect object identification) 연습을 해보셨다면, 이는 logit 출력에서 `"John" - "Mary"` 방향을 살펴보는 것과 비슷합니다. 즉, 두 logit의 차이를 구함으로써 이 두 옵션 사이의 log-likelihood ratio를 얻는 것입니다.

두 개보다 많은 서로 다른 logit을 다룰 때는 nonlinearity가 복잡해지기 때문에 원칙적으로는 약간 덜 정교합니다. 하지만 `blank - (mine + theirs)/2`를 사용하는 것은 여전히 꽤 합리적인 지표입니다:

* `mine`와 `theirs`에 대해 대칭적입니다.
* translation invariant합니다 (즉, `blank`, `mine`, `theirs` 모두에 상수 `c`를 더해도 결과는 변하지 않습니다).
* `blank`를 일정량 `c`만큼 증가시키고 나머지 둘을 그대로 유지하면, 이 지표 또한 `c`만큼 증가합니다.

`mine - theirs` 방향이 더 원칙적입니다.

</details>

단일한 의미 있는 방향을 갖는 것은 매우 중요합니다. 그래야 특정 feature를 해석하거나 그 feature에 개입할 수 있기 때문입니다. 원래의 세 방향은 하나의 자유도를 가지므로, 각 방향은 그 자체로는 임의적입니다.

### 연습 문제 - probe direction 정의하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 5-10 minutes on this exercise.
> 
> The exercise isn't conceptually hard, understanding what the objects in question represent is the important part.
> ```

위에서 주어진 두 방향을 가리키는 tensor `blank_probe`와 `my_prob`를 정의하십시오.

<details>
<summary>도움말 - 여기서 선형 결합(linear combination)을 정확히 어떻게 수행해야 하는지 헷갈립니다.</summary>

사용자의 `linear_probe` tensor는 shape이 `[d_model, row, col, options]`이며, 선택지는 각각 `(blank, theirs, mine)`입니다. 각 tensor가 특정 개념에 대한 probe 방향이 되는 shape `[d_model, row, col]`의 새로운 tensor 2개를 생성하고자 합니다 (`blank - (mine + theirs)/2`로 정의하는 "blank" 개념과 `mine - theirs`로 정의하는 "mine vs theirs" 개념입니다). 따라서 이 두 tensor를 생성하기 위해 `linear_probe`의 마지막 dimension을 따라 슬라이싱(slice)해야 합니다.

</details>

In [ ]:
# YOUR CODE HERE - define `blank_probe` and `my_probe`, from linear combinations of `linear_probe`

tests.test_my_probes(blank_probe, my_probe, linear_probe)

<details><summary>솔루션</summary>

```python
# blank(0) - (theirs(1) + mine(2))/2
blank_probe = linear_probe[..., 0] - linear_probe[..., 1] * 0.5 - linear_probe[..., 2] * 0.5
# mine(2) - theirs(1)
my_probe = linear_probe[..., 2] - linear_probe[..., 1]
```
</details>

이제 probe가 작동하므로, 실제로 어떻게 동작하는지 확인해 보겠습니다!

0번째 게임의 20번째 수를 예시로 사용하겠습니다:

In [ ]:
game_index = 0
move = 20

# Plot board state
utils.plot_board_values(
    focus_states[game_index, move],
    title="Focus game states",
    width=400,
    height=400,
    text=focus_legal_moves_annotation[game_index][move],
)

# Plot model predictions
logprobs = t.full(size=(8, 8), fill_value=-13.0, device=device)
logprobs.flatten()[ALL_SQUARES] = focus_logits[game_index, move].log_softmax(dim=-1)[1:]
utils.plot_board_values(logprobs, title=f"Logprobs after move {move}", width=450, height=400)

이제 `F4`이 검은색에서 흰색으로 바뀌면 게임 상태(즉, 흰색에게 어떤 수가 가능한지)가 어떻게 변할까요?

<details>
<summary>힌트</summary>

한 가지 수는 가능해지고, 한 가지 수는 불가능해집니다.

</details>

<details>
<summary>정답</summary>

- `G4`은 더 이상 4열의 검은색 기물 수직선을 둘러싸지 않게 되므로 불가능해집니다.
- `D2`는 이제 `E3`에 있는 단일 검은색 기물을 대각선으로 둘러싸게 되므로 가능해집니다.

</details>

`OthelloBoardState` 클래스를 사용하여 이를 확인해 보겠습니다:

In [ ]:
cell_r = 5
cell_c = 4
print(f"Flipping the color of cell {'ABCDEFGH'[cell_r]}{cell_c}")

board = utils.OthelloBoardState()
board.update(focus_games_square[game_index, : move + 1].tolist())
valid_moves = board.get_valid_moves()
flipped_board = copy.deepcopy(board)
flipped_board.state[cell_r, cell_c] *= -1
flipped_legal_moves = flipped_board.get_valid_moves()

newly_legal = [utils.square_to_label(move) for move in flipped_legal_moves if move not in valid_moves]
newly_illegal = [utils.square_to_label(move) for move in valid_moves if move not in flipped_legal_moves]
print("newly_legal", newly_legal)
print("newly_illegal", newly_illegal)

이제 "내 색깔 vs 상대방 색깔" 방향을 사용하여 모델의 residual stream에 개입할 수 있습니다. 저는 layer 4 이후에 개입했을 때 가장 좋은 결과를 얻었습니다. 이것은 **linear intervention**입니다. 즉, residual stream의 단일 차원만 변경하고 나머지는 그대로 유지하는 방식입니다. 이는 상당히 간단한 개입이며, 이것이 실제로 작동한다는 점이 놀랍습니다!

저는 주어진 방향의 현재 좌표를 가져와 이를 부정(negate)한 다음, `scale`라고 불리는 하이퍼파라미터를 곱하는 다소 투박한 기법을 적용합니다 (1에서 8 사이의 scale이 가장 잘 작동하는 경향이 있으며, 너무 작으면 충분하지 않고 너무 크면 시스템이 망가지는 경향이 있습니다). 저는 이를 최적화하기 위해 크게 노력하지 않았으며, 충분히 개선될 수 있다고 확신합니다! 예를 들어, 모델의 좌표를 scaling 하는 대신 상수로 대체하는 방식 등이 있습니다. 또한 최적의 scale 파라미터가 무엇인지, 또는 어떤 상황에서 어떤 파라미터가 가장 잘 작동하는지에 대해서는 깊이 파고들지 않았습니다. 아마도 서로 다른 cell들이 그들의 world model에서 서로 다른 activation scale을 가지고 있어 서로 다른 동작이 필요할 가능성이 있습니다!

### 연습 문제 - `apply_scale` 함수 정의하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-20 minutes on this exercise.
> ```

residual stream 값과 관련 hook point를 인자로 받아, 위에서 설명한 방식대로 수정된 residual stream 버전을 반환하는 함수를 정의하십시오.

명확히 하자면, 특정 square(아래에서는 `flip_dir`이라 부릅니다)에 대한 probe의 flip 방향을 $\vec{v}$로 정의한다면, 우리가 관심 있는 `pos=20`에서의 residual stream을 다음과 같은 벡터로 쓸 수 있습니다:

$$
\text{resid}_{20} = \alpha \times \vec{v} + \beta \times \vec{w}
$$

여기서 $\vec{w}$은 $v$에 직교하는 어떤 벡터입니다. 우리는 이 위치의 residual stream을 다음과 같이 변경하고자 합니다:

$$
\text{resid}_{20} = - \text{ scale} \times \alpha \times \vec{v} + \beta \times \vec{w}
$$

벡터 $\vec{v}$를 정규화하는 것을 잊지 마십시오!

In [ ]:
def apply_scale(
    resid: Float[Tensor, "batch seq d_model"],
    flip_dir: Float[Tensor, "d_model"],
    scale: int,
    pos: int,
) -> Float[Tensor, "batch seq d_model"]:
    """
    Returns a version of the residual stream, modified by the amount `scale` in the
    direction `flip_dir` at the sequence position `pos`, in the way described above.
    """
    raise NotImplementedError()


tests.test_apply_scale(apply_scale)

<details><summary>솔루션</summary>

```python
def apply_scale(
    resid: Float[Tensor, "batch seq d_model"],
    flip_dir: Float[Tensor, "d_model"],
    scale: int,
    pos: int,
) -> Float[Tensor, "batch seq d_model"]:
    """
    Returns a version of the residual stream, modified by the amount `scale` in the
    direction `flip_dir` at the sequence position `pos`, in the way described above.
    """
    flip_dir_normed = flip_dir / flip_dir.norm()

    alpha = resid[0, pos] @ flip_dir_normed
    resid[0, pos] -= (scale + 1) * alpha * flip_dir_normed

    return resid
```
</details>

이제 아래 코드를 실행하여 intervention의 결과를 확인할 수 있습니다. 모델의 예측이 변경되어, `D2`을 legal로, `G4`을 illegal로 예측하기 시작하는 것을 볼 수 있습니다.

In [ ]:
flip_dir = my_probe[:, cell_r, cell_c]

logprobs_flipped = []
layer = 4
scales = [0, 1, 2, 4, 8, 16]

# Iterate through scales, generate a new facet plot for each possible scale
for scale in scales:
    # Hook function which will perform flipping in the "F4 flip direction"
    def flip_hook(resid: Float[Tensor, "batch seq d_model"], hook: HookPoint):
        return apply_scale(resid, flip_dir, scale, move)

    # Calculate the logits for the board state, with the `flip_hook` intervention (note that we only
    # need to use :move+1 as input, because of causal attention)
    flipped_logits = model.run_with_hooks(
        focus_games_id[game_index : game_index + 1, : move + 1],
        fwd_hooks=[
            (get_act_name("resid_post", layer), flip_hook),
        ],
    ).log_softmax(dim=-1)[0, move]

    logprobs_flipped_single = t.zeros((64,), dtype=t.float32, device=device) - 10.0
    logprobs_flipped_single[ALL_SQUARES] = flipped_logits.log_softmax(dim=-1)[1:]
    logprobs_flipped.append(logprobs_flipped_single)

flip_state_big = t.stack(logprobs_flipped)
logprobs_repeated = einops.repeat(logprobs.flatten(), "d -> b d", b=6)
color = t.zeros((len(scales), 64)) + 0.2
color[:, utils.to_square(newly_legal)] = 1
color[:, utils.to_square(newly_illegal)] = -1

scatter(
    y=logprobs_repeated,
    x=flip_state_big,
    title=f"Original vs Flipped {utils.square_to_label(8 * cell_r + cell_c)} at Layer {layer}",
    xaxis="Flipped",
    yaxis="Original",
    hover=[f"{r}{c}" for r in "ABCDEFGH" for c in range(8)],
    facet_col=0,
    facet_labels=[f"Translate by {i}x" for i in scales],
    color=color,
    color_name="Newly Legal",
    color_continuous_scale="Geyser",
    width=1400,
)

<details>
<summary>도움말 - 이 그림 / 이 방법론에 대해 여전히 혼란스럽습니다.</summary>

스칼라 `N`에 대해 "`Nx`으로 변환한다"는 것은, F4 square에 대한 `theirs - mine` probe 방향의 residual stream 성분 `x`를 가져와서 이를 `Nx`로 대체하는 것을 의미합니다.

산점도는 `x`의 서로 다른 값들에 대해 `-1x` (원본)과 `Nx` (반전)를 비교합니다. 예를 들어, 첫 번째 facet plot은 probe 방향의 residual stream 성분이 삭제되었을 때 어떤 일이 발생하는지 보여줍니다.

스케일 팩터가 증가함에 따라 (처음에는 다른 square들에 대한 예측에는 큰 변화 없이) `G4`과 `D2`에 대한 모델의 예측이 변한다는 사실(`G4`은 "더 불법적이 되고", `D2`는 "더 합법적이 됨")은 우리의 인과적 개입(causal intervention)이 유효하다는 증거입니다. 다시 말해, 우리의 linear probe `my_probe`에 의해 발견된 방향은 어떤 의미에서 모델의 `theirs - mine` 방향을 나타내며, 이 방향이 모델의 downstream에서 사용되고 있다는 것입니다.
</details>

<br>

> #### 우리가 정의한 유용한 객체들의 최종 요약
>
> 다음과 같은 것들이 있습니다:
> 
> Models
> - `model`은 합법적인 Othello 수를 예측하도록 훈련된 8-layer autoregressive transformer입니다. vocab은 `range(0, 61)`이며, 여기서 0은 "pass"이고 다른 숫자들은 중앙 4개 칸을 제외한 60가지 가능한 수를 나타냅니다.
> - `full_linear_probe.shape = (mode=3, d_model=512, row=8, col=8, options=3)`는 "black/white basis"의 probe입니다 (mode는 black/white/both to play이며, option은 empty/white/black입니다).
> - `linear_probe.shape = (d_model=512, row=8, col=8, options=3)`은 "theirs/mine basis"의 probe입니다 (option은 empty/theirs/mine입니다).
> - `blank_probe`와 `my_probe`는 모두 `(d_model=512, row=8, col=8)`의 shape을 가지며, `linear_probe` option들의 선형 결합으로 생성되었습니다.
> 
> All data
> - `board_seqs_id.shape = (100k, 60)`은 모든 100k 게임의 수들을 포함하고 있습니다 (token id 형태).
> - `board_seqs_square.shape = (100k, 60)`는 모든 100k 게임의 수들을 포함하고 있습니다 (int 형태).
> 
> Focus games data
> - `focus_games_id.shape = (50, 60)`은 50개 게임의 수들을 포함하고 있습니다 (token id 형태).
> - `focus_games_square.shape = (50, 60)`은 50개 게임의 수들을 포함하고 있습니다 (int 형태).
> - `focus_states.shape = (50, 60, 8, 8)`는 각 수 이후의 board state를 포함하고 있습니다 (0 = empty, 1 = black, -1 = white).
> - `focus_legal_moves.shape = (50, 60, 8, 8)`은 각 합법적인 수에 대해 1을, 불법적인 수에 대해 0을 포함하고 있습니다.
> - `focus_logits.shape = (50, 59, 61)`는 focus games에 대한 모델의 output logit을 포함하고 있습니다 (fwd pass에서 마지막 수를 제외했으므로 59, vocab size가 61이므로 61 - 60가지 수 + pass를 위한 1개가 있습니다).

# 2️⃣ 모듈형 회로 찾기

> ##### 학습 목표
>
> - 여러 레이어에 걸쳐 linear probe를 사용하는 방법을 배웁니다.
> - 모델에 대한 가설을 테스트하기 위해 특정 시퀀스 위치에서 activation patching을 적용합니다.
> - 뉴런이 입력 및 출력 weight의 관점에서 어떻게 특징지어질 수 있는지 이해합니다.

## 레이어 전반에 걸친 Probing

probe의 입력은 6개 레이어에 걸쳐 residual stream에 누적됩니다. residual stream은 이전의 각 head와 neuron 출력의 합입니다. 따라서 우리는 어떤 이전 모델 컴포넌트가 전체 probe 계산에 가장 많이 기여하는지 분석할 수 있으며, 이를 통해 world model computing circuit의 끝을 식별할 수 있습니다.

게임 1의 20번째 수를 분석해 보겠습니다. 여기서 우리는 layer 6 이후에 probe가 완벽한 정확도를 보이는 것을 확인할 수 있습니다.

In [ ]:
layer = 6
game_index = 1
move = 20

utils.plot_board_values(
    focus_states[game_index, move],
    text=focus_legal_moves_annotation[game_index][move],
    title=f"Focus game #{game_index}, board after move {move}",
    width=400,
    height=400,
)

plot_probe_outputs(focus_cache, linear_probe, layer, game_index, move, title=f"Probe outputs (layer {layer})")

이제 attention과 MLP layer가 `my_probe` 방향에 기여하는 정도를 그래프로 그려보겠습니다. 놀랍게도, 방금 상대방이 가져간 세로 줄무늬에는 MLP layer가 중요하게 작용하지만, 그 외의 대부분은 attention layer에 의해 수행되는 것으로 보입니다.

### 연습 문제 - attn 및 mlp 기여도 계산하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 10-20 minutes on this exercise.
> This is an important exercise to be able to do - logit attribution is one of the most important tools in the interpretability toolbox.
> ```

아래에서 `attn_contributions`와 `mlp_contributions`를 정의해야 합니다. 각 레이어(레이어 `0`부터 `layer`까지 포함)에서 residual stream에 *기록된* 벡터들과, 앞서 계산한 "my vs their" probe direction(즉, `my_probe`)의 batched dot product를 구하여 이를 수행하십시오.

참고로, 우리는 누적된 residual stream이 아니라 각 컴포넌트가 probe direction에 주는 한계 기여도(marginal contribution)를 찾고 있습니다. 이는 어떤 컴포넌트가 출력에 강한 영향을 미치는지 확인하기 위해서입니다.

<details>
<summary>힌트 - 어떤 activation 이름을 사용해야 할까요?</summary>

`attn_out`와 `mlp_out`을 사용해야 합니다.

이 두 가지 기여도를 각각 계산하려면 activation과 probe 사이의 `einsum`을 구해야 합니다.
</details>

<details>
<summary>힌트 - 어떤 차원에 대해 곱셈을 수행해야 할까요?</summary>

`my_probe`의 shape은 `(d_model=512, rows=8, cols=8)`입니다. 특정 레이어의 residual stream과 `my_probe`을 `d_model` 차원에 대해 곱해야 합니다(probe가 residual stream의 방향을 나타내기 때문입니다). 결과값(특정 게임 인덱스와 수에 대해)의 shape은 `(rows=8, cols=8)`이 되며, 이는 모델의 해당 컴포넌트가 probe direction으로 residual stream에 기록한 양을 나타냅니다.
</details>

In [ ]:
def calculate_attn_and_mlp_probe_score_contributions(
    focus_cache: ActivationCache,
    probe: Float[Tensor, "d_model rows cols"],
    layer: int,
    game_index: int,
    move: int,
) -> tuple[Float[Tensor, "layers rows cols"], Float[Tensor, "layers rows cols"]]:
    # YOUR CODE HERE - define `attn_contributions` and `mlp_contributions` using the cache & probe

    return (attn_contributions, mlp_contributions)


layer = 6
attn_contributions, mlp_contributions = calculate_attn_and_mlp_probe_score_contributions(
    focus_cache, my_probe, layer, game_index, move
)

utils.plot_board_values(
    mlp_contributions,
    title=f"MLP Contributions to my vs their (game #{game_index}, move {move})",
    board_titles=[f"Layer {i}" for i in range(layer + 1)],
    width=1400,
    height=340,
)
utils.plot_board_values(
    attn_contributions,
    title=f"Attn Contributions to my vs their (game #{game_index}, move {move})",
    board_titles=[f"Layer {i}" for i in range(layer + 1)],
    width=1400,
    height=340,
)

<details><summary>솔루션</summary>

```python
def calculate_attn_and_mlp_probe_score_contributions(
    focus_cache: ActivationCache,
    probe: Float[Tensor, "d_model rows cols"],
    layer: int,
    game_index: int,
    move: int,
) -> tuple[Float[Tensor, "layers rows cols"], Float[Tensor, "layers rows cols"]]:
    attn_contributions = einops.einsum(
        t.stack([focus_cache["attn_out", l][game_index, move] for l in range(layer + 1)]),
        probe,
        "layers d_model, d_model rows cols -> layers rows cols",
    )
    mlp_contributions = einops.einsum(
        t.stack([focus_cache["mlp_out", l][game_index, move] for l in range(layer + 1)]),
        probe,
        "layers d_model, d_model rows cols -> layers rows cols",
    )

    return (attn_contributions, mlp_contributions)
```
</details>

다음으로, 전체 probe 점수(즉, layer `layer` 끝에서의 누적된 residual stream으로부터 얻은 점수)를 반환하고 시각화해야 합니다. 코드는 바로 위에서 작성한 코드와 유사해야 하지만, MLP나 attention의 기여분 대신 layer `layer`까지를 포함한 residual stream의 값을 사용합니다.

In [ ]:
def calculate_accumulated_probe_score(
    focus_cache: ActivationCache,
    probe: Float[Tensor, "d_model rows cols"],
    layer: int,
    game_index: int,
    move: int,
) -> Float[Tensor, "layers rows cols"]:
    # YOUR CODE HERE - define `attn_contributions` and `mlp_contributions` using the cache & probe

    return residual_stream_score


residual_stream_score = calculate_accumulated_probe_score(focus_cache, my_probe, layer, game_index, move)

utils.plot_board_values(
    residual_stream_score,
    title=f"Residual stream probe values for 'my vs their' (game #{game_index}, move {move})",
    board_titles=[f"Layer {i}" for i in range(layer + 1)],
    width=1400,
    height=340,
)

<details><summary>솔루션</summary>

```python
def calculate_accumulated_probe_score(
    focus_cache: ActivationCache,
    probe: Float[Tensor, "d_model rows cols"],
    layer: int,
    game_index: int,
    move: int,
) -> Float[Tensor, "layers rows cols"]:
    residual_stream_score = einops.einsum(
        t.stack([focus_cache["resid_post", l][game_index, move] for l in range(layer + 1)]),
        probe,
        "layer d_model, d_model rows cols -> layer rows cols",
    )

    return residual_stream_score
```
</details>

### 연습 문제 - "blank" probe에 대해 반복하십시오

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend ~5-10 minutes on this exercise - it's just repeating previous code.
> ```

`my_probe` 대신 `blank_probe`을 사용하여 정확히 동일한 plot을 그리십시오. 무엇을 발견하셨으며, 그 이유는 무엇입니까?

In [ ]:
# YOUR CODE HERE - repeat the results for `blank_probe`, and interpret them

<details>
<summary>솔루션 코드</summary>

```python
attn_contributions, mlp_contributions = calculate_attn_and_mlp_probe_score_contributions(
    focus_cache, blank_probe, layer, game_index, move
)
utils.plot_board_values(
    mlp_contributions,
    title=f"MLP Contributions to blank probe (game #{game_index}, move {move})",
    board_titles=[f"Layer {i}" for i in range(layer + 1)],
    width=1400,
    height=340,
)
utils.plot_board_values(
    attn_contributions,
    title=f"Attn Contributions to blank probe (game #{game_index}, move {move})",
    board_titles=[f"Layer {i}" for i in range(layer + 1)],
    width=1400,
    height=340,
)

residual_stream_score = calculate_accumulated_probe_score(focus_cache, blank_probe, layer, game_index, move)
utils.plot_board_values(
    residual_stream_score,
    title=f"Residual stream probe values for 'blank' (game #{game_index}, move {move})",
    board_titles=[f"Layer {i}" for i in range(layer + 1)],
    width=1400,
    height=340,
)
```

</details>

<details>
<summary>토론</summary>

"이 셀이 비어 있는가 아니면 아닌가?"라는 알고리즘은 구현하기 매우 쉽습니다. 단지 해당 셀이 게임 도중 어느 시점에 플레이되었는지만 확인하면 됩니다. 셀이 검은색인지 흰색인지 판별하는 것과 달리, 이는 piece-flipping 규칙에 대한 이해를 필요로 하지 않습니다.

모델은 0번째 attention layer 이후로 이에 대해 꽤 잘 이해하고 있는 것으로 보이며, 적어도 이를 이해하기 위한 모든 요소를 갖추고 있습니다 (심지어 더 큰 크기의 기여는 예를 들어 후반부 MLP layer에서 발생합니다).

</details>

## neuron weight 읽기

linear probe를 가짐으로써 얻는 또 다른 멋진 결과는 residual stream 내에 해석 가능한 방향(direction)들의 집합을 갖게 된다는 점입니다. 이는 probe가 제공하는 방향들의 집합을 통해, 어떤 neuron의 input 및 output weight의 의미를 읽어낼 수 있음을 의미합니다.

저의 초기 조사에서 흥미로워 보였던 neuron L5N1393부터 시작하겠습니다.

먼저, probe의 정규화된 버전을 계산하겠습니다 (정규화는 `d_model` 차원을 기준으로 수행됩니다. 즉, `blank_probe_normalized`은 `(d_model, row, col)` 형태를 가지며, `[:, i, j]`번째 항목은 보드의 `i, j`번째 cell에 대한 residual stream probe 방향입니다).

In [ ]:
# Scale the probes down to be unit norm per cell
blank_probe_normalised = blank_probe / blank_probe.norm(dim=0, keepdim=True)
my_probe_normalised = my_probe / my_probe.norm(dim=0, keepdim=True)

# Set the center blank probes to 0, since they're never blank so the probe is meaningless
blank_probe_normalised[:, [3, 3, 4, 4], [3, 4, 3, 4]] = 0.0

### 연습 문제 - 뉴런 입력 가중치 계산하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You shouldn't spend more than 10-25 minutes on this exercise.
> ```

아래의 함수 `calculate_neuron_input_weights`은 `layer`와 `neuron`을 인자로 받습니다. 이 함수는 `(row, col)` 형태의 tensor를 반환해야 하며, 여기서 `[i, j]`번째 항목은 이 뉴런의 입력 가중치를 보드의 `i, j`번째 셀에 해당하는 probe 방향으로 투영한 값입니다.

함수 `calculate_neuron_output_weights`은 매우 유사하지만, 뉴런의 출력 가중치를 probe 방향으로 투영한 값을 반환합니다.

뉴런의 입력 및 출력 가중치에 대해 이야기할 때, 우리는 다음과 같은 분해를 의미한다는 점을 상기하십시오:

$$
\begin{align*}
f(x) &= f(x^T W^{in}) W^{out} \\
&= \sum_n f(x^T W^{in}_{[:, n]}) W^{out}_{[n, :]}
\end{align*}
$$

여기서 $x$는 residual stream의 벡터이고, $W^{in}$은 입력 가중치 행렬, $W^{out}$은 출력 가중치 행렬, $f$는 activation 함수이며, $\sum_n$은 뉴런들에 대한 합계를 나타냅니다.

먼저 주어진 뉴런에 대해 (정규화된) 벡터 $W^{in}_{[:, n]}$과 $W^{out}_{[n, :]}$을 반환하는 헬퍼 함수 `get_w_in`와 `get_w_out`를 작성합니다. 그 다음, 이 헬퍼 함수를 사용하여 `calculate_neuron_input_weights`을 구현합니다.

왜 probe 방향으로 투영하기 전에 정규화를 하나요? 그 이유는 우리가 스케일 인자(scale factor)에는 관심이 없기 때문입니다. 출력 벡터의 크기를 두 배로 늘리고 그에 대응하는 입력 벡터의 크기를 절반으로 줄여도 (bias를 무시한다면) 결과는 동일합니다. 대신, 우리는 모델 가중치의 입력 방향이 residual stream에서 찾은 probe 방향과 얼마나 일치하는지에 관심이 있습니다. probe 또한 정규화했다는 사실은 우리가 벡터들의 **cosine similarity**를 플로팅하게 된다는 것을 의미합니다.

*참고 - 모델의 가중치에 인덱싱하고 연산을 수행할 때, 필요한 경우 `clone()`와 `detach()`을 사용하는 것을 잊지 마십시오. 모델의 가중치를 수정하고 싶지 않기 때문에 `clone()`을 사용하며, 모델 가중치를 통해 gradient를 계산하고 싶지 않기 때문에 `detach()`를 사용합니다.*

In [ ]:
def get_w_in(
    model: HookedTransformer,
    layer: int,
    neuron: int,
    normalize: bool = False,
) -> Float[Tensor, "d_model"]:
    """
    Returns the input weights for the given neuron.

    If normalize is True, the weight is normalized to unit norm.
    """
    raise NotImplementedError()


def get_w_out(
    model: HookedTransformer,
    layer: int,
    neuron: int,
    normalize: bool = False,
) -> Float[Tensor, "d_model"]:
    """
    Returns the output weights for the given neuron.

    If normalize is True, the weight is normalized to unit norm.
    """
    raise NotImplementedError()


def calculate_neuron_input_weights(
    model: HookedTransformer, probe: Float[Tensor, "d_model row col"], layer: int, neuron: int
) -> Float[Tensor, "rows cols"]:
    """
    Returns tensor of the input weights for the given neuron, at each square on the board, projected
    along the corresponding probe directions.

    Assume probe directions are normalized. You should also normalize the model weights.
    """
    raise NotImplementedError()


def calculate_neuron_output_weights(
    model: HookedTransformer, probe: Float[Tensor, "d_model row col"], layer: int, neuron: int
) -> Float[Tensor, "rows cols"]:
    """
    Returns tensor of the output weights for the given neuron, at each square on the board,
    projected along the corresponding probe directions.

    Assume probe directions are normalized. You should also normalize the model weights.
    """
    raise NotImplementedError()


tests.test_calculate_neuron_input_weights(calculate_neuron_input_weights, model)
tests.test_calculate_neuron_output_weights(calculate_neuron_output_weights, model)

<details><summary>솔루션</summary>

```python
def get_w_in(
    model: HookedTransformer,
    layer: int,
    neuron: int,
    normalize: bool = False,
) -> Float[Tensor, "d_model"]:
    """
    Returns the input weights for the given neuron.

    If normalize is True, the weight is normalized to unit norm.
    """
    w_in = model.W_in[layer, :, neuron].detach().clone()
    if normalize:
        w_in /= w_in.norm(dim=0, keepdim=True)
    return w_in


def get_w_out(
    model: HookedTransformer,
    layer: int,
    neuron: int,
    normalize: bool = False,
) -> Float[Tensor, "d_model"]:
    """
    Returns the output weights for the given neuron.

    If normalize is True, the weight is normalized to unit norm.
    """
    w_out = model.W_out[layer, neuron, :].detach().clone()
    if normalize:
        w_out /= w_out.norm(dim=0, keepdim=True)
    return w_out


def calculate_neuron_input_weights(
    model: HookedTransformer, probe: Float[Tensor, "d_model row col"], layer: int, neuron: int
) -> Float[Tensor, "rows cols"]:
    """
    Returns tensor of the input weights for the given neuron, at each square on the board, projected
    along the corresponding probe directions.

    Assume probe directions are normalized. You should also normalize the model weights.
    """
    w_in = get_w_in(model, layer, neuron, normalize=True)

    return einops.einsum(w_in, probe, "d_model, d_model row col -> row col")


def calculate_neuron_output_weights(
    model: HookedTransformer, probe: Float[Tensor, "d_model row col"], layer: int, neuron: int
) -> Float[Tensor, "rows cols"]:
    """
    Returns tensor of the output weights for the given neuron, at each square on the board,
    projected along the corresponding probe directions.

    Assume probe directions are normalized. You should also normalize the model weights.
    """
    w_out = get_w_out(model, layer, neuron, normalize=True)

    return einops.einsum(w_out, probe, "d_model, d_model row col -> row col")
```
</details>

이제 neuron `1393`을 더 자세히 살펴보겠습니다. 이 neuron이 어떤 역할을 하고 있는지 해석해 보시겠습니까?

In [ ]:
layer = 5
neuron = 1393

w_in_L5N1393_blank = calculate_neuron_input_weights(model, blank_probe_normalised, layer, neuron)
w_in_L5N1393_my = calculate_neuron_input_weights(model, my_probe_normalised, layer, neuron)

utils.plot_board_values(
    t.stack([w_in_L5N1393_blank, w_in_L5N1393_my]),
    title=f"Input weights in terms of the probe for neuron L{layer}N{neuron}",
    board_titles=["Blank In", "My In"],
    width=650,
    height=380,
)

<details>
<summary>정답 - 이 neuron이 무엇을 하는지</summary>

이것은 `(C0==BLANK) & (D1==THEIRS) & (E2==MINE)`을 나타내는 것으로 보입니다. 다시 말해, 이 세 가지 조건이 모두 충족될 때 가장 강하게 activation됩니다.

이는 모델에게 유용한데, 이 세 가지 조건이 모두 충족되면 `C0`가 합법적인 수(legal move)가 되기 때문입니다 (`D1`를 뒤집기 때문입니다).

</details>

### 연습 문제 - 가설 테스트 (출력 동작)

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-20 minutes on this exercise.
> ```

위의 그래프들은 뉴런의 입력 동작(즉, 무엇이 뉴런을 활성화시키는가)에 대한 증거이지만, 뉴런이 활성화되었을 때 어떤 영향을 미치는지에 대한 출력 동작은 아직 테스트하지 않았습니다. 이는 우리의 가설과 일치하는지 확인하기 위함입니다. `plot_board_values` 함수를 사용하여 해당 뉴런에 대한 가설을 테스트하기 위한 특정 그래프를 그려보시기 바랍니다. 어떤 그래프를 그려야 할지 찾아내는 것 또한 연습 문제로 남겨두겠습니다!

In [ ]:
# YOUR CODE HERE - create a plot to test your prediction

<details>
<summary>정답 - 어떤 그래프를 그려야 하는가</summary>

이 neuron은 `C0`이 legal함을 예측하기 위해 활성화될 것으로 기대됩니다. 다시 말해, 이 neuron의 output weights를 unembedding matrix를 통해 매핑하면, `C0`에 대해서는 값이 크고 다른 cell들에 대해서는 값이 작아야 합니다. output weights를 가져와 unembedding을 통해 매핑한 뒤, 그 결과를 8x8 정사각형으로 시각화하여 이를 테스트할 수 있습니다.

이 방법의 명칭은 **direct logit attribution**, 또는 DLA입니다. 이는 residual stream에 값을 쓰는 컴포넌트들의 직접적인 효과를 연구하는 표준적인 방법입니다 (즉, 중간 컴포넌트를 거치는 경로는 무시합니다).

</details>

<details>
<summary>솔루션 (그래프 작성을 위한 코드)</summary>

아래 코드를 실행하여 그래프를 생성하십시오. neuron이 활성화될 때 `C0`의 logit을 실제로 높인다는 것을 확인할 수 있을 것입니다. 흥미롭게도, cell `D1`에 대해서도 양의 logit 효과가 나타납니다. 이에 대해서는 이후 연습 문제에서 DLA를 더 깊게 다룰 때 자세히 살펴보겠습니다.

```python
# Get neuron output weights' cos sim with unembedding
w_out_L5N1393 = get_w_out(model, layer, neuron, normalize=True)
W_U_normalized = model.W_U[:, 1:] / model.W_U[:, 1:].norm(dim=0, keepdim=True)  # normalize, slice off logits for "pass"
cos_sim = w_out_L5N1393 @ W_U_normalized  # shape (60,)

# Turn into a (rows, cols) tensor, using indexing
cos_sim_rearranged = t.zeros((8, 8), device=device)
cos_sim_rearranged.flatten()[ALL_SQUARES] = cos_sim

# Plot results
utils.plot_board_values(
    cos_sim_rearranged,
    title=f"Cosine sim of neuron L{layer}N{neuron} with W<sub>U</sub> directions",
    width=450,
    height=380,
)
```

참고 - weights를 정규화하지 않는 방법을 선택할 수도 있습니다. 이 경우 neuron output weight와 unembedding 벡터 사이의 cosine similarity 대신, unembedding basis에서의 output weights를 보여주게 됩니다. 하지만 후자가 더 많은 정보를 제공하는데, 그 이유는 최댓값이 1이라는 것을 알고 있기 때문입니다 (또한 무엇을 기대해야 할지에 대한 아이디어도 있습니다 - ND 공간에서 무작위로 선택된 두 벡터의 cosine similarity의 기대 절대값은 `1/sqrt(N)` 규모로 스케일링됩니다 (유도 과정은 생략합니다). 이는 `1/sqrt(512) ≈ 0.04`을 의미하며, 원하신다면 이를 경험적으로 확인해 보실 수 있습니다).

</details>

### probe가 얼마나 많은 분산을 설명합니까?

우리는 또한 neuron의 input 및 output weight 중 어느 정도의 비율이 probe에 의해 캡처되는지 살펴볼 수 있습니다 (벡터가 unit norm을 갖도록 스케일링되었기 때문에, 투영된 벡터의 squared norm을 확인하면 이 답을 얻을 수 있습니다).

input weight는 이에 의해 잘 설명되는 반면, output weight는 어느 정도만 잘 설명되는 것을 볼 수 있습니다.

In [ ]:
w_in_L5N1393 = get_w_in(model, layer, neuron, normalize=True)
w_out_L5N1393 = get_w_out(model, layer, neuron, normalize=True)

U, S, Vh = t.svd(t.cat([my_probe.reshape(cfg.d_model, 64), blank_probe.reshape(cfg.d_model, 64)], dim=1))

# Remove the final four dimensions of U, as the 4 center cells are never blank and so the blank
# probe is meaningless there.
probe_space_basis = U[:, :-4]

print(f"Fraction of input weights in probe basis: {((w_in_L5N1393 @ probe_space_basis).pow(2).sum()):.4f}")
print(f"Fraction of output weights in probe basis: {((w_out_L5N1393 @ probe_space_basis).pow(2).sum()):.4f}")

<details>
<summary>도움말 - 여기서 무슨 일이 일어나고 있는지 이해가 되지 않습니다.</summary>

연결된 probe directions는 총 모델 dimensionality의 1/4인 rank 128을 가집니다 `d_model=512`. 무작위로 선택된 벡터의 경우, squared norm의 약 1/4이 이러한 probe directions의 span에 있을 것으로 기대할 수 있습니다 (또는 다른 방식으로 표현하자면, probe directions가 벡터 variance의 약 1/4을 설명할 것으로 기대할 수 있습니다).

input weights의 경우, 이 값이 더 크다는 것을 알 수 있습니다. 즉, neuron이 주로 이러한 probe directions에 반응하여 활성화되는 것으로 보입니다 (이는 neuron이 이러한 probe directions와 관련된 특정 보드 상태를 감지하고 있었다는 앞선 관찰 결과와 일치합니다). output weights의 경우, 이 값이 더 작으며, 이는 우리 neuron이 보드 상태를 업데이트하는 데 사용되기보다 주로 "C0가 legal하다"는 것을 예측하고 있다면 타당한 결과입니다. 이후 연습 문제에서 이것이 사실임을 확인하게 될 것입니다!

</details>

### 더 많은 뉴런들

표준 편차(activation 기준)가 가장 높은 layer 3 뉴런들에 대해 이를 시도해 보고, 해당 뉴런들의 output weight가 저의 probe 방향에 어떤 영향을 미치는지 살펴보겠습니다.

In [ ]:
layer = 3
top_neurons = focus_cache["post", layer][:, 3:-3].std(dim=[0, 1]).argsort(descending=True)[:10]

utils.plot_board_values(
    t.stack([calculate_neuron_output_weights(model, blank_probe_normalised, layer, n) for n in top_neurons]),
    title=f"Cosine sim of output weights and the 'blank color' probe for top layer {layer} neurons (by std dev)",
    board_titles=[f"L{layer}N{n.item()}" for n in top_neurons],
    width=1600,
    height=360,
)

utils.plot_board_values(
    t.stack([calculate_neuron_output_weights(model, my_probe_normalised, layer, n) for n in top_neurons]),
    title=f"Cosine sim of output weights and the 'my color' probe for top layer {layer} neurons (by std dev)",
    board_titles=[f"L{layer}N{n.item()}" for n in top_neurons],
    width=1600,
    height=360,
)

참고 - 대신 **kurtosis**(첨도)를 사용하여 실험해 볼 수도 있습니다. 이는 분포의 꼬리 부분이 얼마나 극단적인지를 측정하는 지표입니다. 이를 통해 단순히 작은 값과 큰 값이 적절히 섞인 neuron을 찾는 것보다, 희소하게 activation되는(상당한 outlier 값을 가진) neuron을 더 많이 찾는 데 도움이 될 수 있습니다 (sparsity가 왜 interpretability와 상관관계가 있는 경향이 있는지에 대해서는, 이 장의 sparse autoencoder 관련 내용을 참조하십시오!).

<details>
<summary>kurtosis를 구현하기 위한 코드를 얻으려면 이 드롭다운을 사용하십시오</summary>

```python
def kurtosis(tensor: Tensor, reduced_axes, fisher=True):
    """
    Computes the kurtosis of a tensor over specified dimensions.
    """
    return (
        ((tensor - tensor.mean(dim=reduced_axes, keepdim=True)) / tensor.std(dim=reduced_axes, keepdim=True)) ** 4
    ).mean(dim=reduced_axes, keepdim=False) - fisher * 3


top_layer_3_neurons = einops.reduce(
    focus_cache["post", layer][:, 3:-3], "game move neuron -> neuron", reduction=kurtosis
).argsort(descending=True)[:10]
```

</details>

또한 layer 4의 상위 neuron들을 시각화해 볼 수 있습니다:

In [ ]:
layer = 4
top_neurons = focus_cache["post", layer][:, 3:-3].std(dim=[0, 1]).argsort(descending=True)[:10]

utils.plot_board_values(
    t.stack([calculate_neuron_output_weights(model, blank_probe_normalised, layer, n) for n in top_neurons]),
    title=f"Cosine sim of output weights and the 'blank color' probe for top layer {layer} neurons (by std dev)",
    board_titles=[f"L{layer}N{n.item()}" for n in top_neurons],
    width=1600,
    height=360,
)

utils.plot_board_values(
    t.stack([calculate_neuron_output_weights(model, my_probe_normalised, layer, n) for n in top_neurons]),
    title=f"Cosine sim of output weights and the 'my color' probe for top layer {layer} neurons (by std dev)",
    board_titles=[f"L{layer}N{n.item()}" for n in top_neurons],
    width=1600,
    height=360,
)

왜 모든 top layer 4 neuron들이 blank probe 방향 중 하나와 거의 완벽하게 일치하며("my color" probe 방향과는 매우 낮은 일치도를 보이며), 이토록 놀라운 결과를 보여주는 것일까요?

셀은 오직 blank 상태일 때만 플레이 가능한 상태가 될 수 있습니다(당연합니다). blank 여부를 계산하는 것은 쉽기 때문에(단순히 수가 두어졌는지만 확인하면 됩니다), 모델은 이를 단일 layer에서 수행할 수 있어야 하며, 이것이 바로 위에서 우리가 확인한 neuron들입니다.

**질문 - 만약 이것이 사실이라면, neuron output weights를 unembedding weights와 비교했을 때 어떤 관찰 결과가 예상될까요?**

계속 읽기 전에 이에 대해 생각해 보시기 바랍니다.

<details>
<summary>정답</summary>

만약 이것이 사실이라면, output weights와 unembedding weights의 cosine similarity가 위에서 본 것과 동일한 heatmap 패턴을 보일 것으로 예상할 수 있습니다.

다시 말해, blank 셀에서 강하게 활성화되는 이 neuron들은 residual stream에 직접적으로 값을 쓰고 있으며, 이들의 output이 unembedding에 사용되어 자신이 감지한 blank 셀의 logit score를 높이고 있는 것입니다.

</details>

위의 "test your hypothesis" 연습 문제에서 사용하셨을 direct logit attribution 플로팅과 동일한 방법을 사용하여 이를 테스트해 보겠습니다.

In [ ]:
layer = 4
top_neurons = focus_cache["post", layer][:, 3:-3].std(dim=[0, 1]).argsort(descending=True)[:10]
w_out = t.stack([get_w_out(model, layer, neuron, normalize=True) for neuron in top_neurons])

# Get neuron output weights' cos sim with unembedding
W_U_normalized = model.W_U[:, 1:] / model.W_U[:, 1:].norm(dim=0, keepdim=True)  # normalize, slice off logits for "pass"
cos_sim = w_out @ W_U_normalized

# Turn into a tensor, using indexing
cos_sim_rearranged = t.zeros((10, 8, 8), device=device)
cos_sim_rearranged.flatten(1, -1)[:, ALL_SQUARES] = cos_sim

# Plot results
utils.plot_board_values(
    cos_sim_rearranged,
    title=f"Cosine sim of top neurons with W<sub>U</sub> directions (layer {layer})",
    board_titles=[f"L{layer}N{n.item()}" for n in top_neurons],
    width=1500,
    height=320,
)

좋습니다! 우리는 이 layer의 neuron들이 "빈칸(blankness)"을 직접 계산하고, 이 출력을 unembedding으로 직접 전달하여 빈 칸의 logit score를 높인다는 가설을 검증했습니다 (빈 칸이어야만 플레이 가능한 법적 상태가 되기 때문입니다).

섹션 3️⃣에서는 이러한 방식의 direct logit attribution을 더 많이 수행해 보겠습니다.


<details>
<summary>질문 - 만약 kurtosis 기준으로 layer 4의 상위 neuron들을 플로팅해 본다면, 이 "빈칸 감지(blank detecting)" neuron들을 찾지 못할 것입니다 (layer 3에서는 kurtosis가 더 좋은 결과를 주는 것처럼 보임에도 불구하고 말입니다). 왜 그렇다고 생각하십니까?</summary>

kurtosis로 정렬하는 것은 outlier를 찾는 데 가장 효과적입니다. 반면 std dev로 정렬하면 변동성/분포가 좋은 neuron, 즉 작은 값과 큰 값이 적절히 섞여 있는 neuron을 찾을 수 있습니다. 후자의 설명이 빈칸 감지 neuron들을 더 잘 설명하는데, 왜냐하면 이들은 bimodal 특성을 갖기 때문입니다 (칸이 비어 있으면 off, 채워져 있으면 on이며, 이 두 그룹 모두 상당한 크기를 가집니다).


</details>

### 이 섹션의 요약

우리는 다음과 같은 작업을 수행했습니다:

* 특정 neuron에 대한 특정 MLP weight 벡터를 반환하는 helper 함수 `get_w_in`, `get_w_out`를 정의했습니다.
* neuron의 input 및 output weight를 주어진 probe(예: `my_probe` 또는 `blank_probe`)의 방향으로 투영하여 반환하는 함수 `calculate_neuron_input_weights` 및 `calculate_neuron_output_weights`를 정의했습니다.
* 일부 neuron이 매우 해석 가능한 input weight를 가지고 있다는 것을 발견했습니다. 예를 들어:
    * `L5N1393`의 input weight를 probe 방향과 비교한 결과, 이 neuron이 blank-theirs-mine의 특정 대각선을 감지하고 있음을 발견했습니다. 이는 해당 빈 칸(blank square)이 유효하다는 것을 나타내기 때문입니다.
        * 이는 흥미로운 결과이며, 반드시 사전에 예측할 수 있었던 것은 아닙니다.
        * 또한 이 neuron에 대해 logit lens를 살펴본 결과, 이 패턴이 존재할 때 유효하게 될 square의 logit을 높이고 있다는 것을 발견했습니다.
    * 일부 layer 4 neuron의 input weight를 살펴본 결과, 이들이 square가 빈 칸인지(이는 유효성 판단에 필수적입니다)를 감지하고 있으며, 이들의 output weight가 embedding에서 해당 빈 칸들의 log prob를 높이는 데 직접적으로 사용되는 것으로 보인다는 것을 발견했습니다.
        * 이는 앞선 사례만큼 흥미롭지는 않으며 아마도 예측 가능했을 것입니다. 왜냐하면 "cell이 빈 칸인지 확인"하는 연산은 매우 간단하기 때문입니다.

## Activation Patching

다양한 circuit의 동작을 추적하는 가치 있는 기술은 activation patching입니다. 이는 [David Bau and Kevin Meng's excellent ROME paper](https://rome.baulab.info/)에서 처음 소개되었으며, 그곳에서는 causal tracing이라고 불렸습니다.

activation patching의 설정은 두 가지 서로 다른 입력에 대해 모델을 두 번 실행하는 것입니다. 하나는 clean run이고 다른 하나는 corrupted run입니다. clean run은 정답을 출력하고, corrupted run은 그렇지 않습니다. 핵심 아이디어는 모델에 corrupted 입력을 제공하되, 특정 activation에 **개입(intervene)**하여 clean run의 해당 activation을 **패치(patch)**하고(즉, corrupted activation을 clean activation으로 교체), 실행을 계속하는 것입니다. 그런 다음 출력이 정답 방향으로 얼마나 업데이트되었는지 측정합니다.

한 가지 핵심적인 세부 사항만 다른 clean 입력과 corrupted 입력을 신중하게 선택함으로써, 어떤 모델 컴포넌트가 이 세부 사항을 포착하고 이에 의존하는지 격리할 수 있습니다.

국소화(localise) 능력은 mechanistic interpretability의 핵심적인 단계입니다. 만약 계산이 확산되어 모델 전체에 퍼져 있다면, 무슨 일이 일어나고 있는지에 대해 명확한 mechanistic 스토리를 구성하기가 훨씬 더 어려울 가능성이 큽니다. 하지만 모델의 어느 부분이 중요한지 정확하게 식별할 수 있다면, 해당 부분을 확대하여 그것들이 무엇을 나타내고 서로 어떻게 연결되는지 결정할 수 있으며, 궁극적으로 그것들이 나타내는 기저의 circuit을 역공학(reverse engineer)할 수 있습니다.

아래 다이어그램들은 추상적인 신경망에서의 activation patching을 보여줍니다 (노드는 activation을 나타내며, 노드 사이의 화살표는 weight 연결을 나타냅니다).

clean input에 대한 일반적인 forward pass는 다음과 같습니다:

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/simpler-patching-1c.png" width="300">

그리고 corrupted input(초록색)에서 clean input(검은색)의 forward pass로 activation patching을 수행하는 모습은 다음과 같습니다:

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/simpler-patching-2c.png" width="440">

여기서 점선은 값의 patching을 나타냅니다 (즉, clean input에 대한 forward pass 도중, 노드 $D$를 corrupted input에서 갖는 값으로 대체합니다). 노드 $H$, $G$ 및 $F$는 주황색으로 표시되어 있으며, 이는 이들이 이제 clean 또는 corrupted와는 다른 분포를 따른다는 것을 나타냅니다.

우리는 다양한 방식으로 transformer에 패칭(patching)을 할 수 있습니다 (예: residual stream의 값, MLP, 또는 attention head의 출력 - 아래 내용을 참조하십시오). 또한 특정 sequence 위치에서 패칭을 함으로써 더욱 세밀하게 분석할 수도 있습니다 (다이어그램에는 표시되지 않음).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/simpler-patching-examples.png" width="840">

### patching 설정하기

patching을 수행하기 전에, clean 및 corrupted 데이터셋을 무엇으로 설정할지 결정하고, logit 세트를 평가하기 위한 metric을 생성해야 합니다.

비슷한 게임 결과로 이어지는 clean 및 corrupted move를 설정하는 것은 간단하지 않으므로, 여기서는 하나의 게임을 가져와 가장 최근의 move를 `E0`에서 `C0`로 변경합니다. 이렇게 하면 (백이 둘 move로서의) `F0`가 legal에서 illegal로 바뀌므로, 해당 logit을 patching metric으로 설정하겠습니다.

또한 metric을 logit 차이의 선형 함수로 설정하는 것이 합리적입니다. 이는 metric을 고유하게 지정하기에 충분합니다.

먼저, 이를 시각화하기 위해 original 및 corrupted 보드를 플롯해 보겠습니다:

In [ ]:
cell_r = 5
cell_c = 4
print(f"Flipping the color of cell {'ABCDEFGH'[cell_r]}{cell_c}")

board = utils.OthelloBoardState()
board.update(focus_games_square[game_index, : move + 1].tolist())
valid_moves = board.get_valid_moves()
flipped_board = copy.deepcopy(board)
flipped_board.state[cell_r, cell_c] *= -1
flipped_legal_moves = flipped_board.get_valid_moves()

newly_legal = [utils.square_to_label(move) for move in flipped_legal_moves if move not in valid_moves]
newly_illegal = [utils.square_to_label(move) for move in valid_moves if move not in flipped_legal_moves]
print("newly_legal", newly_legal)
print("newly_illegal", newly_illegal)

In [ ]:
game_index = 4
move = 20

# Get original & corrupted games (as token IDs & ints)
original_game_id = focus_games_id[game_index, : move + 1]
corrupted_game_id = original_game_id.clone()
corrupted_game_id[-1] = utils.label_to_id("C0")
original_game_square = t.tensor([utils.id_to_square(original_game_id)])
corrupted_game_square = t.tensor([utils.id_to_square(corrupted_game_id)])

original_state, original_legal_moves, original_legal_moves_annotation = get_board_states_and_legal_moves(
    original_game_square
)
corrupted_state, corrupted_legal_moves, corrupted_legal_moves_annotation = get_board_states_and_legal_moves(
    corrupted_game_square
)
utils.plot_board_values(
    t.stack([original_state[move], corrupted_state[move]]),
    text=[original_legal_moves_annotation[move], corrupted_legal_moves_annotation[move]],
    title="Focus game states",
    board_titles=["Original game (black plays E0)", "Corrupted game (black plays C0)"],
    width=650,
    height=380,
)

다음으로, 두 게임 모두에 대해 logit과 cache를 가져오겠습니다:

In [ ]:
original_logits, original_cache = model.run_with_cache(original_game_id)
corrupted_logits, corrupted_cache = model.run_with_cache(corrupted_game_id)

original_log_probs = original_logits.log_softmax(dim=-1)
corrupted_log_probs = corrupted_logits.log_softmax(dim=-1)

### 연습 문제 - patching metric 생성하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You shouldn't spend more than 10-15 minutes on this exercise.
> ```

마지막으로, patching metric을 생성해 보겠습니다. 이는 출력 logit에 적용하는 함수로, clean 값에서 (어떤 중요한 방식으로) 얼마나 변경되었는지를 측정하기 위해 사용합니다.

우리는 patching metric이 다음 조건들을 만족하기를 원합니다:

* logit이 clean distribution과 동일할 때 값은 **1**이어야 합니다.
* logit이 corrupted distribution과 동일할 때 값은 **0**이어야 합니다.
    * *참고 - 때로는 반대되는 관례를 사용하기도 합니다. 문맥에 따라 어느 쪽이든 정당화될 수 있습니다. 여기서는 1이라는 값이 "성능의 100%가 보존됨"을 의미하고, 0은 "성능이 사라짐"을 의미한다고 생각하십시오.*
* log prob의 **선형(linear)** 함수여야 합니다.
    * **중요 참고** - 이는 logit의 선형 함수인 것과 *다릅니다*. 그 이유를 알 수 있습니까?
* 마지막 게임 수(clean과 corrupted 사이에서 유일하게 변경되는 수이므로)에서 `F0` token에 대한 logit의 함수여야 합니다.
    * 참고 - 아래에 정의된 `f0_index` 변수를 사용하여 logit의 `d_vocab` 차원을 인덱싱할 수 있습니다.

이 정도면 patching metric을 고유하게 정의하기에 충분할 것입니다. 또한, 이 함수는 **scalar tensor**를 반환해야 한다는 점에 유의하십시오 (이는 transformerlens의 patching 함수들이 작동하는 데 중요합니다).

In [ ]:
F0_index = utils.label_to_id("F0")
original_F0_log_prob = original_log_probs[0, -1, F0_index]
corrupted_F0_log_prob = corrupted_log_probs[0, -1, F0_index]

print("Check that the model predicts F0 is legal in original game & illegal in corrupted game:")
print(f"Clean log prob: {original_F0_log_prob.item():.2f}")
print(f"Corrupted log prob: {corrupted_F0_log_prob.item():.2f}\n")


def patching_metric(patched_logits: Float[Tensor, "batch seq d_vocab"]) -> Float[Tensor, ""]:
    """
    Function of patched logits, calibrated so that it equals 0 when performance is same as on
    corrupted input, and 1 when performance is same as on original input.

    Should be linear function of the logits for the F0 token at the final move.
    """
    raise NotImplementedError()


tests.test_patching_metric(patching_metric, original_log_probs, corrupted_log_probs)

<details><summary>솔루션</summary>

```python
def patching_metric(patched_logits: Float[Tensor, "batch seq d_vocab"]) -> Float[Tensor, ""]:
    """
    Function of patched logits, calibrated so that it equals 0 when performance is same as on
    corrupted input, and 1 when performance is same as on original input.

    Should be linear function of the logits for the F0 token at the final move.
    """
    patched_log_probs = patched_logits.log_softmax(dim=-1)
    return (patched_log_probs[0, -1, F0_index] - corrupted_F0_log_prob) / (original_F0_log_prob - corrupted_F0_log_prob)
```
</details>

### 연습 문제 - patching 함수 작성하기

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> This exercise is very important; getting it right shows you understand activation patching.
> ```

아래에서 `patch_attn_layer_output` 및 `patch_mlp_layer_output` 함수를 완성해야 합니다.

이를 위해 TransformerLens hook을 사용해야 합니다. hook이 작동하는 방식에 대한 간단한 복습입니다:

- Hook 함수는 두 개의 필수 인자를 받습니다:
    - `tensor` (해당 hookpoint에서의 모델 activation을 담은 PyTorch tensor)
    - `hook` (헬퍼 속성 `hook.name`과 메서드 `hook.layer()`을 가진 `HookPoint` 객체)
- `model.run_with_hooks` 함수는 다음 인자들을 받습니다:
    - 실행할 token들 (첫 번째 인자)
    - `fwd_hooks` - `(hook_name, hook_fn)` 튜플의 리스트. `get_act_name`을 사용하여 hook 이름을 가져올 수 있음을 기억하십시오.

팁:
- hook을 추가하고 실행하는 함수의 시작 부분에 `model.reset_hooks()`를 넣는 것이 좋은 습관입니다. 이는 때때로 (실행 중 에러가 발생하여) hook이 제대로 제거되지 않는 경우가 있기 때문입니다. hook 에러를 수정했는데도, 깨진 hook이 지워지지 않았다는 사실을 모른 채 동일한 에러 메시지를 계속 받는 것만큼 답답한 일은 없습니다!
- `HookPoint` 객체는 hook 함수에서 유용하게 사용할 수 있는 `.layer()` 메서드와 `.name` 속성을 가지고 있습니다.

In [ ]:
def patch_final_move_output(
    activation: Float[Tensor, "batch seq d_model"],
    hook: HookPoint,
    clean_cache: ActivationCache,
) -> Float[Tensor, "batch seq d_model"]:
    """
    Hook function which patches activations at the final sequence position.

    Note, we only need to patch in the final sequence position, because the prior moves in the clean
    and corrupted input are identical (and this is an autoregressive model).
    """
    raise NotImplementedError()


def get_act_patch_resid_pre(
    model: HookedTransformer,
    corrupted_input: Float[Tensor, "batch pos"],
    clean_cache: ActivationCache,
    patching_metric: Callable[[Float[Tensor, "batch seq d_model"]], Float[Tensor, ""]],
) -> Float[Tensor, "2 n_layers"]:
    """
    Returns an array of results corresponding to the results of patching at each (attn_out, mlp_out)
    for all layers in the model.
    """
    raise NotImplementedError()


patching_results = get_act_patch_resid_pre(model, corrupted_game_id, original_cache, patching_metric)

pd.options.plotting.backend = "plotly"
pd.DataFrame(to_numpy(patching_results.T), columns=["attn", "mlp"]).plot.line(
    title="Layer Output Patching Effect on F0 Log Prob",
    width=700,
    labels={"value": "Patching Effect", "index": "Layer"},
).show()

<details>
<summary>스포일러 - 기대 결과</summary>

대부분의 layer는 중요하지 않다는 것을 알 수 있습니다! 하지만 MLP0, MLP5, MLP6 그리고 Attn7은 중요합니다! 다음 단계로는 더 세밀하게 접근하여 개별 neuron을 patch하고, 왜 해당 layer들이 중요한지 어디까지 확대해서 살펴볼 수 있을지 확인하는 것입니다. 이상적으로는 대부분의 모델이 여기서는 중요하지 않다는 점을 이용하여, 이러한 neuron들이 서로 어떻게 구성되고 embedding을 어떻게 변경하는지 이해하는 수준까지 도달하는 것입니다. 그런 다음 이 데이터를 앞서 설명한 neuron 이해 기법들과 비교해 보는 것입니다. 이 부분을 직접 탐구하고 싶으시다면, 지금 시점에서 아주 좋은 연습이 될 것입니다 (또는 모든 연습 문제의 마지막에 다시 돌아와서 수행하십시오).

attention layer들이 상당히 중요하지 않다는 점은 놀라운 일이 아닙니다. attention은 token 위치 간에 정보를 이동시키는 데 특화되어 있으며, 우리는 현재 위치의 정보만 변경했기 때문입니다! (attention이 현재 위치에서 작동하는 능력을 갖추고는 있지만, 그것이 주된 역할은 아닙니다).
</details>


<details><summary>정답</summary>

```python
def patch_final_move_output(
    activation: Float[Tensor, "batch seq d_model"],
    hook: HookPoint,
    clean_cache: ActivationCache,
) -> Float[Tensor, "batch seq d_model"]:
    """
    Hook function which patches activations at the final sequence position.

    Note, we only need to patch in the final sequence position, because the prior moves in the clean
    and corrupted input are identical (and this is an autoregressive model).
    """
    activation[0, -1, :] = clean_cache[hook.name][0, -1, :]
    return activation


def get_act_patch_resid_pre(
    model: HookedTransformer,
    corrupted_input: Float[Tensor, "batch pos"],
    clean_cache: ActivationCache,
    patching_metric: Callable[[Float[Tensor, "batch seq d_model"]], Float[Tensor, ""]],
) -> Float[Tensor, "2 n_layers"]:
    """
    Returns an array of results corresponding to the results of patching at each (attn_out, mlp_out)
    for all layers in the model.
    """
    model.reset_hooks()
    results = t.zeros(2, model.cfg.n_layers, device=device, dtype=t.float32)
    hook_fn = partial(patch_final_move_output, clean_cache=clean_cache)

    for i, activation in enumerate(["attn_out", "mlp_out"]):
        for layer in tqdm(range(model.cfg.n_layers)):
            patched_logits = model.run_with_hooks(
                corrupted_input,
                fwd_hooks=[(get_act_name(activation, layer), hook_fn)],
            )
            results[i, layer] = patching_metric(patched_logits)

    return results
```
</details>

### 이번 섹션 요약

우리는 다음과 같은 내용을 학습했습니다:

* **activation patching**이 어떻게 작동하는지 배웠습니다.
* patching을 위해 다음과 같은 데이터셋을 구축했습니다:
    * Clean distribution = 수정되지 않은 게임,
    * Corrupted distribution = 단 하나의 수(move)가 뒤집힌 게임 (특정 칸의 legality를 변경함),
* 어떤 층이 출력을 크게 변화시키는지 확인하기 위해 **attention 및 MLP layer의 출력**에서 patching 효과를 살펴보았습니다.
    * 몇 개의 MLP layer와 마지막 attention layer가 중요하다는 것을 발견했습니다.
        * attention layer가 대부분 중요하지 않았던 것은 놀라운 일이 아닙니다. attention의 주된 역할은 정보를 처리하는 것이 아니라 이동시키는 것이기 때문입니다.
    * 원한다면 이 시점에서 더 세밀하게 들어가, 이 layer들의 어떤 neuron이 유의미한 영향을 미쳤는지 탐색할 수 있습니다.

# 3️⃣ Neuron Interpretability: 심층 분석

> ##### 학습 목표
>
> - **direct logit attribution**을 적용하여 neuron의 output weight가 예측에 어떻게 직접적으로 영향을 미치는지 이해합니다.
> - SVD 기반 기법을 사용하여 neuron의 input/output 동작 중 어느 정도가 특정 subspace에 의해 포착되는지 평가합니다.
> - **max activating datasets** 및 **spectrum plots**와 같은 기법을 사용하고, 이들의 장점과 한계를 이해합니다.

neuron interpretability를 연습하기 위해, 이 neuron을 깊이 있게 이해해 보겠습니다. 여기서 사용하는 기법과 코드는 다른 어떤 neuron에도 상당히 잘 적용될 것입니다!

이 섹션의 취지는 실제로 다른 neuron에 적용할 수 있는 다양한 표준적인 방법들을 연습하는 것입니다. 저는 이 과정을 마친 후에도 여전히 꽤 혼란스럽고, 해결되지 않은 의문점들이 많이 남아 있는 상태로 끝내게 됩니다!

위와 같이, probe weight를 사용하여 입력 weight를 분석할 수 있습니다. 따라서 우리는 이것이 `(C0==BLANK) & (D1==THEIRS) & (E2==MINE)`를 감지하며, `C0` logit을 직접적으로 높인다는 가설을 세웁니다. 이러한 설정은 `C0`가 반드시 legal해야 함을 의미하는데, 왜냐하면 그것이 대각선을 따라 `D1`의 측면에 위치하게 될 것이기 때문입니다!

## Direct logit attribution

이전 섹션에서 짧게 살펴본 것처럼, 뉴런의 출력 가중치인 `w_out @ W_U`의 **direct logit attribution**을 확인하여 해당 뉴런이 출력 logit에 직접적으로 큰 영향을 미치는지 알 수 있습니다. 이는 순수하게 가중치에 기반한 분석이며, 아직 실제 모델 activation을 살펴보는 것은 아니라는 점에 유의하시기 바랍니다.

In [ ]:
layer = 5
neuron = 1393

# Get neuron output weights in unembedding basis
w_out = get_w_out(model, layer, neuron, normalize=False)
w_out_W_U_basis = w_out @ model.W_U[:, 1:]  # shape (60,)

# Turn into a (rows, cols) tensor, using indexing
w_out_W_U_basis_rearranged = t.zeros((8, 8), device=device)
w_out_W_U_basis_rearranged.flatten()[ALL_SQUARES] = w_out_W_U_basis

# Plot results
utils.plot_board_values(
    w_out_W_U_basis_rearranged,
    title=f"Cosine sim of neuron L{layer}N{neuron} with W<sub>U</sub> directions",
    width=450,
    height=380,
)

여기서 우리는 `C0`을 boosting하는 것이 일어나는 일의 중요한 부분임을 즉시 알 수 있습니다 (이는 이 MLP가 포착하고 있던 diagonal pattern에 대한 우리의 이전 가설과 일치합니다). 하지만 `D1` 또한 boosting한다는 점이 놀랍습니다! (우리의 가설에서 `D1`는 이미 채워져 있으므로, 불가능해야 합니다!)

한 가지 가설은 `C0`과 `D1`의 unembed가 매우 aligned되어 있어서, 하나만 boosting하고 다른 하나는 그렇지 않게 만들기 어렵다는 것입니다.

### 연습 문제 - C0와 D1의 unembeds 비교하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵⚪⚪⚪
> 
> You should spend up to 5-10 minutes on this exercise.
> ```

이 가설이 사실일까요? `C0`과 `D1` unembeds의 cosine similarity를 계산하여 테스트해 보십시오.

`"C0"`와 같은 label을 정수 인덱스(unembedding 인덱싱에 사용 가능)로 변환하려면 `utils.label_to_id` helper 함수를 사용할 수 있다는 점을 기억하십시오.

In [ ]:
# YOUR CODE HERE - calculate cosine sim between unembeddings

<details>
<summary>정답 (나와야 하는 결과)</summary>


cosine similarity가 0에 가깝다는 것, 즉 기본적으로 orthogonal하다는 것을 확인할 수 있습니다. 따라서 이 가설은 거짓입니다! 여기에서 또 어떤 일이 일어나고 있다고 생각하시나요?

</details>


<details><summary>풀이</summary>

```python
c0_U = model.W_U[:, utils.label_to_id("C0")].detach()
c0_U /= c0_U.norm()

d1_U = model.W_U[:, utils.label_to_id("D1")].detach()
d1_U /= d1_U.norm()

print(f"Cosine sim of C0 and D1 unembeds: {c0_U @ d1_U:.3f}")
```
</details>

이것이 neuron 출력의 *큰* 부분인지 확인하기 위해, $W_U$ subspace에 의해 캡처되는 출력 분산의 비율을 살펴보겠습니다.

### 연습 문제 - unembedding subspace에 의해 설명되는 분산의 비율 계산하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵⚪⚪⚪
> 
> You should spend up to 10-20 minutes on this exercise.
> This might be a bit harder to understand conceptually, without a linear algebra background.
> ```

참고 - "subspace에 의해 설명되는 벡터 분산의 비율"이란, 원래 벡터의 제곱 노름(squared norm) 중 해당 subspace로의 투영(projection)에 의해 표현되는 비율을 의미합니다. 따라서 벡터 $v$과 직교 행 벡터 행렬로 표현되는 subspace $W$가 있을 때, $v^T W$은 $v$를 $W$로 투영한 성분들이며, subspace $W$에 의해 설명되는 $v$의 분산 비율은 $\|v^T W\|^2 / \|v\|^2$입니다.

이 개념이 혼란스럽다면 [How much variance does the probe explain?](#how-much-variance-does-the-probe-explain) 섹션을 확인하시기 바랍니다. 해당 섹션에서는 지금 수행해야 할 계산과 유사하지만 더 복잡한 계산을 다룹니다.

*주의 - SVD를 수행하기 전에 $W_U$에서 0번째 vocab 항목을 제거해야 함을 기억하십시오. 이는 우리 데이터에 절대 등장하지 않는 "pass" 이동에 해당하기 때문입니다.*

In [ ]:
# YOUR CODE HERE - compute the variance frac of neuron output vector explained by unembedding subspace

<details>
<summary>솔루션 (코드 및 예상 결과)</summary>

```python
w_out = get_w_out(model, layer, neuron, normalize=True)
U, S, Vh = t.svd(model.W_U[:, 1:])
print(f"Fraction of variance captured by W_U: {((w_out @ U).norm().item() ** 2):.4f}")
```

분산의 약 1/4이 설명되는 것을 확인할 수 있습니다:

<pre style="white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace">Fraction of variance captured by W_U: 0.2868</pre>

이는 예상보다 낮은 수치이며, 여기서 더 많은 일이 일어나고 있음을 시사합니다. 아마도 이 neuron의 출력이 unembedding이 적용되기 전에 downstream의 다른 무언가에 의해 사용되는 것일까요?

</details>

또 다른 간단한 sanity check는 캐시에 저장된 50개의 게임에 대해 neuron activation이 어떻게 나타나는지 플로팅하는 것입니다. 이를 통해 해당 neuron이 소수의 게임에서만 중요하게 작용하며, 한 수 건너 한 번씩 중요하게 작용한다는 것을 알 수 있습니다 (이는 내 색상 대 상대 색상으로 정의된 feature의 경우 타당한 결과입니다. 색상이 매 수마다 바뀌기 때문에 이 특성 또한 매 수마다 교차하여 나타납니다!).

In [ ]:
neuron_acts = focus_cache["post", layer, "mlp"][:, :, neuron]

fig = px.imshow(
    to_numpy(neuron_acts),
    title=f"L{layer}N{neuron} Activations over 50 games",
    labels={"x": "Move", "y": "Game"},
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0.0,
    aspect="auto",
    width=900,
    height=400,
)
fig.show()

### 연습 문제 - 이 게임들 중 하나에서 어떤 일이 일어나고 있는지 파악하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 10-25 minutes on this exercise.
> This is a very valuable exercise for testing your ability to develop new hypotheses, not just writing code.
> ```

이 neuron이 활성화되는 게임 중 하나를 선택하십시오 (예를 들어, 게임 5가 좋은 예시로 보입니다). `plot_board_values`을 사용하여 게임의 상태를 플롯하십시오 (이 노트북의 앞부분에 있는 코드를 가져와 사용할 수 있습니다).

<details>
<summary>정답 (게임 5의 경우)</summary>

우리의 neuron은 10, 12, 14, 16, 18, 20번째 수에서 활성화됩니다. 우리의 이론에 따르면, 이 수들이 바로 `C0`가 합법적인 수인 경우일 것이라고 예상합니다. 왜냐하면 `D1`을 포함하는 대각선을 따라 캡처할 수 있기 때문입니다.

```python
imshow(
    focus_states[5, :25],
    facet_col=0,
    facet_col_wrap=5,
    y=list("ABCDEFGH"),
    facet_labels=[f"Move {i}" for i in range(25)],
    title="First 16 moves of first game",
    color_continuous_scale="Greys",
    coloraxis_showscale=False,
    width=1000,
    height=1000,
)
```

이것이 우리가 발견한 정확한 결과입니다. `C0`은 10, 12, 14, 16, 18로 표시된 상태에서 합법적인 다음 수입니다. 모든 경우에 백의 차례이며, 합법적인 이유는 `D1`는 흑이 차지하고 있고 `E2`은 백이 차지하고 있기 때문입니다. `C0` 칸은 다른 일부 상태(예: 22번째 수 이후 흑의 차례일 때)에서도 합법적이지만, 이 경우에는 `C0-D1-E2` 대각선을 따라 캡처할 수 있어서 합법적인 것이 아니므로 neuron이 활성화되지 않을 것으로 예상합니다.
</details>

## Max Activating Datasets

Max activating datasets는 뉴런의 activation을 더 잘 이해하기 위해 사용할 수 있는 유용하지만 때로는 오해의 소지가 있는 도구입니다. 저의 [Dynalist notes](https://dynalist.io/d/n2ZWtnoYHrU1s4vnFSAQ519J#z=pwjeUj-94p6EwwMO_41Kb3h1)에 따르면 다음과 같습니다:

* **Max Activating Dataset Examples** (또는 **dataset examples**, **max examples**)는 뉴런 interpretability를 위한 간단한 기법입니다. 모델을 많은 데이터 포인트에 대해 실행하고, 해당 뉴런을 얼마나 많이 activate 시키는지에 따라 상위 K개의 데이터 포인트를 선택합니다.
    * 때로는 이러한 입력값들에서 명확한 패턴이 나타나며 (예: 모두 배 사진인 경우), 이는 해당 뉴런이 그 패턴을 감지한다는 (약한!) 증거가 됩니다. 때로는 서로 다른 패턴에 따라 여러 입력 클러스터가 존재하며, 이는 해당 뉴런이 **polysemantic** (즉, 여러 서로 다른 feature의 조합을 표현함)임을 시사합니다.
    * 이는 매우 단순하고 무분별한 기법이며 비판을 받아왔습니다. 예를 들어 [The Interpretability Illusion](https://arxiv.org/abs/2104.07143)에서는 서로 다른 데이터셋이 서로 다른 예시 세트를 제공하며, 각 세트가 서로 **다른** 명확한 패턴을 가지고 있음을 발견했습니다.
    * 이미지 모델에 대한 결과는 [OpenAI Microscope](https://microscope.openai.com/)에서, 언어 모델에 대한 결과는 [Neuroscope](https://neuroscope.io/)에서 확인하십시오.

많은 게임에 대해 이를 제대로 수행하는 것은 많은 노력이 필요하지만, 여기서는 50개의 게임(3000 moves)에 대한 출력값이 캐싱되어 있으므로 상위 30개 게임을 살펴볼 수 있습니다.

In [ ]:
# Get top 30 games & plot them all
top_moves = neuron_acts > neuron_acts.quantile(0.99)
top_focus_states = focus_states[:, :-1][top_moves.cpu()]
top_focus_states_flip = focus_states_theirs_vs_mine[:, :-1][top_moves.cpu()]
utils.plot_board_values(
    top_focus_states,
    boards_per_row=10,
    board_titles=[f"{act=:.2f}" for act in neuron_acts[top_moves]],
    title=f"Top 30 moves for neuron L{layer}N{neuron}",
    width=1600,
    height=500,
)

# Plot heatmaps for how frequently any given square is mine/theirs/blank in those top 30
utils.plot_board_values(
    t.stack([top_focus_states_flip == 0, top_focus_states_flip == 1, top_focus_states_flip == 2]).float().mean(1),
    board_titles=["Blank", "Theirs", "Mine"],
    title=f"Aggregated top 30 moves for neuron L{layer}N{neuron}, in 'blank/mine/theirs' basis",
    width=800,
    height=380,
)

빈 셀들이 높은 빈도로 나타난다는 점에 유의하십시오. 이는 [you have to watch out for](https://arxiv.org/abs/2104.07143) 와 같은 사례 중 하나입니다. 데이터셋을 정확히 어떻게 정의하느냐에 따라 최종적으로 찾게 되는 max activating dataset의 특징이 달라지기 때문입니다 (그리고 각 데이터셋이 명확하지만 오해의 소지가 있는 패턴을 보일 수도 있습니다).

여기서는 "mine vs theirs"의 구분이 가장 흥미롭기 때문에 주로 이 부분에 집중합니다. 따라서 "mine"은 `-1` 이고 "theirs"는 `+1` 인 새로운 tensor를 생성하겠습니다. 그런 다음 이 tensor의 평균을 구하면 "mine vs theirs"에 대한 아이디어를 얻을 수 있습니다.

In [ ]:
focus_states_theirs_vs_mine_pm1 = t.zeros_like(focus_states_theirs_vs_mine, device=device)
focus_states_theirs_vs_mine_pm1[focus_states_theirs_vs_mine == 2] = 1
focus_states_theirs_vs_mine_pm1[focus_states_theirs_vs_mine == 1] = -1

board_state_at_top_moves = focus_states_theirs_vs_mine_pm1[:, :-1][top_moves].float().mean(0)
board_state_at_top_moves.shape

utils.plot_board_values(
    board_state_at_top_moves,
    title=f"Aggregated top 30 moves for neuron L{layer}N{neuron}<br>(1 = theirs, -1 = mine)",
    height=380,
    width=450,
)

### 연습 문제 - 더 많은 neuron 조사하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 15-30 minutes on this exercise.
> This exercise involves applying the same techniques you used above to other neurons, in a batched way.
> ```

max activating dataset의 모든 수(move)에 대해, `D1`은 상대방의 것이고 `E2`는 저의 것임을 알 수 있습니다. 이는 우리가 생각하는 대로 해당 neuron이 작동하고 있다는 상당히 강력한 증거입니다.

이 섹션 동안 우리가 그렸던 plot들의 종류를 검토해 보겠습니다. 두 가지 종류가 있었습니다:

1. **Direct logit attribution plot**: 해당 neuron과 각 square에 대응하는 unembedding weights 사이의 cosine similarity를 계산한 plot입니다.

<details>
<summary>Example</summary>

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/plottwo.png" width="950">

</details>

2. **Max activating dataset plot**: 우리가 선택한 dataset(특정 neuron에 대한 max activating dataset) 전체에서 특정 square가 얼마나 자주 저의 것 또는 상대방의 것이었는지를 보여주는 heatmap입니다.

<details>
<summary>Example</summary>

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/plotone.png" width="750">
</details>

이 두 가지 plot을 **layer 5의 상위 10개 neuron 전체**에 대해 그려보세요 (이전과 마찬가지로 "상위 10개"는 `focus_cache` 데이터의 모든 게임과 수에 걸친 neuron activation의 표준 편차로 측정합니다). 이전의 코드 일부를 복사하여 붙여넣어 사용할 수 있습니다.

plot을 그리기 위한 코드는 이미 제공되었습니다. 여러분이 해야 할 일은 `output_weights_in_logit_basis` 및 `board_states` tensor를 계산하는 것입니다. 이 tensor들은 위에서 plot을 생성하는 데 사용된 tensor들의 batched 버전이어야 합니다 (0번 축은 neuron index여야 하며, 즉 이 tensor들의 `[0, ...]`번째 slice가 이 섹션 앞부분에서 plotting 함수에 입력했던 값들이어야 합니다).

이 neuron들이 각각 무엇을 하고 있는지 추측할 수 있나요? `L5N1393` 때처럼 일부 neuron들을 logit basis로 plot 하는 것이 도움이 될까요?

In [ ]:
# YOUR CODE HERE - investigate the top 10 neurons by std dev of activations, see what you can find!

<details><summary>솔루션</summary>

```python
layer = 5
top_neurons = focus_cache["post", layer].std(dim=[0, 1]).argsort(descending=True)[:10]
board_states = []
output_weights_in_logit_basis = []

for neuron in top_neurons:
    # Get output weights in logit basis
    w_out = get_w_out(model, layer, neuron, normalize=False)
    state = t.zeros(8, 8, device=device)
    state.flatten()[ALL_SQUARES] = w_out @ model.W_U[:, 1:]
    output_weights_in_logit_basis.append(state)

    # Get max activating dataset aggregations
    neuron_acts = focus_cache["post", 5, "mlp"][:, :, neuron]
    top_moves = neuron_acts > neuron_acts.quantile(0.99)
    board_state_at_top_moves = focus_states_theirs_vs_mine_pm1[:, :-1][top_moves].float().mean(0)
    board_states.append(board_state_at_top_moves)


output_weights_in_logit_basis = t.stack(output_weights_in_logit_basis)
board_states = t.stack(board_states)

utils.plot_board_values(
    output_weights_in_logit_basis,
    title=f"Output weights of top 10 neurons in layer {layer}, in the output logit basis",
    board_titles=[f"L{layer}N{n.item()}" for n in top_neurons],
    width=1600,
    height=360,
)
utils.plot_board_values(
    board_states,
    title=f"Aggregated top 30 moves for each top 10 neuron in layer {layer}",
    board_titles=[f"L{layer}N{n.item()}" for n in top_neurons],
    width=1600,
    height=360,
)
```
</details>

결과를 어떻게 해석하시겠습니까?

<details>
<summary>답변 (몇 가지 예시)</summary>

이전 neuron과 마찬가지로, 이들 중 일부는 해석 가능해야 합니다. 예시:

* `L5N1406` - 만약 `D4`가 그들의 것이고 `D5`가 비어 있다면, 이는 `D5`에 대한 logit을 높입니다.
* `L5N1985` - 만약 `F4`이 비어 있고, 상대방의 기물들이 그 옆에 인접해 있다면, 이는 `F4`에 대한 logit을 높입니다.
</details>

## Spectrum Plots

뉴런에 대한 가설을 검증하는 가장 좋은 방법 중 하나는 **spectrum plot**을 사용하는 것입니다. spectrum plot에서는 전체 데이터 분포(또는 최소한 일부 무작위 샘플)에 걸친 뉴런 activation의 히스토그램을 그리고, 각 activation이 우리가 생각하는 뉴런의 탐지 특성을 가지고 있는지에 따라 분류합니다. 우리는 50개의 게임에 대한 뉴런 activation을 사용하여 이를 간략하게 구현할 수 있습니다 (대부분의 게임이 가설로 설정한 구성을 가지고 있지 않으므로, 정규화를 위해 각 히스토그램 그룹을 그룹 크기의 백분율로 표시합니다).

흥미롭게도, 우리의 가설이 뉴런을 완전히 포착하지 *못했다*는 것을 알 수 있습니다. 거의 모든 높은 activation이 가설로 설정한 구성을 가지고 있었지만, 일부 낮은 activation 또한 그러했습니다! 여기서 정확히 어떤 일이 일어나고 있는지는 확실하지 않지만, 이는 max activating dataset 예시의 약점을 드러냅니다. 즉, 가설이 false positive를 허용할 때 이를 알아차리기 어렵게 만듭니다!

<details>
<summary>질문 - 왜 일부 activation이 음수인지 설명해 주실 수 있나요?</summary>

우리가 `GELU`를 사용하고 있으며, 이는 음수가 될 수 있다는 점을 기억하십시오.
</details>

In [ ]:
c0 = focus_states_theirs_vs_mine_pm1[:, :, 2, 0]
d1 = focus_states_theirs_vs_mine_pm1[:, :, 3, 1]
e2 = focus_states_theirs_vs_mine_pm1[:, :, 4, 2]

label = (c0 == 0) & (d1 == -1) & (e2 == 1)

neuron_acts = focus_cache["post", 5][:, :, 1393]


def make_spectrum_plot(neuron_acts: Float[Tensor, "batch"], label: Bool[Tensor, "batch"], **kwargs) -> None:
    """
    Generates a spectrum plot from the neuron activations and a set of labels.
    """
    px.histogram(
        pd.DataFrame({"acts": neuron_acts.tolist(), "label": label.tolist()}),
        x="acts",
        color="label",
        histnorm="percent",
        barmode="group",
        color_discrete_sequence=px.colors.qualitative.Bold,
        nbins=100,
        **kwargs,
    ).show()


make_spectrum_plot(
    neuron_acts.flatten(),
    label[:, :-1].flatten(),
    title="Spectrum plot for neuron L5N1393 testing C0==BLANK & D1==THEIRS & E2==MINE",
    width=1200,
    height=400,
)

### 연습 문제 - 이 spectrum plot을 조사해 보세요

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Importance: 🔵🔵🔵🔵⚪
> 
> This is a very open-ended exercise, with no time estimates. You can come back to this at the end if you like.
> ```

이 설정과 낮은 activation을 가진 수들을 살펴보세요. 어떤 일이 일어나고 있습니까? 보드 상태에서 어떤 패턴이 보입니까? 수에서는 어떤 패턴이 보입니까? 게임이 진행되는 동안 neuron activation은 어떤 모습입니까?

### 연습 문제 - 더 많은 spectrum plot 만들기

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> This is a very open-ended exercise, with no time estimates. You can come back to this at the end if you like.
> ```

해석 가능한 attention pattern을 가진 다른 neuron을 찾아보십시오. 해당 neuron에 대한 spectrum plot을 그려보십시오. 어떤 결과가 나오나요?

### 이 섹션의 요약

이 섹션에서 우리는 다음을 수행했습니다:

* **direct logit attribution**을 사용하여 뉴런의 출력 가중치가 각 square의 출력 logit에 어떻게 영향을 미치는지 확인했습니다.
    * `L5N1393`이 cell `C0`의 logit을 높이고 있음을 발견했습니다.
* **max activating datasets**를 사용하여 어떤 종류의 게임 상태가 특정 뉴런을 강하게 활성화시키는지 확인했습니다.
    * `L5N1393`가 `(C0==BLANK) & (D1==THEIRS) & (E2==MINE)` 패턴일 때 강하게 활성화됨을 발견했습니다.
* 다른 여러 뉴런에 대해 이 두 가지 플롯을 반복했고, 많은 뉴런이 유사한 방식으로 해석 가능하다는 것을 발견했습니다 (즉, 모두 특정 패턴에서 강하게 활성화되었으며, 해당 패턴에서 플레이 가능한 square의 logit에 긍정적인 영향을 주었습니다).
* **spectrum plot**을 작성했고, 뉴런에 대한 우리의 설명이 전체 이야기가 아니라는 것을 발견했습니다 (해당 패턴을 가진 일부 게임 상태에서는 뉴런이 활성화되지 않았습니다).
    * 이는 뉴런의 동작에 대한 완전한 설명을 찾는 방법으로서 max activating datasets가 가진 약점을 드러냈습니다.

# 4️⃣ Probe 학습시키기

> ##### 학습 목표
>
> - linear probe를 설정하고 학습시키는 방법을 배웁니다.
> - 여러 개의 probe를 동시에 학습시키고, 그 성능을 Weights & Biases에 기록하는 방법을 알아봅니다.

이 마지막 섹션에서는 앞서 다루었던 linear probe로 돌아가서, 이를 처음부터 어떻게 학습시킬 수 있는지에 대해 논의하겠습니다.

여기서 전체 규모의 학습 과정을 수행하지는 않을 것이며, 대신 우리가 이미 사용한 `board_seqs_square_small.npy` 데이터셋을 포함하는 작은 예시를 살펴보겠습니다 (이 연습 문제들에서 사용해 온 실제 probe는 더 큰 `board_seqs_int.pth` 데이터셋을 사용하여 학습되었습니다).

참고 - 만약 첫째 주에 [model training](https://arena-ch0-fundamentals.streamlit.app/[0.3]_ResNets) 관련 내용을 학습했거나 ("transformer from scratch" 자료에서 학습한) ARENA 참가자라면, 이 모든 내용이 익숙하실 것입니다. 표준 ML 학습 루프를 작성해 본 경험이 없다면, 이 섹션을 시작하기 전에 해당 섹션을 먼저 학습하시는 것을 권장합니다.

한 가지 분명히 해야 할 중요한 점은, 여기서 transformer 모델 자체를 학습시키는 것이 아니라는 점입니다. 만약 이것이 표준 학습 루프였다면, 모델을 training mode로 실행하고 logit 출력과 실제 black/white/blank 레이블 사이의 cross entropy loss를 줄이는 방식으로 gradient를 업데이트했을 것입니다. 대신, 우리는 **모델을 inference mode로 실행하여 residual stream 값들을 캐싱하고, 이 값들에 probe를 적용한 다음, probe의 출력과 실제 mine/theirs/blank 레이블 사이의 cross entropy loss를 줄이는 방식으로 probe의 weight를 업데이트합니다.**

In [ ]:
utils.plot_board_values(
    focus_states[0, :16],
    boards_per_row=8,
    board_titles=[f"Move {i}" for i in range(1, 17)],
    title="First 16 moves of first game",
    width=1400,
    height=440,
)

이제 probe 학습 인자들을 저장하기 위한 **dataclass**를 생성하겠습니다. 이는 모든 변수를 한곳에 모아 관리할 수 있는 아주 좋은 방법입니다 (또한 VSCode의 자동 완성 기능과도 잘 작동합니다!). 또한, dataclass의 멋진 기능 중 하나인 이전 속성을 사용하여 속성을 정의할 수 있다는 점에 주목해 주세요 (예: `length` 속성을 확인하십시오).

또한 적절하게 정규화된 가중치를 가진 랜덤하게 초기화된 probe를 제공하는 `setup_linear_probe` 메서드도 포함했습니다.

In [ ]:
@dataclass
class ProbeTrainingArgs:
    # Determine the activations we'll train the probe on
    layer: int = 6
    pos_start: int = 5
    pos_end: int = -5  # i.e. we slice [pos_start: model.n_ctx + pos_end]

    # Game state (options are blank/mine/theirs)
    options: int = 3
    rows: int = 8
    cols: int = 8

    # Standard training hyperparams
    epochs: int = 3
    num_games: int = 10_000

    # Hyperparams for optimizer
    batch_size: int = 32
    lr: float = 1e-3  # high LR for quick convergence in these exercises; you may want to reduce
    betas: tuple[float, float] = (0.9, 0.99)
    weight_decay: float = 0.01

    # Saving & logging
    use_wandb: bool = False
    wandb_project: str | None = "othellogpt-probe"
    wandb_name: str | None = None

    # Code to get randomly initialized probe
    def setup_linear_probe(self, model: HookedTransformer):
        linear_probe = t.randn(model.cfg.d_model, self.rows, self.cols, self.options, device=device) / np.sqrt(
            model.cfg.d_model
        )
        linear_probe.requires_grad = True
        return linear_probe

일부 항목들의 의미를 다시 상기시켜 드리겠습니다:
* `modes`은 "흑 차례/홀수 수", "백 차례/짝수 수", 그리고 "모든 수"를 나타냅니다. 이전 연습 문제에서는 "흑 차례"만 사용했습니다 (모델이 "흑/백"보다는 "내 색깔/상대 색깔"을 감지하고 있기 때문에 이 선택은 크게 중요하지 않았습니다).
* `options` (우리의 linear probe용)은 "비어 있음", "흑", 그리고 "백"을 나타냅니다. 학습을 마친 후, "비어 있음", "상대방 것", 그리고 "내 것"으로 구성된 버전을 만들 것입니다.

이제 메인 코드 블록으로 넘어가서, linear probe를 학습시키기 위한 클래스를 작성하겠습니다.

### 연습 문제 - 아래의 빈 코드를 채워 넣으십시오

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 30-40 minutes on this exercise.
> There are several steps to this exercise, so after trying for some time you're recommended to look at the solution.
> ```

방금 `training_step` 함수를 미완성 상태로 두었으므로, 이 부분을 채워 넣으셔야 합니다. 이 함수는 backpropagate를 수행할 loss를 반환해야 합니다 (실제로 backprop 알고리즘을 수행하는 코드는 아래에서 확인하실 수 있습니다).

이 연습 문제를 더 쉽게 만들기 위해, 짝수 수(even moves)에 대해서만 단일 probe mode를 학습하도록 설정했습니다 (보너스 연습 문제로 3개의 mode를 동시에 학습시키는 것을 시도해 볼 수 있습니다. 이 경우 "even & odd" probe mode의 loss를 계산할 때 홀수 수와 짝수 수의 label을 반환해야 함을 기억하십시오).

작성하실 `training_step` 함수는 다음과 같이 동작해야 합니다:

- `model.run_with_cache`을 사용하여 학습 중인 layer의 cached residual stream 값들을 가져옵니다.
    - 팁: 불필요한 계산을 방지하기 위해 `stop_at_layer` 인자를 사용할 수 있습니다.
    - 팁: 필요한 activation만 가져오기 위해 `names_filter` 인자를 사용할 수도 있습니다 (이는 hook 이름이거나 hook 이름을 bool로 매핑하는 함수일 수 있습니다).
    - 팁: 여기서는 probe에 입력할 activation을 얻기 위해 모델을 실행하는 것이므로, inference mode를 사용해야 함을 기억하십시오.
    - 팁: 제공된 `games_id` tensor의 shape가 `(batch_size, 60)`이므로, 모델에 전달하기 전에 마지막 게임을 slice 하여 제거해야 함을 기억하십시오.
- activation을 적절하게 slice 합니다 (어떤 position을 사용할지는 `args.pos_start`과 `args.pos_end`가 알려주며, 현재는 짝수 position만 사용합니다).
- `d_model` dimension에 대해 inner product를 취하고 position들에 대해 합산하여 probe output을 계산합니다.
- probe output을 logprobs로 변환하고, indexing을 통해 정답 logprobs를 가져옵니다.
    - 팁: 짝수 position을 사용하고 있으므로, board state를 probe basis로 변환할 때 "mine=black, theirs=white"를 사용함을 기억하십시오.
    - 팁: 이 indexing을 위해서는 게임 시퀀스로부터 board state를 계산하는 이전 코드가 필요합니다. `get_board_states_and_legal_moves`을 사용하고 반환된 `state` object를 유지하는 것을 권장합니다. 이 state object는 empty, white, black 옵션에 대응하는 0, 1, 2 값을 가집니다.
- row 및 column dimension에 대해 합산하고 batch 및 seqpos dimension에 대해 평균을 내어 loss를 계산합니다 (후자는 데이터 batch dimension이지만, 전자는 각 board square에 대해 여러 probe를 동시에 학습시키므로 실질적으로 probe batch dimension이기 때문입니다).
- loss를 반환합니다 (이 코드는 이미 제공되어 있습니다).

이 함수의 구현을 마치면 아래 코드를 실행할 수 있습니다. training loss는 `64 * ln(3) ≈ 70` (각 square를 점유하는 대상에 대해 균등하게 무작위로 추측했을 때 얻게 되는 loss) 아래로 빠르게 떨어져야 합니다. 기본 hyperparameter를 사용한다면 학습 종료 시점까지 이 loss는 약 10-20 정도로 떨어져야 합니다.

에러 없이 코드가 작동할 때까지는 `use_wandb=False`를 설정하고 작업하시는 것을 권장합니다.

In [ ]:
class LinearProbeTrainer:
    def __init__(self, model: HookedTransformer, args: ProbeTrainingArgs):
        self.model = model
        self.args = args
        self.linear_probe = args.setup_linear_probe(model)

    def training_step(self, indices: Int[Tensor, "n_games"]) -> Float[Tensor, ""]:
        # Use indices to slice our batch of games (remember, games_id = token IDs
        # from 1 to 60, and games_square = indices of squares in board)
        indices_cpu = indices.cpu()
        games_id = board_seqs_id[indices_cpu]  # shape [batch n_moves=60]
        games_square = board_seqs_square[indices_cpu]  # shape [batch n_moves=60]

        # Define seqpos slicing (note, we add n_ctx to pos_end to deal with the zero case)
        pos_start = self.args.pos_start
        pos_end = self.args.pos_end + self.model.cfg.n_ctx

        # YOUR CODE HERE - define loss

        if self.args.use_wandb:
            wandb.log(dict(loss=loss.item()), step=self.step)
        self.step += 1

        return loss

    def shuffle_training_indices(self):
        """
        Returns the tensors you'll use to index into the training data.
        """
        n_indices = self.args.num_games - (self.args.num_games % self.args.batch_size)
        full_train_indices = t.randperm(self.args.num_games)[:n_indices]
        full_train_indices = einops.rearrange(
            full_train_indices,
            "(batch_idx game_idx) -> batch_idx game_idx",
            game_idx=self.args.batch_size,
        )
        return full_train_indices

    def train(self):
        self.step = 0
        if self.args.use_wandb:
            wandb.init(project=self.args.wandb_project, name=self.args.wandb_name, config=self.args)

        optimizer = t.optim.AdamW(
            [self.linear_probe],
            lr=self.args.lr,
            betas=self.args.betas,
            weight_decay=self.args.weight_decay,
        )

        for epoch in range(self.args.epochs):
            print(f"Epoch {epoch + 1}/{self.args.epochs}")
            full_train_indices = self.shuffle_training_indices()
            progress_bar = tqdm(full_train_indices)
            for indices in progress_bar:
                loss = self.training_step(indices)
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()
                progress_bar.set_description(f"Loss = {loss:.4f}")

        if self.args.use_wandb:
            wandb.finish()


t.set_grad_enabled(True)

args = ProbeTrainingArgs()
trainer = LinearProbeTrainer(model, args)
trainer.train()

<details><summary>솔루션</summary>

```python
class LinearProbeTrainer:
    def __init__(self, model: HookedTransformer, args: ProbeTrainingArgs):
        self.model = model
        self.args = args
        self.linear_probe = args.setup_linear_probe(model)

    def training_step(self, indices: Int[Tensor, "n_games"]) -> Float[Tensor, ""]:
        # Use indices to slice our batch of games (remember, games_id = token IDs
        # from 1 to 60, and games_square = indices of squares in board)
        indices_cpu = indices.cpu()
        games_id = board_seqs_id[indices_cpu]  # shape [batch n_moves=60]
        games_square = board_seqs_square[indices_cpu]  # shape [batch n_moves=60]

        # Define seqpos slicing (note, we add n_ctx to pos_end to deal with the zero case)
        pos_start = self.args.pos_start
        pos_end = self.args.pos_end + self.model.cfg.n_ctx

        # Cache resid_post from all our games (ignoring the last one)
        with t.inference_mode():
            _, cache = model.run_with_cache(
                games_id[:, :-1].to(device),
                return_type=None,
                names_filter=lambda name: name.endswith("resid_post"),
            )

        # We slice from the first even index (on or after pos_start), since we're
        # just looking at predictions made after white has played a move.
        pos_start_even = pos_start + (pos_start % 2)
        seqpos_indices = np.arange(pos_start_even, pos_end, 2)
        resid_post = cache["resid_post", self.args.layer][:, seqpos_indices]

        # Get probe output, i.e. probe_logits[g, p, r, c] = the 3-vector of logit
        # predictions for what color is in square [r, c] AFTER the p-th move is
        # played in game g.
        probe_logits = einops.einsum(
            resid_post,
            self.linear_probe,
            "batch pos d_model, d_model rows cols options -> batch pos rows cols options",
        )
        probe_logprobs = probe_logits.log_softmax(-1)

        # Get the actual game state. The original state has {0: empty, 1: black, -1: white} and
        # we want our probe to be in the basis {0: empty, 1: theirs, 2: mine}. We're only training
        # on even moves i.e. black just played and mine = white, so we just need to map -1 -> 2.
        state = get_board_states_and_legal_moves(games_square)[0]  # shape [batch moves 8 8]
        state = state[:, seqpos_indices]  # shape [batch pos 8 8]
        state[state == -1] = 2

        # Index into probe logprobs to get the logprobs for correct board state, and then
        # return loss as the mean over games & posns, and sum over rows & cols (since each
        # row & col is effectively an independent probe).
        correct_probe_logprobs = eindex(probe_logprobs, state, "game pos row col [game pos row col]")
        loss = -einops.reduce(correct_probe_logprobs, "game pos row col -> row col", "mean").sum()

        if self.args.use_wandb:
            wandb.log(dict(loss=loss.item()), step=self.step)
        self.step += 1

        return loss

    def shuffle_training_indices(self):
        """
        Returns the tensors you'll use to index into the training data.
        """
        n_indices = self.args.num_games - (self.args.num_games % self.args.batch_size)
        full_train_indices = t.randperm(self.args.num_games)[:n_indices]
        full_train_indices = einops.rearrange(
            full_train_indices,
            "(batch_idx game_idx) -> batch_idx game_idx",
            game_idx=self.args.batch_size,
        )
        return full_train_indices

    def train(self):
        self.step = 0
        if self.args.use_wandb:
            wandb.init(project=self.args.wandb_project, name=self.args.wandb_name, config=self.args)

        optimizer = t.optim.AdamW(
            [self.linear_probe],
            lr=self.args.lr,
            betas=self.args.betas,
            weight_decay=self.args.weight_decay,
        )

        for epoch in range(self.args.epochs):
            print(f"Epoch {epoch + 1}/{self.args.epochs}")
            full_train_indices = self.shuffle_training_indices()
            progress_bar = tqdm(full_train_indices)
            for indices in progress_bar:
                loss = self.training_step(indices)
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()
                progress_bar.set_description(f"Loss = {loss:.4f}")

        if self.args.use_wandb:
            wandb.finish()
```
</details>

마지막으로, 이전과 동일한 accuracy plot을 그려서 얼마나 잘 작동하는지 확인해 보겠습니다. 이번에는 짝수 및 홀수 모드 probe를 평균 내어 새로운 probe를 구성하는 것이 아니라, 단일 짝수 모드 probe만을 가져와 사용한다는 점에 유의하시기 바랍니다 (아래 보너스 연습 문제에서 3가지 모드를 한 번에 학습시키는 방법을 다룹니다).

In [ ]:
# Getting the probe's output, and then its predictions
probe_out = einops.einsum(
    focus_cache["resid_post", args.layer],
    trainer.linear_probe,
    "game move d_model, d_model row col options -> game move row col options",
)
probe_out_value = probe_out.argmax(dim=-1).cpu()

# See what the accuracy was in 3 cases: odd moves, even moves, and aggregate moves
is_correct = probe_out_value == focus_states_theirs_vs_mine[:, :-1]
accuracies_odd = einops.reduce(is_correct[:, 5:-5:2].float(), "game move row col -> row col", "mean")
accuracies_even = einops.reduce(is_correct[:, 6:-6:2].float(), "game move row col -> row col", "mean")
accuracies_all = einops.reduce(is_correct[:, 5:-5].float(), "game move row col -> row col", "mean")

utils.plot_board_values(
    1 - t.stack([accuracies_odd, accuracies_even, accuracies_all], dim=0),
    title="Average Error Rate of Linear Probe",
    board_titles=["Black to play", "White to play", "All Moves"],
    zmax=0.25,
    zmin=-0.25,
    height=400,
    width=900,
)

### 연습 문제 - 3가지 모드를 동시에 학습시키기 (보너스)

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵⚪⚪⚪⚪
> 
> You should spend up to 20-40 minutes on this exercise.
> ```

이 연습 문제는 매우 중요하거나 개념적으로 깊은 내용은 아니며 (상당히 까다로울 수 있습니다), 완결성을 위해 포함되었습니다. 여기서 인덱싱을 정확하게 맞추는 것이 꽤 까다롭습니다! 여러분이 해야 할 일은 다음과 같습니다:

- linear probe가 추가적인 `mode` 차원을 가지는 새로운 probe 학습 인자 클래스를 사용합니다 (아래에 관련 코드를 제공했습니다).
- (새로운 `LinearMultiProbeTrainer` 클래스 내의) `training_step` 함수를 다시 작성하여 3가지 모드를 병렬로 학습시킵니다.

마지막에 실행할 수 있는 샘플 코드도 제공해 드렸으며, 이를 통해 3개 probe 각각의 정확도(및 학습되지 않은 parity로의 전이 능력)를 그래프로 확인할 수 있습니다. 3가지 모드 각각의 학습 loss가 이전 probe의 전체 loss와 비슷한 크기이며, 모두 거의 동일한 속도로 감소하는 것을 확인할 수 있을 것입니다.

<details>
<summary>도움말 - <code>training_step</code> 함수를 어떻게 다시 작성해야 할지 헷갈립니다.</summary>

이전에는 짝수 이동(even moves)에 대해서만 하나의 linear probe를 학습시켰습니다. 이를 위한 단계별 과정은 다음과 같았습니다:

- shape가 `(games, posn=59, rows=8, cols=8, options=3)`인 probe logprobs를 계산합니다.
- 보드 상태 `state`를 basis `{0: empty, 1: white, -1: black}`에서 `{0: empty, 1: theirs, 2: mine}`로 다시 매핑합니다.
    - 짝수 값 위치만 사용했으므로, 해당 부분만 슬라이싱하여 매핑 `{0, 1, -1} -> {0, 1, 2}`을 수행할 수 있었습니다.
- basis-mapped 된 `state`로 logprobs의 인덱싱을 수행하여 shape가 `(games, posns, rows, cols)`인 올바른 logprobs를 얻습니다.
- loss = 올바른 logprobs의 음수 평균(짝수 위치만 포함)을 계산합니다.

이제 짝수 이동 전용, 홀수 이동 전용, 그리고 두 가지 모두를 위한 linear probe 3개를 동시에 학습시키고자 합니다. 과정은 다음과 같습니다:

- shape가 `(modes=3, games, posn=59, rows=8, cols=8, options=3)`인 probe logprobs를 계산합니다.
- 보드 상태 `state`를 basis `{0: empty, 1: white, -1: black}`에서 `{0: empty, 1: theirs, 2: mine}`로 다시 매핑합니다.
    - 홀수 및 짝수 위치를 모두 사용하므로, 이는 짝수 위치는 `{0, 1, -1} -> {0, 1, 2}`으로, 홀수 위치는 `{0, 1, -1} -> {0, 2, 1}`로 매핑하는 것을 의미합니다.
- basis-mapped 된 `state`로 logprobs의 인덱싱을 수행하여 각 probe에 대한 올바른 logprobs를 얻으며, 이는 shape가 `(modes, games, posn, rows, cols)`이 됩니다.
- loss = 올바른 logprobs의 음수 평균(짝수 probe는 짝수 위치만, 홀수 probe는 홀수 위치만, all probe는 모든 위치를 포함)을 계산합니다.

다시 말해, 과정은 매우 비슷하지만 전체 `state` 텐서를 사용해야 하며(홀수 및 짝수 이동에 대해 서로 다른 basis 매핑 적용), 각 loss를 올바른 sequence 위치 집합에 대해 계산하고 있는지 확인해야 합니다.

</details>

In [ ]:
@dataclass
class MultiProbeTrainingArgs(ProbeTrainingArgs):
    modes: int = 3  # even, odd, both (i.e. the data we train on)

    def setup_linear_probe(self, model: HookedTransformer):
        linear_probe = t.randn(
            self.modes,
            model.cfg.d_model,
            self.rows,
            self.cols,
            self.options,
            device=device,
        ) / np.sqrt(model.cfg.d_model)
        linear_probe.requires_grad = True
        return linear_probe


class LinearMultiProbeTrainer(LinearProbeTrainer):
    def training_step(self, indices: Int[Tensor, "n_games"]) -> Float[Tensor, ""]:
        indices_cpu = indices.cpu()
        games_id = board_seqs_id[indices_cpu]  # shape [batch n_moves=60]
        games_square = board_seqs_square[indices_cpu]  # shape [batch n_moves=60]

        pos_start = self.args.pos_start
        pos_end = self.args.pos_end + self.model.cfg.n_ctx

        # YOUR CODE HERE - define loss_even, loss_odd, loss_both
        loss = loss_even + loss_odd + loss_both

        if self.args.use_wandb:
            wandb.log(
                dict(
                    loss=loss.item(),
                    loss_even=loss_even.item(),
                    loss_odd=loss_odd.item(),
                    loss_both=loss_both.item(),
                ),
                step=self.step,
            )
        self.step += 1

        return loss


t.set_grad_enabled(True)

args = MultiProbeTrainingArgs(epochs=5)
trainer = LinearMultiProbeTrainer(model, args)
trainer.train()

In [ ]:
# Here, we test out each of our 3 probe modes (even / odd / both) on each of these 3 settings
# (even / odd / both). Hopefully we should see all 3 probes generalize!

probe_out = einops.einsum(
    focus_cache["resid_post", args.layer],
    trainer.linear_probe,
    "game move d_model, mode d_model row col options -> mode game move row col options",
)
probe_out_value = probe_out.argmax(dim=-1).cpu()  # mode game move row col

# For each mode, get the accuracy on even / odd / both
is_correct = probe_out_value == focus_states_theirs_vs_mine[:, :-1]  # mode game move row col
accuracies_even = einops.reduce(is_correct[:, 6:-6:2].float(), "mode game move row col -> mode row col", "mean")
accuracies_odd = einops.reduce(is_correct[:, 5:-5:2].float(), "mode game move row col -> mode row col", "mean")
accuracies_all = einops.reduce(is_correct[:, 5:-5].float(), "mode game move row col -> mode row col", "mean")

# Get all 3x3 accuracies, stacked over first dim
accuracies_stacked = t.concat([accuracies_even, accuracies_odd, accuracies_all], dim=0)

# Plot results!
board_titles = [
    f"{probe_mode} probe on {data_mode} data"
    for data_mode in ["even", "odd", "all"]
    for probe_mode in ["even", "odd", "both"]
]

utils.plot_board_values(
    1 - accuracies_stacked,
    title="Average Error Rate of Linear Probe",
    board_titles=board_titles,
    boards_per_row=3,
    zmax=0.25,
    zmin=-0.25,
    height=1000,
    width=900,
)

<details><summary>솔루션</summary>

```python
@dataclass
class MultiProbeTrainingArgs(ProbeTrainingArgs):
    modes: int = 3  # even, odd, both (i.e. the data we train on)

    def setup_linear_probe(self, model: HookedTransformer):
        linear_probe = t.randn(
            self.modes,
            model.cfg.d_model,
            self.rows,
            self.cols,
            self.options,
            device=device,
        ) / np.sqrt(model.cfg.d_model)
        linear_probe.requires_grad = True
        return linear_probe


class LinearMultiProbeTrainer(LinearProbeTrainer):
    def training_step(self, indices: Int[Tensor, "n_games"]) -> Float[Tensor, ""]:
        indices_cpu = indices.cpu()
        games_id = board_seqs_id[indices_cpu]  # shape [batch n_moves=60]
        games_square = board_seqs_square[indices_cpu]  # shape [batch n_moves=60]

        pos_start = self.args.pos_start
        pos_end = self.args.pos_end + self.model.cfg.n_ctx

        # Cache resid_post from all our games (ignoring the last one)
        with t.inference_mode():
            _, cache = model.run_with_cache(
                games_id[:, :-1].to(device),
                return_type=None,
                names_filter=lambda name: name.endswith("resid_post"),
            )

        # We're training on all modes, so we slice all resid values in our range.
        resid_post = cache["resid_post", self.args.layer][:, pos_start:pos_end]

        # Get probe output, i.e. probe_logits[m, g, p, r, c] = the 3-vector of logit predictions from
        # mode-m probe, for what color is in square [r, c] AFTER the p-th move is played in game g.
        probe_logits = einops.einsum(
            resid_post,
            self.linear_probe,
            "game pos d_model, mode d_model rows cols options -> mode game pos rows cols options",
        )
        probe_logprobs = probe_logits.log_softmax(-1)

        # Get the actual game state. The original state has {0: empty, 1: black, -1: white} and
        # we want our probe to be in the basis {0: empty, 1: theirs, 2: mine}. For even moves,
        # mine = white, so we map -1 -> 2. For odd moves, mine = black, so we map {1, -1} -> {2, 1}.
        state = get_board_states_and_legal_moves(games_square)[0]
        state[:, ::2][state[:, ::2] == -1] = 2
        state[:, 1::2][state[:, 1::2] == 1] = 2
        state[:, 1::2][state[:, 1::2] == -1] = 1
        state = state[:, pos_start:pos_end]

        # Index into the probe logprobs with the correct indices (note, each of our 3 probe modes
        # gives us a different tensor of logprobs).
        correct_probe_logprobs = eindex(
            probe_logprobs,
            state,
            "mode game pos row col [game pos row col]",  # -> shape [mode game pos row col]
        )
        # Get the logprobs we'll be using for our 3 different probes. Remember that for the even
        # and odd probes we need to take only the even/odd moves respectively (and also that we've
        # already sliced logprobs from pos_start: pos_end).
        pos_start_even, pos_start_odd = (0, 1) if pos_start % 2 == 0 else (1, 0)
        even_probe_logprobs = correct_probe_logprobs[0, pos_start_even::2]
        odd_probe_logprobs = correct_probe_logprobs[1, pos_start_odd::2]
        both_probe_logprobs = correct_probe_logprobs[2]
        # Get our 3 different loss functions
        loss_even = -einops.reduce(even_probe_logprobs, "game pos row col -> row col", "mean").sum()
        loss_odd = -einops.reduce(odd_probe_logprobs, "game pos row col -> row col", "mean").sum()
        loss_both = -einops.reduce(both_probe_logprobs, "game pos row col -> row col", "mean").sum()
        # We backprop on the sum of all 3 losses
        loss = loss_even + loss_odd + loss_both

        if self.args.use_wandb:
            wandb.log(
                dict(
                    loss=loss.item(),
                    loss_even=loss_even.item(),
                    loss_odd=loss_odd.item(),
                    loss_both=loss_both.item(),
                ),
                step=self.step,
            )
        self.step += 1

        return loss
```
</details>

보너스 연습 문제로 다음 사항들을 시도해 볼 수 있습니다:

- 학습 코드에 evaluation loop를 추가해 보세요. 이는 학습이 끝날 때까지 기다리지 않고 probe classification accuracy를 확인하고 싶을 때 유용합니다!
- 짝수 및 홀수 모드 probe의 평균을 내어 hybrid probe를 만들어 보세요 (이전 연습 문제에서 제공된 probe를 사용하여 probe를 만들었던 방식과 유사합니다). 이 probe의 accuracy가 짝수 또는 홀수 모드 probe보다 더 높습니까? 또한, 짝수와 홀수 이동 모두에 대해 학습된 probe보다 더 높습니까?

# ☆ 보너스

> 참고 - 이 내용은 [second post in the OthelloGPT sequence](https://www.lesswrong.com/s/nhGNHyJHbrofpPbRG/p/qgK7smTvJ4DB8rZ6h#A_Transformer_Circuit_Laboratory)에서 직접 가져온 것입니다. 원하신다면 대신 해당 내용을 읽으셔도 좋습니다. 마지막 주차를 위한 추천 재현 과제들을 확인하시려면 끝까지 스크롤해 주세요.

_This is the second in a three post sequence about interpreting Othello-GPT. See [the first post](https://neelnanda.io/othello) for context._

_This post covers future directions I'm excited to see work on, why I care about them, and advice to get started. Each section is self-contained, feel free to skip around._

_Look up unfamiliar terms [here](https://neelnanda.io/glossary)_

The above sections leave me (and hopefully you!) pretty convinced that I've found something real and dissolved the mystery of whether there's a linear vs non-linear representation. But I think there's a lot of exciting mysteries left to uncover in Othello-GPT, _and_ that doing so may be a promising way to get better at reverse-engineering LLMs (the goal I actually care about). In the following sections, I try to:

*   Justify [why I think further work on Othello-GPT is interesting](/posts/qgK7smTvJ4DB8rZ6h/othello-gpt-future-work-i-am-excited-about#Why_and_when_to_work_on_toy_models)
    *   (Note that my research goal here is to get better at transformer mech interp, [not to specifically understand emergent world models better](/posts/qgK7smTvJ4DB8rZ6h/othello-gpt-future-work-i-am-excited-about#This_is_not_about_world_models))
*   Discuss how this unlocks [finding modular circuits](/posts/qgK7smTvJ4DB8rZ6h/othello-gpt-future-work-i-am-excited-about#Finding_Modular_Circuits), and some preliminary results
    *   Rather than purely studying circuits mapping input tokens to output logits (like basically all prior transformer circuits work), using the probe we can study circuits mapping the input tokens to the world model, and the world model to the output logits - the difference between thinking of a program as a massive block of code vs being split into functions and modules.
    *   If we want to reverse-engineer large models, I think we need to get good at this!
*   Discuss how we can [interpret Othello-GPT's neurons](/posts/qgK7smTvJ4DB8rZ6h/othello-gpt-future-work-i-am-excited-about#Neuron_Interpretability_and_Studying_Superposition) - we're very bad at interpreting transformer MLP neurons, and I think that Othello-GPT's are simple enough to be tractable yet complex enough to teach us something!
*   Discuss how, more broadly, [Othello-GPT can act as a laboratory](/posts/qgK7smTvJ4DB8rZ6h/othello-gpt-future-work-i-am-excited-about#A_Transformer_Circuit_Laboratory) to get data on many other questions in transformer circuits - it's simple enough to have a ground truth, yet complex enough to be interesting

My hope is that some people reading this are interested enough to actually try working on these problems, and I end this section with [advice on where to start](/posts/qgK7smTvJ4DB8rZ6h/othello-gpt-future-work-i-am-excited-about#Where_to_start_).

## Why and when to work on toy models

_This is a long and rambly section about my research philosophy of mech interp, and you should feel free to move on to [the next section](/posts/qgK7smTvJ4DB8rZ6h/othello-gpt-future-work-i-am-excited-about#Finding_Modular_Circuits) if that's not your jam_

At first glance, playing legal moves in Othello (not even playing _good_ moves!) has nothing to do with language models, and I think this is a strong claim worth justifying. Can working on toy tasks like Othello-GPT really help us to reverse-engineer LLMs like GPT-4? I'm not sure! But I think it's a plausible bet worth making.

To walk through my reasoning, it's worth first thinking on what's holding us back - why haven't we already reverse-engineered the most capable models out there? I'd personally point to a few key factors (though note that this is my personal hot take, is not comprehensive, and I'm sure other researchers have their own views!):

*   **Conceptual frameworks**: To reverse-engineer a transformer, you need to know how to think like a transformer. Questions like: What kinds of algorithms is it natural for a transformer to represent, and how? Are features and circuits the right way to think about it? Is it even reasonable to expect that reverse-engineering is possible? How can we tell if a hypothesis or technique is principled vs hopelessly confused? What does it even mean to have truly identified a feature or circuit?
    *   I personally thought [A Mathematical Framework](https://transformer-circuits.pub/2021/framework/index.html) significantly clarified my conceptual frameworks for transformer circuits!
    *   This blog post is fundamentally motivated by forming better conceptual frameworks - do models form linear representations?
*   **Practical Knowledge/Techniques**: Understanding models is hard, and being able to do this in practice is hard. Getting better at this both looks like forming a better toolkit of techniques that help us form true beliefs about models, and also just having a bunch of practical experience with finding circuits and refining the tools - can we find any cases where they break? How can we best interpret the results?
    *   A concrete way this is hard is that models contain many circuits, each of which only activates on certain inputs. To identify a circuit we must first identify where it is and what it does, out of the morass! Activation patching (used in ROME, Interpretability in the Wild and refined with causal scrubbing) is an important innovation here.
*   **Understanding MLP Layers**: 2/3 of the parameters in transformers are in MLP layers, which process the information collected at each token position. We're pretty bad understanding them, and getting better at this is vital!
    *   We think these layers represent features as directions in space, and if each neuron represents a single feature, we're pretty good! But in practice this seems to be false, because of the poorly understood phenomena of superposition and polysemanticity
    *   [Toy Models of Superposition](https://transformer-circuits.pub/2022/toy_model/index.html) helped clarify my conceptual frameworks re superposition, but there's still a lot more to de-confuse! And a _lot_ of work to do to form the techniques to deal with it in practice. I'm still not aware of a single satisfying example of really understanding a circuit involving MLPs in a language model
*   **Scalability**: LLMs are _big_, and getting bigger all the time. Even if we solve all of the above in eg four layer transformers, this could easily involve some very ad-hoc and labour intensive techniques. Will this transfer to models far larger? And how well do the conceptual frameworks we form transformer - do they just break on models that are much more complex?
    *   This often overlaps with forming techniques (eg, causal scrubbing is an automated algorithm with the potential to scale, modulo figuring out many efficiency and implementation details). But broadly I don't see much work on this publicly, and would be excited to see more - in particular, checking how well our conceptual frameworks transfer, and whether all the work on small models is a bit of a waste of time!
        *   My personal hot take is that I'm more concerned about never getting _really_ good at interpreting a four layer model, than about scaling _if_ we're really good at four layer models - both because I just feel pretty confused about even small models, and because taking understood yet labour-intensive techniques and making them faster and more automatable seems hard but doable (especially with near-AGI systems!). But this is a complex empirical question and I could easily be wrong.

Within this worldview, what should our research goals be? Fundamentally, I'm an empiricist - models are hard and confusing, it's easy to trick yourself, and often intuitions can mislead. The core thing of any research project is getting feedback from reality, and using it to form true beliefs about models. This can either look like forming explicit hypotheses and testing them, or exploring a model and seeing what you stumble upon, but the fundamental question is whether you have the potential to be surprised and to get feedback from reality.

This means that any project is a trade-off between tractability and relevance to the end goal. Studying toy, algorithmic models is a double edged sword. They can be very tractable: they're clean and algorithmic which incentivises clean circuits, there's an available ground truth for what the model _should_ be doing, and they're often in a simple and nicely constrained domain. But it's extremely easy for them to cease to be relevant to real LLMs and become a nerd-snipe. (Eg, I personally spent a while working on grokking, and while this was very fun, I think it's just not very relevant to LLMs)

It's pretty hard to do research by constantly checking whether you're being nerd-sniped, and to me there are two natural solutions:

*   (1) To pick a concrete question you care about in language models, and to set out to specifically answer that, in a toy model that you're confident is a good proxy for that question
    *   Eg [Toy Models of Superposition](https://transformer-circuits.pub/2022/toy_model/index.html) built a pretty good toy model of residual stream superposition
*   (2) To pick a toy model that's a good enough proxy for LLMs _in general_, and just try hard to get as much traction on reverse-engineering that model as you can.
    *   Eg [A Mathematical Framework](https://transformer-circuits.pub/2021/framework/index.html) - I think that "train a model exactly like an LLM, but with only 1 or 2 layers" is pretty good as proxies go, though not perfect.

To me, working on Othello-GPT is essentially a bet on (2), that there in gneeral some are underlying principles of transformers and how they learn circuits, and that the way they manifest in Othello-GPT can teach us things about real models. This is definitely wrong in _some_ ways (I don't expect the specific circuits we find to be in GPT-3!), and it's plausible this is wrong in enough ways to be not worth working on, but I think it seems plausible enough to be a worthwhile research direction. My high-level take is just "I think this is a good enough proxy about LLMs that studying it hard will teach us generally useful things".

There's a bunch of key disanalogies to be careful of! Othello is fundamentally not the same task as language: Othello is a much simpler task, there's only 60 moves, there's a rigid and clearly defined syntax with correct and incorrect answers (not a continuous mess), the relevant info about moves so far can be _fully_ captured by the current board state, and generally many sub-tasks in language will not apply.

But it's also surprisingly analogous, at least by the standards of toy models! Most obviously, it's a transformer trained to predict the next token! But the task is also much more complex than eg modular addition, and it has to do it in weird ways! The way I'd code Othello is by doing it recursively - find the board state at move n and use it to get the state at move n+1. But transformers can't do this, they need to do things with a fixed number of serial steps but with a lot of computation in parallel (ie, at every move it must simultaneously compute the board state _at that move_ in parallel) - it's not obvious to me how to do this, and I expect that the way it's encoded will teach me a lot about how to represent certain kinds of algorithms in transformers. And it needs to be solving a bunch of sub-tasks that interact in weird ways (eg, a piece can be taken multiple times in each of four different directions), computing and remembering a lot of information, and generally forming coherent circuits.

In the next few sections I'll argue for how finding modular circuits can help build **practical knowledge and techniques**, what we could learn from **understanding its MLPs**, and more broadly how it could act as a laboratory for forming better **conceptual frameworks** (it's clearly not a good way to study scalability lol)

### This is not about world models

A high-level clarification: Though the focus of the original paper was on understanding how LLMs can form emergent world models, this is _not_ why I am arguing for these research directions. My interpretation of the original paper was that it was strong evidence for the fact that it's _possible_ for "predict the next token" models to form world emergent models, despite never having explicit access to the ground truth of the world/board state. I personally was already convinced that this was possible, but think the authors did great work that showed this convincingly and well (and I am even more convinced after my follow-up!), and that there's not much more to say on the "is this possible" question.

There's many interesting questions about whether these happen _in practice_ in LLMs and what this might look like and how to interpret it - my personal guess is that they do _sometimes_, but are pretty expensive (in terms of parameters and residual stream bandwidth) and only form when it's high value for reducing next token loss and the model is big enough to afford it. Further, there's often much cheaper hacks, eg, BingChat doesn't need to have formed an explicit chess board model to be decent at playing legal moves in chess! Probably not even for reasonably good legal play: the chess board state is _way_ easier than Othello, pieces can't even change colour! And you can get away with an implicit rather than explicit world model that just computes the relevant features from the context, eg to see where to a move a piece from, just look up the most recent point where that piece was played and look at the position it was moved to.

But Othello is very disanalogous to language here - playing legal moves in Othello has a single, perfectly sufficient world model that I can easily code up (though not quite in four transformer layers!), and which is incredibly useful for answering the underlying task! Naively, Othello-GPT roughly seems to be spending 128 of its 512 residual stream dimensions of this model, which is very expensive (though it's probably using superposition). So while it's a proof of concept that world models are possible, I don't think the finer details here tell us much about whether these world models actually happen in real LLMs. This seems best studied by actually looking at language models, and [I think there's many exciting questions here](/posts/XNjRwEX9kxbpzWFWd/200-cop-in-mi-looking-for-circuits-in-the-wild)! (eg doing mech interp on [Patel et al's work](https://openreview.net/pdf?id=gJcEM8sxHK)) The point of my investigation was more to refine our conceptual frameworks for thinking about models/transformers, and the goal of these proposed directions is to push forward transformer mech interp in general.

## Finding Modular Circuits

Basically all prior work on circuits (eg, [induction heads](https://dynalist.io/d/n2ZWtnoYHrU1s4vnFSAQ519J#z=_Jzi6YHRHKP1JziwdE02qdYZ), [indirect object identification](https://dynalist.io/d/n2ZWtnoYHrU1s4vnFSAQ519J#z=iWsV3s5Kdd2ca3zNgXr5UPHa), [the docstring circuit](/posts/u6KXXmKFbXfWzoAXn/a-circuit-for-python-docstrings-in-a-4-layer-attention-only), and [modular addition](https://arxiv.org/pdf/2301.05217.pdf)) have been on what I call **end-to-end** circuits. We take some model behaviour that maps certain inputs to certain outputs (eg the input of text with repetition, and the output of logits correctly predicting the repetition), and analyse the circuit going from the inputs to the outputs.

This makes sense as a place to start! The inputs and outputs are inherently interpretable, and the most obvious thing to care about. But it stands in contrast to much of the image circuits work, that identified neurons representing interpretable features ([like curves](https://distill.pub/2020/circuits/curve-circuits)) and studied how they were computed and how these were used to computed more sophisticated features (like [car wheels -> cars](https://distill.pub/2020/circuits/zoom-in)). Let's consider the analogy of mech interp to reverse-engineering a compiled program binary to source code. End-to-end circuits are like thinking of the source code as a single massive block of code, and identifying which sections we can ignore.

But a natural thing to aim for is to find **variables**, corresponding to interpretable activations within the network that correspond to **features**, some property of the input. The linear representation hypothesis says that these should be directions in activation space. It's not guaranteed that LLMs _are_ modular in the sense of forming interpretable intermediate features, but this seems implied by exiasting work, eg in the residual stream (often studied with probes), or in the MLP layers ([possibly as interpretable neurons](https://transformer-circuits.pub/2022/solu/index.html#section-6-3)). If we _can_ find interpretable variables, then the reverse-engineering task becomes much easier - we can now separately analyse the circuits that form the feature(s) from the inputs or earlier features, and the circuits that use the feature(s) to compute the output logits or more complex feature.

I call a circuit which starts or ends at some intermediate activation a **modular circuit** (in contrast to end-to-end circuits). These will likely differ in two key ways from end-to-end circuits:

*   They will likely be **shallower**, ie involving fewer layers of composition, because they're not end-to-end. Ideally we'd be able to eg analyse a single neuron or head in isolation.
    *   And hopefully easier to find!
*   They will be **composable** - rather than needing to understand a full end-to-end circuit, we can understand different modular circuits in isolation, and need only understand the input and output features of each circuit, not the circuits that computed them.
    *   Hopefully this also makes it easier to predict model behaviour off distribution, by analysing how interpretable units may compose in unexpected ways!

I think this is just obviously a thing we're going to need to get good at to have a shot at real frontier models! Modular circuits mean that we can both re-use our work from finding circuits before, and hopefully have many fewer levels of composition. But they introduce a new challenge - how do we find exactly what direction corresponds to the feature output by the first circuit, ie the **interface** between the two circuits? I see two natural ways of doing this:

*   Exploiting a privileged basis - finding interpretable neurons or attention patterns (if this can be thought of as a feature?) and using these as our interpretable foothold.
    *   This is great if it works, but superposition means this likely won't be enough.
*   Using probes to find an interpretable foothold in the residual stream or other activations - rather than assuming there's a basis direction, we learn the correct direction
    *   This seems the only kind of approach that's robust to superposition, and there's [a lot of existing academic work](https://arxiv.org/abs/2102.12452) to build upon!
    *   But this introduces new challenges - rather than analysing discrete units, it's now crucial to find the _right_ direction and easy to have errors. It seems hard to produce composable circuits if we can't find the right interface.

So what does any of this have to do with Othello-GPT? I think we'll learn a lot by practicing finding modular circuits in Othello-GPT. Othello-GPT has a world model - clear evidence of spontaneous modularity - and our linear probe tells us where it is in the residual stream. And this can be intervened upon - so we know there are downstream circuits that use it. This makes it a great case study! By about layer 4, of the 512 dimensions of the residual stream, we have 64 directions corresponding to which cell has "my colour" and 60 directions corresponding to which cells are blank (the 4 center cells are never blank). This means we can get significant traction on what _any_ circuit is reading or writing from the residual stream.

This is an attempt to get at the "**practical knowledge/techniques**" part of [my breakdown of mech interp bottlenecks](/posts/qgK7smTvJ4DB8rZ6h/othello-gpt-future-work-i-am-excited-about#Why_and_when_to_work_on_toy_models) - Othello-GPT is a highly imperfect model of LLMs, but I expect finding modular circuits here to be highly tractable and to tell us a lot. Othello-GPT cares a lot about the world model - the input format of a sequence of moves is hard and messy to understand, while "is this move legal" can be answered purely from the board state. So the model will likely devote significant resources to computing board state, forming fairly clean circuits. Yet I still don't fully know how to do it, and I expect it to be hard enough to expose a bunch of the underlying practical and conceptual issues and to teach us useful things about doing this in LLMs.

**Gnarly conceptual issues**:

*   How to find the _right_ directions with a probe. Ie the correct interface between world-model-computing circuits and world-model-using circuits, such that we can think of the two independently. I see two main issues:
    *   Finding _all_ of the right direction - a probe with cosine sim of 0.7 to the "true" direction might work totally fine
        *   In particular, can we stop the probe from picking up on features that are constant in this context? Eg "is cell B6 my colour" is only relevant if "is cell B6 blank" is False, so there's naively no reason for the probe to be orthogonal to it.
    *   Ignoring features that correlate but are not causally linked - the corner cell can only be non-blank if at least one of the three neighbouring cells are, so the "is corner blank" direction _should_ overlap with these.
        *   But my intuition is that the model is learning a _causal_ world model, not correlational - if you want to do complex computations it's useful to explicitly distinguish between "is corner blank" as a thing to compute and use downstream, and all the other features. Rather than picking up on statistical correlations in the data.
*   If we find interpretable directions in the residual stream that are not orthogonal, how do we distinguish between "the model genuinely wants them to overlap" vs "this is just interference from superposition"?
    *   Eg, the model should want "is cell A4 blank" to have positive cosine sim with the unembed for the "A4 is legal" logit - non-blank cells are never legal!
*   The world model doesn't seem to be fully computed by layer X and only used in layer X+1 onwards - you sometimes need to intervene before layer 4, and sometimes the calculation hasn't finished before layer 5. How can we deal with overlapping layers? Is there a clean switchover layer _per cell_ that we can calculate separately?
*   How can we distinguish between two features having non-zero dot product because of noise/superposition, vs because they are correlated and the model is using one to compute the other.

**Questions I want answered**:

*   How _can_ we find the true probe directions, in a robust and principled way? Ideas:
    *   Use high weight decay to get rid of irrelevant directions. SGD (maybe with momentum) may be cleaner than AdamW here
    *   Use more complex techniques than logistic regression, like [amnesiac probing](https://arxiv.org/abs/2006.00995) (I found [Eleuther's Tuned Lens](https://arxiv.org/pdf/2303.08112.pdf) paper a useful review)
    *   Find the directions that work best for causal interventions instead.
    *   Maybe use the janky probe directions to try to find the heads and neurons that compute the world model, and use the fact that these are a privileged-ish basis to refine our understanding of the probe directions - if they never contribute to some component of the probe, probably that component shouldn't be there!
    *   Maybe implicitly assume that the probe directions should form an orthogonal set
    *   Maybe train a probe, then train a second probe on the residual stream component orthogonal to the first probe. Keep going until your accuracy sucks, and then take some kind of weighted average of the residual stream.
*   How is the blank world model computed?
    *   This _should_ be really easy - a cell is blank iff it has never been played, so you can just have an attention head that looks at previous moves. Maybe it's done after the layer 0 attention!
    *   This is trivial with an attention head per cell, but probably the model wants to be more efficient. What does this look like?
        *   Eg it might have a single attention head look at _all_ previous moves with uniform attention. This will get all of the information, but at magnitude `1/current_move`, maybe it has the MLP0 layer sharpen this to have constant magnitude?
    *   Meta question: What's a principled way to find the "is blank" direction here? The problem is one of converting a three-way classifier (blank vs my vs their) to a binary classifier that can be summarised with a single direction. I'm currently taking `blank - (my + their)/2`, but this is a janky approach
*   How is the "my vs their" world model computed?
    *   This seems like where the actual meat of the problem is!
        *   Consider games where
*   Which techniques work well here? My money is on activation patching and direct logit attribution being the main place to start, see [activation patching demoed in the accompanying notebook](https://neelnanda.io/othello-notebook).
    *   I'd love for someone to try out [attribution patching](https://www.neelnanda.io/mechanistic-interpretability/attribution-patching) here!
    *   By activation patching, I both mean resample ablations (patching a corrupted activation into a clean run to see which activations are vs aren't necessary) and causal tracing (patching a clean activation into a corrupted run to see which activations contain sufficient information to get the task right)

### Preliminary Results On Modular Circuits

The point of this section is to outline exciting directions of future work, but as a proof of concept I've done some preliminary poking around. The meta-level point that makes me excited about this is that linear probes are _really_ nice objects for interpretability. Fundamentally, transformers are made of linear algebra! Every component (layer, head and neuron) reads its input from the residual stream with a linear map, and writes it output by adding it to the residual stream, which is a _really_ nice structure.

**Probing across layers**: One way this is nice is that we can immediately get a foothold into understanding how the world model is computed. The residual stream is the sum of the embeddings and the output of every previous head and neuron. So when we apply a linear map like our probe, we can also break this down into a direct contribution from each previous head and neuron.

This is the same key idea as direct logit attribution, but now our projection is onto a probe direction rather than the unembed direction for a specific next token. This means we can immediately zoom in to the step of the circuit immediately before the probe, and see which components matter for each cell!

As an example, let's look at move 20 in this game:

<img src="https://res.cloudinary.com/lesswrong-2-0/image/upload/f_auto,q_auto/v1/mirroredImages/qgK7smTvJ4DB8rZ6h/heamef6n9xzedn5lkd1a" width="300">

The probe can perfectly predict the board state by layer 4

<img src="https://res.cloudinary.com/lesswrong-2-0/image/upload/f_auto,q_auto/v1/mirroredImages/qgK7smTvJ4DB8rZ6h/cozpptrltfdnolvyjvfo" width="500">

We can now look at how much the output of each attention and each MLP layer contributed to this (concretely we take the output of each attention and each MLP layer on move 30, and project them onto the is\_blank direction and the is\_mine direction for each cell, and plot this as a heatmap - [check the accompanying notebook for details](https://neelnanda.io/othello-notebook)). The MLP layer contributions to whether a cell has my or their colour is particularly interesting - we can see that it normally does nothing, but has a strong effect on the central stripe of cells that were just taken by the opponent - plausibly MLPs calculate when a cell is taken, and attention aggregates this? I'd love to see if there are specific neurons involve.

<img src="https://res.cloudinary.com/lesswrong-2-0/image/upload/f_auto,q_auto/v1/mirroredImages/qgK7smTvJ4DB8rZ6h/tqfl3q5mzwcbycxpcfi5" width="700">

**Reading Off Neuron Weights**: Another great thing about a linear probe is that it gives us a meaningful set of directions and subspace in the residual stream (beyond that given by the embedding and unembedding). This means that we can take any component's input or output weights, and project them onto the probe directions to see how that component reads to or writes from the probe's subspace - from this we can often just read off what's going on!

The probe intervention works best between layer 4 and layer 5, so we might hypothesise that some neurons in layer 5 are reading from the probe's subspace - we can check by taking the cosine sim of the neuron's input vector and the probe's directions to see how it responds to each, [see the accompanying notebook for details](https://neelnanda.io/othello-notebook). Here's neuron L5N1393 which seems to mostly represent C0==BLANK & D1==THEIRS & E2==MINE (cherry-picked for reasons unrelated to the probe, discussed more in post 3). Reading the figure: Blue = THEIRS, Red=MINE, White can be either blank or 50-50 mine vs their's, so can't be read easily.

<img src="https://res.cloudinary.com/lesswrong-2-0/image/upload/f_auto,q_auto/v1/mirroredImages/qgK7smTvJ4DB8rZ6h/egyxbgsfnmipmzunhtsj" width="600">

Here's the neurons with the largest standard deviation of activation in layer 3 (a pretty arbitrary way of choosing some that might be interesting) - when we take the cosine sim of the _output_ weights of these and the my colour probe, we see some that are pretty striking (though note that this is only a 0.3 cosine sim, so other stuff may be going on!)

<img src="https://res.cloudinary.com/lesswrong-2-0/image/upload/f_auto,q_auto/v1/mirroredImages/qgK7smTvJ4DB8rZ6h/og3g5rxemalzsggr9qpd" width="800">

Note that this is a deliberately janky analysis - eg, I'm not ensuring that the probe directions are orthogonal so I may double count, and I'm not looking for other residual stream features. You can track how reasonable this approach by tracking what fraction of the neuron's input is explained by the probe's subspaces, which is 64% in this case (these could otherwise be entirely spurious numbers!).

I go into neuron interpretability in more detail in the next section, but I think this technique is exciting in combination with what I discuss there, because it provides another somewhat uncorrelated technique - if many janky techniques give the same explanation about a neuron, it's probably legit!

## Neuron Interpretability and Studying Superposition

As argued earlier, I think that the current biggest open problem in transformer mech interp is understanding the MLP layers of transformers. These represent over 2/3 of the parameters in models, but we've had much more traction understanding attention-focused circuits. I'm not aware of a single public example of what I'd consider a well-understood circuit involving transformer MLP layers (beyond possibly my work on modular addition in a one layer transformer, but that's cheating). There are tantalising hints about the circuits they're used in in eg [SoLU](https://transformer-circuits.pub/2022/solu/index.html) and [ROME](https://rome.baulab.info/), but I broadly still feel confused re what is mechanistically going on. I think this is a thing we obviously need to make progress on as a field! And I think we'll learn useful things from trying to understand Othello-GPT's MLP layers!

What could progress on understanding MLPs in general look like? I think that we both need to get practice just studying MLP layers, and that we need to form clearer conceptual frameworks. A lot of our intuitions about transformer neurons come from image models, where neurons seem to (mostly?) represent features, have ReLU activations, and seem to be doing fairly discrete kinds of logic, eg "if car wheel present and car body present and car window present (in the right places) -> it's a car".

<img src="https://res.cloudinary.com/lesswrong-2-0/image/upload/f_auto,q_auto/v1/mirroredImages/qgK7smTvJ4DB8rZ6h/ze4jai5ioub31ovgewdj" width="800">

Transformers are different in a bunch of ways - there's attention layers, there's a residual stream (with significantly smaller dimension than the number of neurons in each layer!), and smoother and weirder GELU activations. Most importantly, polysemanticity seem to be a much bigger deal - single neurons often represent multiple features rather than a feature per neuron - and we think this is because models are using superposition - they represent features as linear combinations of neurons and use this to compress in more features than they have dimensions. This was argued for pretty convincingly in [Toy Models of Superposition](https://transformer-circuits.pub/2022/toy_model/index.html), but their insights were derived from a toy model, which can easily be misleading. I'm not aware of any work so far exhibiting superposition or properly testing the predictions of that paper in a real model. I expect some ideas will transfer but some will break, and that I'll learn a lot from seeing which is which!

Othello-GPT is far from a real language model, but I expect that understanding its MLP layers would teach me a bunch of things about how transformer MLP layers work in general. The model needs to compress a fairly complex and wide-ranging set of features and computation into just eight layers, and the details of _how_ it does this will hopefully expose some principles about what is and is not natural for a transformer to express in MLP neurons.

What would progress here look like? My high-level take is that a solid strategy is just going out, looking for interesting neurons, and trying to understand them deeply - no grander purpose or high-level questions about the model needed. I'd start with similar goals as I gave in [the previous section](/posts/qgK7smTvJ4DB8rZ6h/othello-gpt-future-work-i-am-excited-about#Finding_Modular_Circuits) - look for the neurons that are used to compute the probe, and directly used by the probe. I also outline [some further preliminary results](/posts/qgK7smTvJ4DB8rZ6h/othello-gpt-future-work-i-am-excited-about#Preliminary_Results_On_Modular_Circuits) that may serve as inspiration.

I've learned a lot from case studies looking deeply at concrete case studies of circuits in models: [Interpretability in the Wild](https://arxiv.org/abs/2211.00593) found backup heads (that took over when earlier heads were ablated) and negative heads (that systematically boosted _incorrect_ solutions), and [the docstring circuit](/posts/u6KXXmKFbXfWzoAXn/a-circuit-for-python-docstrings-in-a-4-layer-attention-only) found a polysemantic attention head, and a head which used the causal attention mask to re-derive positional information. I would love to have some similar case studies of meaningful neurons!

### Empirically Testing Toy Models of Superposition

_The sections of my mech interp explainer on [superposition](https://dynalist.io/d/n2ZWtnoYHrU1s4vnFSAQ519J#z=3br1psLRIjQCOv2T4RN3V6F2) and on the [toy models of superposition](https://dynalist.io/d/n2ZWtnoYHrU1s4vnFSAQ519J#z=EuO4CLwSIzX7AEZA1ZOsnwwF) paper may be useful references_

I'm particularly excited about using Othello-GPT to test and validate some of the predictions of [Toy Models of Superposition](https://transformer-circuits.pub/2022/toy_model/index.html) about what we might find in transformers. Empirical data here seems really valuable! Though there are some important ways that the setup of Othello-GPT differs from their toy model. Notably, they study continuous (uniform \[0, 1\]) features, while Othello-GPT's features seem likely to be binary (on or off), as they're discrete and logical functions of the board state and of the previous moves. Binary features seem more representative of language, especially early token-level features like bigrams and multi-token words, and are also easier to put into superposition, because you don't need to distinguish low values of the correct feature from high values of the incorrect feature

A broader point is whether we expect Othello-GPT to use superposition at all? Their model has more features to represent than dimensions, and so _needs_ to use superposition to pack things in. It's not obvious to me how many features Othello-GPT wants to represent, and how this compares to the number of dimensions - my _guess_ is that it still needs to use superposition, but it's not clear. Some considerations:

*   There's actually a lot of very specific features it might want to learn - eg in the board state -> output logit parts there seems to be a neuron representing C0==BLANK & D1==THEIR'S & E2==MINE, ie can I place a counter in C0 such that it flanks exactly one counter on the diagonal line to the down and right - if this kind of thing is useful, it suggests the model is dealing with a large combinatorial explosion of cases for the many, many similar configurations!
    *   Further, computing the board state from the moves also involves a lot of messy cases, eg dealing with the many times and directions a piece can be flipped and combining this all into a coherent story.
        *   Reminder: Transformers are not recurrent - it can't compute the board state at move n from the state at move n-1, it needs to compute the state at every move _simultaneously_ with just a few layers of attention to move partial computation forwards. This is actually really hard, and it's not obvious to me how you'd implement this in a transformer!
*   There are two different kinds of superposition, residual stream superposition and neuron superposition (ie having more features than dimensions in the residual stream vs in the MLP hidden layer).
    *   The residual stream has 512 dimensions, but there's 8 layers of 2048 neurons each (plus attention heads) - unless many neurons do nothing or are highly redundant, it seems very likely that there's residual stream superposition!
        *   Though note that it's plausible it just has way fewer than 2048 features worth computing, and is massively over-parametrised. I'm not sure what to think here!
        *   The board state alone consumes 25% of the dimensions, if each feature gets a dedicated dimension, and I expect there's probably a bunch of other features worth computing and keeping around?

Concrete questions I'd want to test here - note that the use of dropout may obfuscate these questions (by incentivising redundancy and backup circuits), and this may be best answered in a model without dropout. These also may be best answered in a smaller model with fewer layers and a narrower residual stream, and so with a stronger incentive for superposition!:

*   Do important features get dedicated dimensions in the residual stream? (ie, orthogonal to all other features)
    *   Guesses for important features - whether black or white is playing, the board state, especially features which say which center cells have my colour vs their's.
*   Conversely, can we find evidence that there _is_ overlap between features in the residual stream?
    *   This is surprisingly thorny, since you need to distinguish this kind of genuine interference vs intentional overlapping, eg from the source of the first feature actually wanting to contribute a bit to feature two as well.
*   Do the important neurons seem monosemantic?
    *   Important could mean many things eg high effect when patching, high average activation or standard deviation of activation, high cost when ablated, or any other range of measurements, high gradient or gradient x activation
    *   My workflow would be to use the probe and unembed to interpret neuron weights, max activating dataset examples to help form a hypothesis, and then use a spectrum plot to properly analyse it (discussed more below).
*   Do we get seemingly unrelated features sharing a neuron? The paper predicts superposition is more likely when there are two uncorrelated or anti-correlated features, because then the model doesn't need to track the simultaneous interference of _both_ being there at once.
*   Can we find examples of a feature being computed that _needs_ more than one neuron? Analogous to how eg [modular addition uses ReLUs to multiply two numbers together](https://arxiv.org/pdf/2301.05217.pdf), which takes at least three to do properly. This is a bit of a long shot, since I _think_ any kind of discrete, Boolean operation can probably be done with a single GELU, but I'd love to be proven wrong!
*   Do features actually seem neuron aligned at all?
    *   If we find features in superposition, do they tend to still be sparse (eg linear combinations of 5 ish neurons) or diffuse (no noticable alignment with the neuron basis)
*   Can we find any evidence of spontaneous sorting of superposed features into geometric configurations? (A la the toy models paper)
*   Can you construct any adversarial examples using evidence from the observed polysemanticity?
*   Can you find any circuits used to deal with interference superposition? Or any motifs, like [the asymmetric inhibition motif](https://dynalist.io/d/n2ZWtnoYHrU1s4vnFSAQ519J#z=ZsydEQzRb1l2wuBzqHu9_gQF)?

### Preliminary Results On Neuron Interpretability

_Note that this section has some overlap with results discussed in [my research process](/posts/TAz44Lb9n9yf52pv8/othello-gpt-reflections-on-the-research-process#The_Research_Process)_

In addition to the results above [using the probe to interpret neuron weights](/posts/qgK7smTvJ4DB8rZ6h/othello-gpt-future-work-i-am-excited-about#Preliminary_Results_On_Modular_Circuits), an obvious place to start is **max activating dataset examples** - run the model over a bunch of games and see what moves the neuron activates the most on. This is actually a fair bit harder to interpret than language, since "what are the connections between these sequences of moves" isn't obvious. I got the most traction from studying board state - in particular, the average number of times each cell is non-empty, and the average number of times a cell is mine vs their's. Here's a plot of the latter for neuron L5N1393 that seems immediately interpretable - D1 is always their's, E2 is always mine! (across 50 games, so 3000 moves) I sometimes get similar results with other layer 5 and layer 6 neurons, though I haven't looked systematically.

<img src="https://res.cloudinary.com/lesswrong-2-0/image/upload/f_auto,q_auto/v1/mirroredImages/qgK7smTvJ4DB8rZ6h/d8babgvrgumkbgxixl4i" width="600">

Looking at the fraction of the time a cell is blank or not seems to give pretty interesting results for layer 3 and layer 4 neurons.

<img src="https://res.cloudinary.com/lesswrong-2-0/image/upload/f_auto,q_auto/v1/mirroredImages/nmxzr2zsjNtjaHh7x/qozdjzolfpembkvjn9iy" width="700">

I expect you can stretch max activating dataset examples further by taking into account more things about the moves - what time in the game they happened, which cells are flipped this turn (and how many times in total!), which cell was played, etc.

My guess from this and probe based analysis earlier was that neuron L5N1393 monosemantically represented the diagonal line configuration C0==BLANK & D1==THEIR'S & E2==MINE. This makes sense as a useful configuration since it says that C0 is a legal move, because it and E2 flank D1! But this seems inconsistent with the direct logit attribution of the neuron (ie the output vector of the neuron projected by the unembed onto the output logits), which seems to boost C0 a lot but also D1 a bit - which seems wildly inconsistent with it firing on D1 being their colour (and thus not a legal place to play!)

<img src="https://res.cloudinary.com/lesswrong-2-0/image/upload/f_auto,q_auto/v1/mirroredImages/qgK7smTvJ4DB8rZ6h/fsa4iovike7xuoz6tirm" width="500">

These techniques can all be misleading - max activating dataset examples [can cause interpretability illusions](https://arxiv.org/abs/2104.07143), direct logit attribution can fail for neurons that mostly indirectly affect logits, and probes can fail to interpret neurons that mostly read out unrelated features. One of the more robust tools for checking what a neuron means is a **spectrum plot** - if we think a neuron represents some feature, we plot a histogram of the "full spectrum" of the neuron's activations by just taking the neuron activation on a ton of data, and plotting a histogram grouped by whether the feature is present or not (used in [curve detectors](https://distill.pub/2020/circuits/curve-detectors) and [multimodal neurons](https://distill.pub/2021/multimodal-neurons/)). If a neuron is monosemantic, this should fairly cleanly separate into True being high and False being low!

<img src="https://res.cloudinary.com/lesswrong-2-0/image/upload/f_auto,q_auto/v1/mirroredImages/qgK7smTvJ4DB8rZ6h/ip8scaqsrftsd4qizbga" width="800">

Note that the y axis is percent (ie it's normalised by group size so both True and False's histograms add up to 100 in total, though True is far more spread out so it doesn't look it. This is hard to read, so here it is on a log scale (different to read in a different way!).

<img src="https://res.cloudinary.com/lesswrong-2-0/image/upload/f_auto,q_auto/v1/mirroredImages/qgK7smTvJ4DB8rZ6h/kgpl9w2vze8we6fmblqk" width="800">

These plots are somewhat hard to interpret, but my impression is that this neuron is plausibly monosemantic-ish, but with a more refined feature - basically all of the high activations have the diagonal line hypothesised, but this is necesssary _not_ sufficient - there's a bunch of negative activations with the line as well! Plausibly it's still monosemantic but there's some extra detail I'm missing, I'm not sure! My next steps would be to refine the hypothesis by inspecting the most positive and most negative True examples, and if I can get a cleaner histogram to then try some causal interventions (eg mean ablating the neuron and seeing if it has the effect my hypothesis would predict). I'd love to see someone finish this analysis, or do a similar deep dive on some other neurons!

Spectrum plots are a pain to make in general, because they require automated feature detectors to do properly (though you can do a janky version by manually inspecting randomly sampled examples, eg a few examples from each decile). One reason I'm excited about neuron interpretability in Othello-GPT is that it's really easy to write automated tests for neurons and thus get spectrum plots, and thus to really investigate monosemanticity! If we want to be able to make real and robust claims to have identified circuits involving neurons or to have mechanistically reverse-engineered a neurons, I want to better understand whether we can claim the neuron is genuinely only used for a single purpose (with noise) or is also used more weakly to represent other features. And a concrete prediction of the toy models framework is that there _should_ be some genuinely monosemantic neurons for the most important features.

That said, showing genuine monosemanticity is hard and spectrum plots are limited. Spectrum plots will still fall down for superposition with very rare features - these can be falsely dismissed as just noise, or just never occur in the games studied! And it's hard to know where to precisely draw the line for "is monosemantic" - it seems unreasonable to say that the smallest True activation must be larger than the largest False one! To me the difference is whether the differences genuinely contribute to the model having low loss, vs on average contributing nothing. I think questions around eg how best to interpret these plots are an example of the kind of practical knowledge I want to get from practicing neuron interpretability!

#### Case Study: Neurons and Probes are Confusing

As a case study in how this can be confusing, here's an earlier draft graph for the section on finding modular circuits - looking at the output weights of top layer 4 neurons (by std) in the blank probe basis. It initially seems like these are all neurons dedicated to computing that a single cell is blank. And I initially got excited and thought this made a great graph for the post! But on reflection this is weird and surprising (exercise: think through why before you read on)

<img src="https://res.cloudinary.com/lesswrong-2-0/image/upload/f_auto,q_auto/v1/mirroredImages/qgK7smTvJ4DB8rZ6h/yczdmmagc3g8wjz3dvay" width="800">

I argue that this is weird, because figuring out whether a cell is blank should be pretty easy - a cell can never become non-empty, so a cell is blank if and only if it has never been played. This can probably be done in a single attention layer, and the hard part of the world model is computing which cells are mine vs their's. So what's up with this?

It turns out that what's actually going on is that the blank probe is highly correlated with the unembed (the linear map from the final residual to the logits). A cell can be legal _only if_ it is blank, if a cell has a high logit at the end of the model, then it's probably blank. But our probe was computed after layer 6, when there's a lot of extraneous information that probably obscures the blankness information - probably, the probe also learned that if there's going to be a high logit for a cell then that cell is definitely blank, and so the blank directions are partially aligned with the unembed directions. Though on another interpretation, `is_blank` and the unembed are _intentionally_ aligned, because the model knows there's a causal link and so uses the `is_blank` subspace to also contribute to the relevant unembed.

And we see that the alignment with the unembed is even higher! (Around cosine sim of 0.8 to 0.9)

<img src="https://res.cloudinary.com/lesswrong-2-0/image/upload/f_auto,q_auto/v1/mirroredImages/qgK7smTvJ4DB8rZ6h/jckacomrew1ju1utpetf" width="800">

## A Transformer Circuit Laboratory

My final category is just the meta level point that I'm confused in many ways about the right **conceptual frameworks** when thinking about transformer circuits, and think that there's a lot of ways we could make progress here! Just as Othello-GPT helped provide notable evidence for the hypothesis that models form linear representations of features, I hope it can help clarify some of these - by concretely understanding what happens inside of it, we can make more informed guesses about transformers in general. Here's a rough brainstorm of weird hypotheses and confusions about what we might find inside transformers - I expect that sufficient investigation of Othello-GPT will shed light on many of them!

Since Othello-GPT is an imperfect proxy for LLMs, it's worth reflecting on what evidence here looks like. I'm most excited about Othello-GPT providing "existence proofs" for mysterious phenomena like memory management: case studies of specific phenomena, making it seem more likely that they arise in real language models. Proofs that something was not used/needed are great, but need to be comprehensive enough to overcome the null hypothesis of "this was/wasn't there but we didn't look hard enough", which is a high bar!

*   Does it do **memory management** in the residual stream? Eg overwriting old features when they're no longer needed. I'd start by looking for neurons with high negative cosine sim between their input and output vectors, ie which basically erase some direction.
    *   One hypothesis is that it implicitly does memory management by increasing the residual stream norm over time - LayerNorm scales it to have fixed norm, so this suppresses earlier features. If this is true, we might instead observe **signal boosting** - key features get systematically boosted over time (eg whether we're playing black or white)
    *   This might come up with cells that flip many times during previous moves - maybe the model changes its guess for the cell's colour back and forth several times as it computes more flips? Do each of these write to the probe direction and overwrite the previous one, or is it something fancier?
*   Do heads and neurons seem like the right units of analysis of the model? Vs eg entire layers, superposition-y linear combinations of neurons/heads, subsets of heads, etc.
*   Do components (heads and neurons) tend to form **tightly integrated circuits** where they strongly compose with just a few other components to form a coherent circuit, or tend to be **modular**, where each component does something coherent in isolation and composes with many other components.
    *   For example, an induction head could be either tightly integrated (the previous token head is highly coupled to the induction head and not used by anything else, and just communicates an encoded message about the previous token directly to the induction head) or could form two separate modules, where the previous token head's output writes to a "what was in the previous position" subspace that many heads (including the induction head!) read from
        *   My guess is the latter, but I don't think anyone's checked! Most working finding concrete circuits seems to focus on patching style investigations on a narrow distribution, rather than broadly checking behaviour on diverse inputs.
    *   On a given input, can we clearly detect which components are composing? Is _this_ sparse?
*   When two components (eg two heads or a head and a neuron) compose with each other, do they tend to write to some shared subspace that many other components read and write from, or is there some specific encod
    *   Do components form modules vs integrated circuits vs etc.
*   Can we find examples of head polysemanticity (a head doing different things in different contexts) or head redundancy (multiple heads doing seemingly the same thing).
    *   Do we see backup heads? That is, heads that compensate for an earlier head when that head is ablated. This model was trained with attention dropout, so I expect they do!
        *   Do these backup heads do anything when _not_ acting as backups?
        *   Can we understand mechanistically how the backup behaviour is implemented?
        *   Are there backup backup heads?
    *   Can we interpret the heads at all? I found this pretty hard, but there must be something legible here!
    *   If we find head redundancy, can we distinguish between head superposition (there's a single "effective head" that consists of a linear combination of these )
    *   Can we find heads which seem to have an attention pattern doing a single thing, but whose OV circuit is used to convey a bunch of different information, read by different downstream circuits
    *   Can we find heads which have very similar attention patterns (ie QK circuits) whose OV circuits add together to simulate a single head with an OV circuit of twice the rank?
*   Is LayerNorm ever used as a meaningful non-linearity (ie, the scale factor differs between tokens in a way that does useful computation), or basically constant? Eg, can you linearly replace it?
    *   Are there emergent features in the residual stream? (ie dimensions in the standard basis that are much bigger than the rest). Do these disproportionately affect LayerNorm?
*   The model has clearly learned some redundancy (because it was trained with dropout, but also likely would learn some without any dropout). How is this represented mechanistically?
    *   Is it about having backup circuits that takeover when the first thing is ablated? Multiple directions for the same feature? Etc.
*   Can you find more evidence for or against the hypothesis that features are represented linearly?
    *   If so, do these get represented orthogonally?
*   Ambitiously, do we have a shot at figuring out _everything_ that the model is doing? Does it seem remotely possible to fully-reverse engineer it?
    *   Is there a long tail of fuzzy, half-formed features that aren't clean enough to interpret, but slightly damage loss if ablated? Are there neurons that just do nothing either way?
    *   Some ambitious plans for interpretability for alignment involve aiming for [enumerative safety](https://transformer-circuits.pub/2022/toy_model/index.html#strategic-safety), the idea that we might be able to enumerate _all_ features in a model and inspect this for features related to dangerous capabilities or intentions. Seeing whether this is remotely possible for Othello-GPT may be a decent test run.
*   Do the residual stream or internal head vectors have a privileged basis? Both with [statistical tests like kurtosis](https://transformer-circuits.pub/2023/privileged-basis/index.html), and in terms of whether you can actually interp directions in the standard basis?
*   Do transformers behave like [ensembles of shallow paths](https://arxiv.org/abs/1605.06431)? Where each meaningful circuit tends to only involve a few of the 16 sublayers, and makes heavy use of the residual stream (rather than 16 serial steps of computation).
    *   Prior circuits work and techniques like [the logit lens](/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens) seems to heavily imply this, but it would be good to get more data!
    *   A related hypothesis - when a circuit involves several components (eg a feature is computed by several neurons in tandem) are these always in the same layer? One of my fears is that superposition gives rise to features that are eg linear combinations of 5 neurons, but that these are spread across adjacent layers!

## Where to start?

If you've read this far, hopefully I've convinced you there are interesting directions here that could be worth working on! The next natural question is, where to start? Some thoughts:

*   Read the [original paper carefully](https://arxiv.org/pdf/2210.13382.pdf)
*   If you're new to mech interp, check out [my getting started guide](https://neelnanda.io/getting-started).
    *   I particularly recommend getting your head around how a transformer works, and being familiar with linear algebra
*   Use [my accompanying notebook](https://neelnanda.io/othello-notebook) as a starting point which demonstrates many of the core techniques
    *   I highly recommend using [my TransformerLens library](https://github.com/neelnanda-io/TransformerLens) for this, I designed it to enable this kind of research
    *   [Check out the underlying codebase](https://github.com/likenneth/othello_world) (made by the original authors, thanks to Kenneth Li for the code and for letting me make additions!)
*   My [concrete open problems](https://neelnanda.io/concrete-open-problems) sequence has a bunch of tips on doing good mech interp research, especially in the posts on circuits in toy language models, on neuron interpretability, and on superposition.
*   Read through [my notes on my research process](/posts/TAz44Lb9n9yf52pv8/othello-gpt-reflections-on-the-research-process#The_Research_Process) to get a sense of what making progress on this kind of work looks like, and in particular the decisions I made and why.

### Concrete starter projects

I'll now try to detail some concrete open problems that I think could be good places to start. Note that these are just preliminary suggestions - the above sections outline my underlying philosophy of which questions I'm excited about and a bunch of scattered thoughts about how to make progress on them. If there's a direction you personally feel excited about, you should just jump in.

Ideas for gentle starter projects (Note that I have not actually tried these - I expect them to be easy, but I expect at least one is actually cursed! If you get super stuck, just move on):

*   How does the model decide that the cell for the current move is not blank?
    *   What's the natural way for a transformer to implement this? (Hint: Do you need information about previous moves to answer this?)
    *   At which layer has the model figured this out?
    *   Try patching between two possibilities for the current move (with the same previous game) and look at what's going on
*   Pick a specific cell (eg B3). How does the model compute that it's blank?
    *   I'd start by studying the model on a few specific moves. At which layer does the model conclude that it's blank? Does this come from any specific head or neuron?
    *   Conceptually, a cell is not blank if and only if it was played as a previous move - how could a transformer detect this? (Hint: A single attention head per cell would work)
*   Take a game where a center cell gets flipped many times. Look at what colour the model thinks that cell is, after each layer and move. What patterns can you see? Can you form any guesses about what's going on? (This is a high-level project - the goal is to form hypotheses, not to reach clear answers)
*   Take the `is_my_colour` direction for a specific cell (eg D7) and look for neurons whose input weight has high cosine similarity with this. Look at this neuron's cosine sim with every other probe direction, and form a guess about what it's doing (if it's a mess then try another neuron/cell). Example guesses might be
    *   Then look at the max activating dataset examples (eg the top 10 over 50 games) and check if your guess worked!
    *   Extension: Plot a [spectrum plot](/posts/qgK7smTvJ4DB8rZ6h/othello-gpt-future-work-i-am-excited-about#Preliminary_Results_On_Neuron_Interpretability) and check how monosemantic it actually is
*   Repeat the above for the `is_blank` direction.
*   Take the average of the even minus the average of the odd positional embeddings to get an "I am playing white" direction. Does this seem to get its own dedicated dimension, or is it in superposition?
    *   A hard part about answering this question is distinguishing there being non-orthogonal features, vs other components doing memory management and eg systematically signal boosting the "I am playing white" direction so it's a constant fraction of the residual stream. Memory management should act approximately the same between games, while other features won't.

### Cleaning Up

This was (deliberately!) a pretty rushed and shallow investigation, and I cut a bunch of corners. There's some basic cleaning up I would do if I wanted to turn this into a real paper or build a larger project, and this might be a good place to start!

*   Training a better probe: I cut a lot of corners in training this probe... Some ideas:
    *   Train it on both black and white moves! (to predict my vs their's, so flip the state every other move)
    *   I cut out the first and last 5 moves - does this actually help/matter? Check how well the current probe works on early and late moves.
    *   The state of different cells will be correlated (eg a corner can only be filled if a neighbouring cell is filled), so the probes may be non-orthogonal for boring reasons. Does it help to constrain them to be orthogonal?
    *   What's the right layer to train a probe on?
    *   The probe is 3 vectors (three-way logistic regression), but I want a `is_blank_vs_filled` and `is_mine_vs_theirs_conditional_on_not_being_blank` direction - what's the most principled way of doing this?
*   Rigorously testing interventions: I'm pretty convinced that intervening the probe does _something_, but
    *   Currently I take the _current_ coordinate with respect to the probe direction, negate that, and then scale. Plausibly, this is dumb and the magnitude of the original coordinate doesn't matter, and I should instead replace it with a constant magnitude. The place I'd start is to just plot a histogram of the coordinates in the probe directions
    *   Replicating the paper's analysis of whether their intervention works (their natural and unnatural benchmark)
*   Re-train the model: The model was trained with attention and residual dropout - this is not representative of modern LLMs, and incentivises messy and redundant representations and backup circuits, I expect that training a new model from scratch with no dropout will make your life much easier. _(Note that someone is currently working on this)_
    *   The current model is 8 layers with a residual stream of width 512. I speculate this is actually much bigger than it needs to be, and things might be cleaner with fewer layers and a wider stream, a narrower stream, or both.

## 추천 논문 재현 과제

### [Inside the mind of a superhuman Go model: How does Leela Zero read ladders?](https://www.lesswrong.com/posts/FF8i6SLfKb4g7C4EL/inside-the-mind-of-a-superhuman-go-model-how-does-leela-zero-2)

저자들은 초인적인 바둑 모델인 Leela Zero에 대해 interpretability 연구를 수행했습니다. logit lens와 유사한 기법을 통해, 그들은 Leela Zero의 residual 구조가 네트워크 전반에 걸쳐 선호되는 basis를 유도하며, 이로 인해 지속적이고 해석 가능한 channel이 생성된다는 것을 발견했습니다. policy 및 value head의 weight를 직접 분석함으로써, 모델이 바둑판 상단 가장자리의 pass 수 확률과 관련된 정보와 checkerboard 패턴의 바둑판 가치와 관련된 정보를 저장하고 있다는 것을 찾아냈습니다. 또한 저자들은 바둑의 특정 기술인 '축(ladder)'을 깊게 분석하여, 모델의 축 판단에 인과적으로 책임이 있는 매우 작은 모델 컴포넌트 서브셋을 식별했습니다.

이러한 모델에 interpretability 기법을 적용하는 것은 상당히 어려울 수 있습니다 (Leela Zero는 transformer 모델보다는 AlphaGo의 아키텍처에 기반한 깊은 CNN입니다). 하지만 이러한 결과 중 일부를 재현하는 것은 좋은 도전이 될 것입니다!

다음과 같은 경우에 이 재현 과제를 추천합니다:

- 이 섹션의 연습 문제들이 즐거웠던 분
- OthelloGPT의 게임 메커니즘에 대해 생각하고, 모델이 사용하는 특정 메커니즘을 깊게 분석하는 것이 즐거웠던 분
- 표준적인 interpretability 기법(예: logit lens 및 neuron analysis)을 서로 다른 아키텍처에 적용하는 아이디어에 흥미가 있는 분
- 바둑을 둘 줄 아는 분 (약하게 권장하며, 필수 사항은 아닙니다!)

### [Chess-GPT's Internal World Model](https://adamkarvonen.github.io/machine_learning/2024/01/03/chess-world-models.html)

이 연구에서 Adam Karvonen은 Neel Nanda의 OthelloGPT 방법론을 확장하여, 자신이 직접 학습시킨 체스 모델에 적용했습니다. 이 모델은 5,000만 개의 파라미터를 가진 GPT로, 500만 판의 체스 게임으로 학습되었으며, 4대의 RTX 3090 GPU를 사용하여 하루 만에 약 1300 Elo 수준의 실력을 학습했습니다.

링크된 포스트의 분석은 모델에 대한 몇 가지 간략한 분석과 더불어 여러 가지 제안된 향후 연구 방향을 제공합니다. 포스트의 결과를 재현하고, 제안된 향후 연구 방향 중 일부를 조사해 보시겠습니까?

다음과 같은 경우에 이 재현 과제를 추천합니다:

- 이 섹션의 연습 문제들이 즐거웠던 분
- OthelloGPT의 게임 메커니즘에 대해 생각하고, 모델이 사용하는 특정 메커니즘을 깊게 분석하는 것이 즐거웠던 분
- 바둑보다 체스 규칙에 더 익숙한 분
- 상대적으로 조금 더 쉬운 프로젝트를 원하는 분 (적어도 LeelaZero 작업에 비해서는 쉬우며, 이미 수행한 연습 문제들과 훨씬 더 유사할 것입니다)